In [ ]:
# Identity-Disjoint Audio-Visual Deepfake Detection
# An Empirical Study of Biometric Embeddings, Recursive Fusion, and Task Decomposition
#
# Paper: IJCB 2026 Submission
# Dataset: FakeAVCeleb v1.2 (21,544 videos, 500 identities)
# Split: 17,017 train / 2,100 val / 2,000 test (identity-disjoint)
# Encoders: ArcFace 512-dim (face) + ECAPA-TDNN 192-dim (voice) = 704-dim concat
# Primary metric: Macro F1 (4-class) | Secondary: Binary AUC
#
# Run order:
#   Phase 0  #env-setup          Imports, Drive mount
#   Phase 1  #data-split         Unzip, 3-way identity-disjoint split
#   Phase 2  #eda                EDA figures (optional)
#   Phase 3  #pixel-mlp          Pixel MLP lower bound
#   Phase 4  #mvf-baseline       MVF binary reference (skip if done)
#   Phase 5  #feat-extract       ArcFace + ECAPA encoders
#   Phase 6  #emb-mlp            Embedding MLP
#   Phase 7  #vfd-cosine         VFD cosine similarity
#   Phase 8  #one-shot-trm       One-Shot Transformer
#   Phase 9  #simplified-trm     Simplified TRM (GRU, no attention)
#   Phase 10 #full-trm-vfd       Full TRM-VFD (cross-attention + GRU)
#   Phase 10 #two-stage-mlp      Two-Stage MLP Pipeline
#   Phase 10 #improved-trm       Improved Full TRM (tokenized)
#   Phase 11 #ablation           Recursive steps K=1-5
#   Phase 11 #seed-stability     Seed stability [OPTIONAL]
#   Phase 11 #binary-table       Table 3 binary results
#   Phase 11 #results-table      Table 2 main results
#   Phase 11 #cross-dataset      Face-only transfer to Celeb-DF v2
#   Phase 11 #figures            ROC, confusion, F1 bar chart, t-SNE
#
# Key results:
#   Best 4-class Macro F1:  Two-Stage MLP Pipeline  0.66
#   Best single-stage F1:   Simplified TRM           0.65
#   Best Binary AUC:        Full TRM-VFD             0.82
#   Face-only transfer AUC: Simplified TRM           0.65 (Celeb-DF v2)
#
# Protocol:
#   Labels: 0=real 1=fake_audio 2=fake_video 3=fake_both
#   Loss: FocalLoss gamma=2.0 (all except Improved TRM)
#   Sampler: WeightedRandomSampler 1.5x boost on real + fake_audio
#   Val: model selection only. Test: touched once per model.
#   MVF: binary-only reference, excluded from Table 2.

In [ ]:
# CELL 2: IMPORT STATEMENTS
import subprocess, sys
import inspect
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "dotmap", "einops", "timm", "pyyaml", "pydub",
    "yacs", "insightface", "speechbrain",
    "onnxruntime-gpu"], check=False)

import os, sys, random, shutil, json, re, yaml, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
from collections import Counter, defaultdict
from PIL import Image
import cv2
import librosa
import soundfile as sf
import torchaudio
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, TensorDataset, WeightedRandomSampler

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve
)
from torchvision import transforms
from insightface.app import FaceAnalysis
try:
    from speechbrain.inference.speaker import SpeakerRecognition
except ImportError:
    from speechbrain.pretrained import SpeakerRecognition

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
os.makedirs("/content/report_figures", exist_ok=True)
os.makedirs("/content/results", exist_ok=True)
print("All imports done")
print("CUDA available:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
# CELL 3: Mount Drive + Clone MVF
from google.colab import drive
drive.mount('/content/drive')
os.environ["WANDB_MODE"] = "disabled"
!git clone -q https://github.com/xaCheng1996/MVF.git /content/MVF || true
%pip install -q dotmap einops timm pyyaml pydub yacs insightface speechbrain onnxruntime-gpu
print("Setup done")

In [ ]:
# CELL 4: Unzip FakeAVCeleb dataset
!mkdir -p /content/dataset
!unzip -q -o "/content/drive/MyDrive/FakeAVCeleb_v1.2.zip" -d /content/dataset
total = len(glob("/content/dataset/**/*.mp4", recursive=True))
print(f"Total videos found: {total}")

In [ ]:
# CELL 5: Verify all 4 categories exist
root = "/content/dataset/FakeAVCeleb_v1.2"
categories = ["RealVideo-RealAudio","RealVideo-FakeAudio","FakeVideo-RealAudio","FakeVideo-FakeAudio"]
for c in categories:
    path = os.path.join(root, c)
    exists = os.path.exists(path)
    count = len(glob(os.path.join(path,"**","*.mp4"), recursive=True)) if exists else 0
    print(f"{c}: exists={exists}, videos={count}")

In [ ]:
# CELL 6 │ Phase 1 │ Build Identity-Disjoint 3-Way Split
import random as random
rng = random.Random(42)

def clean_name(s):
    s = s.replace(" ","_").replace("(","").replace(")","")
    return re.sub(r"[^A-Za-z0-9._-]","_",s)

# find dataset root
fakeav_base = None
for p in ["/content/dataset/FakeAVCeleb_v1.2",
          "/content/dataset/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2",
          "/content/dataset"]:
    if len(glob(os.path.join(p,"**","*.mp4"),
                recursive=True)) > 1000:
        fakeav_base = p
        break
assert fakeav_base, "Dataset not found"
print(f"Dataset root: {fakeav_base}")

# 4-class label mapping
CATEGORY_TO_BUCKET = {
    "RealVideo-RealAudio":  ("real",       0),
    "RealVideo-FakeAudio":  ("fake_audio", 1),
    "FakeVideo-RealAudio":  ("fake_video", 2),
    "FakeVideo-FakeAudio":  ("fake_both",  3),
}
id_to_name = {
    0:"real", 1:"fake_audio",
    2:"fake_video", 3:"fake_both"}

# paths
mvf_root   = "/content/MVF_data"
face_root  = os.path.join(mvf_root, "face")
voice_root = os.path.join(mvf_root, "voice")
train_list = os.path.join(mvf_root, "train_frame.txt")
val_list   = os.path.join(mvf_root, "val_frame.txt")
test_list  = os.path.join(mvf_root, "test_frame.txt")
label_csv  = os.path.join(mvf_root, "FakeAVCeleb.csv")

if os.path.exists(mvf_root):
    shutil.rmtree(mvf_root)
os.makedirs(face_root,  exist_ok=True)
os.makedirs(voice_root, exist_ok=True)

# collect videos by identity
groups = defaultdict(list)
for category, (bucket_name, label) in \
        CATEGORY_TO_BUCKET.items():
        for vp in sorted(glob(os.path.join(
            fakeav_base, category,"**","*.mp4"),
            recursive=True)):
        parts = os.path.relpath(
            vp, fakeav_base).split(os.sep)
        if len(parts) < 5:
            continue
        ident = (f"{clean_name(parts[1])}__"
                 f"{clean_name(parts[2])}__"
                 f"{clean_name(parts[3])}")
        groups[ident].append({
            "video_path":     vp,
            "category":       category,
            "bucket_name":    bucket_name,
            "label":          label,
            "category_clean": clean_name(parts[0]),
            "ethnicity":      clean_name(parts[1]),
            "gender":         clean_name(parts[2]),
            "identity":       clean_name(parts[3]),
            "filename":       clean_name(
                os.path.splitext(parts[4])[0])
        })

print(f"Total identities: {len(groups)}")

# prevents category order bias in first 50 items
for ident in groups:
    rng.shuffle(groups[ident])

# 3-WAY identity-disjoint split:
keys = list(groups.keys())
rng.shuffle(keys)          # reproducible shuffle

n_total = len(keys)
n_train = int(0.8 * n_total)   # ~400
n_val   = int(0.1 * n_total)   # ~50
# remaining ~50 = test

train_ids = keys[:n_train]
val_ids   = keys[n_train:n_train + n_val]
test_ids  = keys[n_train + n_val:]

print(f"Train identities: {len(train_ids)}")
print(f"Val identities:   {len(val_ids)}  ← model selection")
print(f"Test identities:  {len(test_ids)} ← final report only")

# verify no overlap
assert len(set(train_ids) & set(val_ids))  == 0
assert len(set(train_ids) & set(test_ids)) == 0
assert len(set(val_ids)   & set(test_ids)) == 0
print("✓ No identity overlap between splits")

MAX_PER_IDENTITY = 50

def build_records(id_list, out_records):
    for ident in id_list:
        # items already shuffled — just truncate
        for item in groups[ident][:MAX_PER_IDENTITY]:
            vp    = item["video_path"]
            label = item["label"]
            fdir  = os.path.join(
                face_root,
                item["category_clean"],
                item["ethnicity"],
                item["gender"],
                item["identity"])
            vdir  = os.path.join(
                voice_root,
                item["category_clean"],
                item["ethnicity"],
                item["gender"],
                item["identity"])
            os.makedirs(fdir, exist_ok=True)
            os.makedirs(vdir, exist_ok=True)

            img_path = os.path.join(
                fdir, item["filename"]+".jpg")
            wav_path = os.path.join(
                vdir, item["filename"]+".wav")

            if not os.path.exists(img_path):
                os.system(
                    f'ffmpeg -y -i "{vp}" -frames:v 1 '
                    f'"{img_path}" -loglevel error')
            if not os.path.exists(wav_path):
                os.system(
                    f'ffmpeg -y -i "{vp}" -ar 16000 '
                    f'-ac 1 -vn "{wav_path}" '
                    f'-loglevel error')

            if (os.path.exists(img_path) and
                    os.path.exists(wav_path)):
                out_records.append((img_path, label))

# build all three splits
print("\nBuilding records...")
train_records, val_records, test_records = [], [], []
build_records(train_ids,  train_records)
build_records(val_ids,    val_records)
build_records(test_ids,   test_records)

# save all three txt files
with open(train_list,"w") as f:
    for p,y in train_records:
        f.write(f"{p} {y}\n")
with open(val_list,"w") as f:
    for p,y in val_records:
        f.write(f"{p} {y}\n")
with open(test_list,"w") as f:
    for p,y in test_records:
        f.write(f"{p} {y}\n")

# save label CSV
present = sorted(set(
    y for _,y in
    train_records+val_records+test_records))
with open(label_csv,"w") as f:
    f.write("id;name\n")
    for y in present:
        f.write(f"{y};{id_to_name[y]}\n")

# count labels
tc  = Counter(y for _,y in train_records)
vc  = Counter(y for _,y in val_records)
tsc = Counter(y for _,y in test_records)

print(f"\nTrain: {len(train_records)} | {dict(tc)}")
print(f"Val:   {len(val_records)}   | {dict(vc)}")
print(f"Test:  {len(test_records)}  | {dict(tsc)}")

for name, counts in [
        ("train", tc),
        ("val",   vc),
        ("test",  tsc)]:
    missing = [c for c in range(4) if counts[c] == 0]
    assert not missing, \
        f"{name} missing classes: {missing}"
    print(f"✓ {name}: all 4 classes present")

print(f"\n✓ train_frame.txt: {len(train_records)} samples")
print(f"✓ val_frame.txt:   {len(val_records)} samples")
print(f"✓ test_frame.txt:  {len(test_records)} samples")
print("⚠ test_frame.txt locked — use only for final eval")


In [ ]:
# CELL 7 │ Phase 1 │ Backup Split Files to Drive
drive_mvf = "/content/drive/MyDrive/TRM_DATASETS/MVF_data"
os.makedirs(drive_mvf, exist_ok=True)

# Save the 3 split files + label CSV
for fname in ["train_frame.txt", "val_frame.txt",
              "test_frame.txt", "FakeAVCeleb.csv"]:
    src = f"/content/MVF_data/{fname}"
    dst = f"{drive_mvf}/{fname}"
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"✓ Saved: {fname}")

print(f"\n✓ Split files saved to Drive")
print(f"  Location: {drive_mvf}")

def count_lines(path):
    with open(path) as f:
        return sum(1 for line in f if line.strip())

train_samples = count_lines("/content/MVF_data/train_frame.txt")
val_samples   = count_lines("/content/MVF_data/val_frame.txt")
test_samples  = count_lines("/content/MVF_data/test_frame.txt")

# Identity counts from the split variables (set in Cell 6)
n_train_ids = len(train_ids)
n_val_ids   = len(val_ids)
n_test_ids  = len(test_ids)

summary = {
    "train_samples":    train_samples,
    "val_samples":      val_samples,
    "test_samples":     test_samples,
    "train_ids":        n_train_ids,
    "val_ids":          n_val_ids,
    "test_ids":         n_test_ids,
    "seed":             42,
    "max_per_identity": MAX_PER_IDENTITY,
    "train_dist":       dict(tc),
    "val_dist":         dict(vc),
    "test_dist":        dict(tsc),
}

with open(f"{drive_mvf}/split_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"\n── Split Summary (computed from actual data) ──")
print(f"  Train: {train_samples:,} samples | {n_train_ids} identities | dist: {dict(tc)}")
print(f"  Val:   {val_samples:,} samples   | {n_val_ids} identities | dist: {dict(vc)}")
print(f"  Test:  {test_samples:,} samples  | {n_test_ids} identities | dist: {dict(tsc)}")
print(f"  Seed:  42 | MAX_PER_IDENTITY: {MAX_PER_IDENTITY}")
print(f"✓ split_summary.json saved — no hardcoded numbers")


In [ ]:
# CELL 8: Verify split files

import glob

def find_frame_for_video(video_path):
    """Find the extracted jpg frame for a given mp4 video."""
    parts   = video_path.replace("/content/dataset/FakeAVCeleb_v1.2/","").split("/")
    # parts = [category, ethnicity, gender, identity, video.mp4]
    if len(parts) < 5:
        return None
    category  = parts[0]
    ethnicity = parts[1]
    gender    = parts[2]
    identity  = parts[3]
    video_stem = os.path.splitext(parts[4])[0]

    # Search pattern in MVF_data/face
    pattern = (f"/content/MVF_data/face/{category}/"
               f"{ethnicity}/{gender}/{identity}/"
               f"*{video_stem}*.jpg")
    matches = glob.glob(pattern)
    if matches:
        return matches[0]

    # Fallback: any frame for this identity
    pattern2 = (f"/content/MVF_data/face/{category}/"
                f"{ethnicity}/{gender}/{identity}/*.jpg")
    matches2 = glob.glob(pattern2)
    return matches2[0] if matches2 else None

print("Deriving frame paths from video paths...")
frame_paths = []
found = 0
for vp in df_eda["video_path"]:
    fp = find_frame_for_video(vp)
    frame_paths.append(fp)
    if fp: found += 1

df_eda["frame_path"] = frame_paths
print(f"✓ Frame paths found: {found}/{len(df_eda)} "
      f"({found/len(df_eda)*100:.1f}%)")

# Now compute pixel features
print("Computing pixel features...")
pm, ps, ws, hs = [], [], [], []
for i, fp in enumerate(df_eda["frame_path"]):
    try:
        img = cv2.imread(fp) if fp else None
        if img is not None:
            g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            pm.append(float(g.mean()))
            ps.append(float(g.std()))
            hs.append(img.shape[0])
            ws.append(img.shape[1])
        else:
            pm.append(np.nan); ps.append(np.nan)
            hs.append(np.nan); ws.append(np.nan)
    except Exception:
        pm.append(np.nan); ps.append(np.nan)
        hs.append(np.nan); ws.append(np.nan)
    if (i+1) % 3000 == 0:
        print(f"  {i+1}/{len(df_eda)}")

df_eda["pixel_mean"] = pm
df_eda["pixel_std"]  = ps
df_eda["width"]      = ws
df_eda["height"]     = hs

valid = df_eda["pixel_mean"].notna().sum()
print(f"✓ Valid pixels: {valid}/{len(df_eda)} "
      f"({valid/len(df_eda)*100:.1f}%)")

In [ ]:
# CELL 9│ Phase 2 │ EDA — Build Dataset DataFrame
CATEGORY_TO_LABEL = {
    "RealVideo-RealAudio": 0,
    "RealVideo-FakeAudio": 1,
    "FakeVideo-RealAudio": 2,
    "FakeVideo-FakeAudio": 3,
}
LABEL_TO_NAME = {
    0:"real", 1:"fake_audio",
    2:"fake_video", 3:"fake_both"}

rows = []
root = "/content/dataset/FakeAVCeleb_v1.2"

for category, label in CATEGORY_TO_LABEL.items():
    folder = os.path.join(root, category)
    if not os.path.isdir(folder):
        continue
    for v in sorted(glob.glob(
            os.path.join(folder,"**","*.mp4"),
            recursive=True)):
        parts = v.split("/")
        rows.append({
            "video_path":  v,
            "category":    category,
            "label":       label,
            "label_name":  LABEL_TO_NAME[label],
            "ethnicity":   parts[-4] if len(parts)>4 else "unknown",
            "gender":      parts[-3] if len(parts)>3 else "unknown",
            "identity":    parts[-2] if len(parts)>2 else "unknown",
        })

df_eda = pd.DataFrame(rows)
print(f"Total videos: {len(df_eda)}")
print(f"\nClass distribution:")
print(df_eda["label_name"].value_counts())
print(f"\nUnique identities: {df_eda['identity'].nunique()}")
print(f"Gender split:")
print(df_eda["gender"].value_counts())

In [ ]:
# CELL 10 │ Phase 2 │ EDA — Generate 7 Figures for Paper
import warnings, glob as _glob
warnings.filterwarnings("ignore")

# Step 1: Compute pixel features if missing
if "pixel_mean" not in df_eda.columns or df_eda["pixel_mean"].isna().all():
    print("Computing pixel features from MVF frame files...")

    def find_frame(vid_path):
        parts = vid_path.split("/")
        # /content/dataset/FakeAVCeleb_v1.2/CATEGORY/ETH/GENDER/ID/video.mp4
        if len(parts) < 5:
            return None
        category  = parts[-5]
        ethnicity = parts[-4]
        gender    = parts[-3]
        identity  = parts[-2]
        pat = (f"/content/MVF_data/face/{category}/"
               f"{ethnicity}/{gender}/{identity}/*.jpg")
        matches = _glob.glob(pat)
        return matches[0] if matches else None

    pm, ps, ws, hs = [], [], [], []
    found = 0
    for i, vp in enumerate(df_eda["video_path"]):
        fp  = find_frame(vp)
        img = cv2.imread(fp) if fp else None
        if img is not None:
            g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            pm.append(float(g.mean()))
            ps.append(float(g.std()))
            hs.append(img.shape[0])
            ws.append(img.shape[1])
            found += 1
        else:
            pm.append(float("nan"))
            ps.append(float("nan"))
            hs.append(float("nan"))
            ws.append(float("nan"))
        if (i + 1) % 3000 == 0:
            print(f"  {i+1}/{len(df_eda)} | found: {found}")

    df_eda["pixel_mean"] = pm
    df_eda["pixel_std"]  = ps
    df_eda["width"]      = ws
    df_eda["height"]     = hs
    print(f"✓ Pixel features done: {found}/{len(df_eda)} valid "
          f"({found/len(df_eda)*100:.1f}%)")
else:
    valid = df_eda["pixel_mean"].notna().sum()
    print(f"✓ Pixel features already in df_eda: {valid} valid rows")

# Step 2: Build plot_df
plot_df = df_eda.dropna(
    subset=["pixel_mean","pixel_std","width","height"]).copy()
print(f"plot_df: {len(plot_df)} rows with valid pixel features")

if len(plot_df) == 0:
    print("⚠ No valid pixel features found.")
    print("  Skipping EDA figures — proceed to Cell 11.")
else:
    features     = ["pixel_mean","pixel_std","width","height"]
    label_names  = {0:"real",1:"fake_audio",
                    2:"fake_video",3:"fake_both"}
    colors       = ["#2ecc71","#e74c3c","#3498db","#9b59b6"]

    # Fig 1 — Histograms
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle("Fig 1: Histograms of Key Features", fontsize=13)
    for ax, feat in zip(axes.flatten(), features):
        ax.hist(plot_df[feat].dropna(), bins=40,
                color="#4C72B0", edgecolor="white")
        ax.set_title(feat)
        ax.set_xlabel("Value")
        ax.set_ylabel("Count")
    plt.tight_layout()
    plt.savefig("/content/report_figures/fig1_histograms.png", dpi=150)
    plt.show(); plt.close()
    print("✓ Fig 1 saved")

    # Fig 2 — Boxplots
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle("Fig 2: Boxplots of Key Features", fontsize=13)
    for ax, feat in zip(axes.flatten(), features):
        ax.boxplot(plot_df[feat].dropna(), patch_artist=True,
                   boxprops=dict(facecolor="#4C72B0"),
                   medianprops=dict(color="orange", linewidth=2))
        ax.set_title(feat)
        ax.set_xticks([])
    plt.tight_layout()
    plt.savefig("/content/report_figures/fig2_boxplots.png", dpi=150)
    plt.show(); plt.close()
    print("✓ Fig 2 saved")

    # Fig 3 — Scatter feature vs label
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle("Fig 3: Feature vs Label (Scatter)", fontsize=13)
    for ax, feat in zip(axes.flatten(), features):
        for lbl, name in label_names.items():
            sub = plot_df[plot_df["label"] == lbl]
            ax.scatter(sub[feat], sub["label"],
                       alpha=0.3, s=10,
                       color=colors[lbl], label=name)
        ax.set_xlabel(feat)
        ax.set_ylabel("Label")
        ax.set_yticks([0, 1, 2, 3])
        ax.set_yticklabels(["real","fake_audio",
                             "fake_video","fake_both"])
        ax.legend(fontsize=7, markerscale=2)
    plt.tight_layout()
    plt.savefig("/content/report_figures/fig3_scatter.png", dpi=150)
    plt.show(); plt.close()
    print("✓ Fig 3 saved")

    # Fig 4 — Correlation matrix
    fig, ax = plt.subplots(figsize=(7, 5))
    corr = plot_df[features + ["label"]].corr()
    im   = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr.columns)))
    ax.set_yticks(range(len(corr.columns)))
    ax.set_xticklabels(corr.columns, rotation=45, ha="right")
    ax.set_yticklabels(corr.columns)
    plt.colorbar(im, ax=ax)
    for i in range(len(corr)):
        for j in range(len(corr)):
            ax.text(j, i, f"{corr.iloc[i,j]:.2f}",
                    ha="center", va="center", fontsize=9,
                    color="white" if abs(corr.iloc[i,j])>0.5
                    else "black")
    ax.set_title("Fig 4: Correlation Matrix — Features and Label")
    plt.tight_layout()
    plt.savefig("/content/report_figures/fig4_correlation.png", dpi=150)
    plt.show(); plt.close()
    print("✓ Fig 4 saved")

    # Fig 5 — Class balance
    full_counts = df_eda["label"].value_counts().sort_index()
    fig, axes   = plt.subplots(1, 2, figsize=(12, 5))
    bars = axes[0].bar(
        [label_names[i] for i in full_counts.index],
        full_counts.values, color=colors, edgecolor="white")
    axes[0].set_title(
        f"Fig 5: Class Distribution — Full Dataset "
        f"({len(df_eda):,} videos)")
    axes[0].set_ylabel("Count")
    for bar, val in zip(bars, full_counts.values):
        pct = val / len(df_eda) * 100
        axes[0].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 50,
            f"{val}\n({pct:.1f}%)",
            ha="center", fontsize=9)
    ratios = [full_counts[i] / full_counts.min() for i in range(4)]
    axes[1].barh([label_names[i] for i in range(4)],
                 ratios, color=colors)
    axes[1].set_title("Imbalance Ratio vs Minority Class")
    axes[1].set_xlabel("Ratio (× minority count)")
    for i, r in enumerate(ratios):
        axes[1].text(r + 0.2, i, f"{r:.1f}×",
                     va="center", fontsize=9)
    plt.tight_layout()
    plt.savefig("/content/report_figures/fig5_class_balance.png",
                dpi=150)
    plt.show(); plt.close()
    print("✓ Fig 5 saved")

    # Fig 6 — Per-class feature distributions
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle("Fig 6: Per-Class Feature Distributions",
                 fontsize=13)
    for ax, feat in zip(axes.flatten(), features):
        for lbl, name in label_names.items():
            sub = plot_df[plot_df["label"] == lbl][feat].dropna()
            if len(sub) > 0:
                ax.hist(sub, bins=30, alpha=0.5,
                        label=name, color=colors[lbl])
        ax.set_title(feat)
        ax.set_xlabel("Value")
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig("/content/report_figures/fig6_per_class.png",
                dpi=150)
    plt.show(); plt.close()
    print("✓ Fig 6 saved")

    # Fig 7 — Ethnicity + Gender
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    eth = df_eda["ethnicity"].value_counts()
    axes[0].barh(eth.index, eth.values, color="#4C72B0")
    axes[0].set_title("Fig 7: Samples per Ethnicity")
    for i, (idx, val) in enumerate(eth.items()):
        axes[0].text(val + 50, i,
                     f"{val/len(df_eda)*100:.1f}%",
                     va="center", fontsize=9)
    gen = df_eda["gender"].value_counts()
    axes[1].pie(gen.values, labels=gen.index,
                autopct="%1.1f%%",
                colors=["#4C72B0","#DD8452"],
                startangle=90)
    axes[1].set_title("Gender Distribution")
    plt.tight_layout()
    plt.savefig("/content/report_figures/fig7_ethnicity_gender.png",
                dpi=150)
    plt.show(); plt.close()
    print("✓ Fig 7 saved")

    print("\n✓ All 7 EDA figures saved to /content/report_figures/")

In [ ]:
# CELL 11 │ Phase 3 │ Baseline 1 — Load Pixel Arrays
img_size  = 64
transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
])

def load_face_data(list_file, max_samples=None):
    X, y = [], []
    with open(list_file) as f:
        lines = f.readlines()
    if max_samples:
        lines = lines[:max_samples]
    for line in lines:
        line = line.strip()
        if not line:
            continue
        img_path, label = line.rsplit(" ", 1)
        try:
            img = Image.open(img_path).convert("RGB")
            vec = transform(img).numpy().reshape(-1)
            X.append(vec)
            y.append(int(label))
        except:
            continue
    return (np.array(X, dtype=np.float32),
            np.array(y, dtype=np.int64))

# load all 3 splits
print("Loading pixel data...")
X_pix_train, y_pix_train = load_face_data(train_list)
X_pix_val,   y_pix_val   = load_face_data(val_list)

# ADD THIS test split
X_pix_test,  y_pix_test  = load_face_data(test_list)

print(f"X_pix_train: {X_pix_train.shape} | {np.bincount(y_pix_train, minlength=4)}")
print(f"X_pix_val:   {X_pix_val.shape}   | {np.bincount(y_pix_val,   minlength=4)}")
print(f"X_pix_test:  {X_pix_test.shape}  | {np.bincount(y_pix_test,  minlength=4)}")

# verify all 4 classes in each split
for name, y in [("train", y_pix_train),
                ("val",   y_pix_val),
                ("test",  y_pix_test)]:
    counts = np.bincount(y, minlength=4).tolist()
    missing = [i for i, c in enumerate(counts) if c == 0]
    if missing:
        print(f"  {name}: ✗ MISSING classes {missing} — counts: {counts}")
    else:
        print(f"  {name}: ✓ all 4 classes present {counts}")
# build DataLoaders for all 3 splits
pix_train_ds = TensorDataset(
    torch.tensor(X_pix_train),
    torch.tensor(y_pix_train))
pix_val_ds   = TensorDataset(
    torch.tensor(X_pix_val),
    torch.tensor(y_pix_val))
pix_test_ds  = TensorDataset(
    torch.tensor(X_pix_test),
    torch.tensor(y_pix_test))

# weighted sampler for train
pix_class_counts  = np.bincount(y_pix_train)
pix_class_weights = len(y_pix_train) / (
    len(pix_class_counts) *
    pix_class_counts.astype(float))
pix_sample_weights = torch.tensor(
    [pix_class_weights[y] for y in y_pix_train])
pix_sampler = WeightedRandomSampler(
    pix_sample_weights,
    len(pix_sample_weights),
    replacement=True)

train_loader_pix = DataLoader(
    pix_train_ds, batch_size=64, sampler=pix_sampler)
val_loader_pix   = DataLoader(
    pix_val_ds,   batch_size=64, shuffle=False)
test_loader_pix  = DataLoader(
    pix_test_ds,  batch_size=64, shuffle=False)

print(f"\nPixel loaders ready:")
print(f"  train_loader_pix: {len(train_loader_pix)} batches")
print(f"  val_loader_pix:   {len(val_loader_pix)} batches")
print(f"  test_loader_pix:  {len(test_loader_pix)} batches ← final eval only")
print("⚠ test_loader_pix locked until all training complete")
print(f"✓ Pixel train: {X_pix_train.shape[0]:,} samples")
print(f"  Val:  {X_pix_val.shape[0]:,} | Test: {X_pix_test.shape[0]:,}")
print("  Embedding split verification will run after Phase 5 loads.")


In [ ]:
# CELL 12 │ Phase 3 │ Baseline 1 — Define SimpleMLP + FocalLoss
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        # get true class log-probabilities
        logp   = F.log_softmax(inputs, dim=1)
        p      = torch.exp(logp)

        # gather values for true class only
        logp_t = logp.gather(
            1, targets.unsqueeze(1)).squeeze(1)
        p_t    = p.gather(
            1, targets.unsqueeze(1)).squeeze(1)

        # focal modulating factor
        focal  = (1 - p_t) ** self.gamma

        # base focal loss
        loss   = -focal * logp_t

        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            loss    = alpha_t * loss

        return loss.mean()
class SimpleMLP(nn.Module):
    def __init__(self, input_dim=12288, num_classes=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes))
    def forward(self, x):
        return self.net(x)

# class weights
class_counts  = np.bincount(y_pix_train)
class_weights = len(y_pix_train) / (
    len(class_counts) * class_counts.astype(float))
class_weights[0] *= 1.5   # real boost
class_weights[1] *= 1.5   # fake_audio boost

weights_pix = torch.tensor(
    class_weights, dtype=torch.float32).to(device)

print(f"\nClass counts:  {class_counts}")
print(f"Class weights: {class_weights.round(2)}")
assert class_weights[0] < 30, \
    f"Real weight too high: {class_weights[0]:.1f}"
print("✓ Weights safe")

# WeightedRandomSampler for train
sample_weights = torch.tensor(
    [class_weights[y] for y in y_pix_train])
sampler_pix    = WeightedRandomSampler(
    sample_weights, len(sample_weights), replacement=True)

# DataLoaders — all 3 splits
train_ds_pix = TensorDataset(
    torch.tensor(X_pix_train),
    torch.tensor(y_pix_train))
val_ds_pix   = TensorDataset(
    torch.tensor(X_pix_val),
    torch.tensor(y_pix_val))
test_ds_pix  = TensorDataset(
    torch.tensor(X_pix_test),
    torch.tensor(y_pix_test))

train_loader_pix = DataLoader(
    train_ds_pix, batch_size=64, sampler=sampler_pix)
val_loader_pix   = DataLoader(
    val_ds_pix,   batch_size=64, shuffle=False)
test_loader_pix  = DataLoader(
    test_ds_pix,  batch_size=64, shuffle=False)

# initialise model
pixel_mlp = SimpleMLP(
    input_dim=X_pix_train.shape[1],
    num_classes=4).to(device)
criterion_pix = FocalLoss(alpha=weights_pix, gamma=2.0)
optimizer_pix = optim.Adam(
    pixel_mlp.parameters(),
    lr=1e-3, weight_decay=1e-4)
scheduler_pix = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_pix, T_max=30)

n_params = sum(p.numel() for p in pixel_mlp.parameters()
               if p.requires_grad)
print(f"\nSimpleMLP | params={n_params:,}")
print(f"lr=1e-3 | epochs=30 | FocalLoss gamma=2.0")
print(f"✓ Split: {X_pix_train.shape[0]:,} train | "
      f"{X_pix_val.shape[0]:,} val | "
      f"{X_pix_test.shape[0]:,} test")
print("✓ Ready to train Pixel MLP")


In [ ]:
# CELL 13 │ Phase 3 │ Baseline 1 — Train + Test Pixel MLP

os.makedirs("/content/results", exist_ok=True)
torch.manual_seed(42)
np.random.seed(42)

num_epochs = 30
best_f1_pix, best_epoch_pix = 0, 0

print("="*60)
print("TRAINING PIXEL MLP")
print(f"  train: {len(train_ds_pix)} samples")
print(f"  val:   {len(val_ds_pix)} samples  ← selection only")
print(f"  test:  {len(test_ds_pix)} samples ← locked")
print("="*60)

for epoch in range(num_epochs):
    pixel_mlp.train()
    train_loss = 0
    for xb, yb in train_loader_pix:
        xb, yb = xb.to(device), yb.to(device)
        optimizer_pix.zero_grad()
        loss = criterion_pix(pixel_mlp(xb), yb)
        loss.backward()
        optimizer_pix.step()
        train_loss += loss.item()
    scheduler_pix.step()

    #  val evaluation — model selection only
    pixel_mlp.eval()
    all_p, all_l = [], []
    with torch.no_grad():
        for xb, yb in val_loader_pix:
            p = pixel_mlp(xb.to(device)).argmax(dim=1)
            all_p.extend(p.cpu().numpy())
            all_l.extend(yb.numpy())

    acc = accuracy_score(all_l, all_p)
    bal = balanced_accuracy_score(all_l, all_p)
    f1  = f1_score(all_l, all_p,
                   average='macro', zero_division=0)

    if f1 > best_f1_pix:
        best_f1_pix    = f1
        best_epoch_pix = epoch + 1
        torch.save(pixel_mlp.state_dict(),
                   "/content/results/best_pixel_mlp.pth")

    print(f"Ep {epoch+1:02d}/{num_epochs} | "
          f"Loss:{train_loss/len(train_loader_pix):.4f} | "
          f"Acc:{acc:.4f} | BalAcc:{bal:.4f} | "
          f"F1:{f1:.4f}"
          + (" ← best" if epoch+1==best_epoch_pix else ""))

print(f"\nBest Val F1: {best_f1_pix:.4f} "
      f"at epoch {best_epoch_pix}")

# TEST evaluation — run once after training
print("\n" + "="*60)
print("PIXEL MLP — FINAL TEST EVALUATION")
print("="*60)

pixel_mlp.load_state_dict(
    torch.load("/content/results/best_pixel_mlp.pth"))
pixel_mlp.eval()

all_p, all_l, all_sc = [], [], []
with torch.no_grad():
    for xb, yb in test_loader_pix:   # ← TEST SET
        logits = pixel_mlp(xb.to(device))
        probs  = torch.softmax(logits, dim=1)
        all_p.extend(logits.argmax(1).cpu().numpy())
        all_l.extend(yb.numpy())
        all_sc.extend(probs.cpu().numpy())

all_p  = np.array(all_p)
all_l  = np.array(all_l)
all_sc = np.array(all_sc)

acc  = accuracy_score(all_l, all_p)
bal  = balanced_accuracy_score(all_l, all_p)
f1   = f1_score(all_l, all_p,
                average='macro', zero_division=0)
auc  = roc_auc_score(
    (all_l!=0).astype(int), 1-all_sc[:,0])
cm   = confusion_matrix(all_l, all_p)

print(f"Test samples:      {len(all_l)}")
print(f"Accuracy:          {acc:.4f}")
print(f"Balanced Accuracy: {bal:.4f}")
print(f"Macro F1:          {f1:.4f}  ← report in paper")
print(f"Binary AUC:        {auc:.4f}")
print(classification_report(
    all_l, all_p,
    target_names=["real","fake_audio",
                  "fake_video","fake_both"],
    zero_division=0))
print("Confusion Matrix:")
print(cm)

# Save result
import json, shutil

result = {
    "model":                  "Pixel MLP",
    "val_samples":            int(len(val_ds_pix)),
    "val_macro_f1":           float(best_f1_pix),
    "best_epoch":             int(best_epoch_pix),
    "test_samples":           int(len(all_l)),
    "test_macro_f1":          float(f1),
    "test_binary_auc":        float(auc),
    "test_balanced_accuracy": float(bal),
    "test_accuracy":          float(acc),
    "test_confusion_matrix":  cm.tolist(),
    "evaluation_note":
        "val used for model selection only. "
        "test is held-out identity-disjoint set."
}

with open("/content/results/B1_pixel_mlp.json","w") as fj:
    json.dump(result, fj, indent=2)

# backup to Drive
DRIVE_RESULTS = \
    "/content/drive/MyDrive/TRM_DATASETS/results"
os.makedirs(DRIVE_RESULTS, exist_ok=True)
with open(f"{DRIVE_RESULTS}/B1_pixel_mlp.json", "w") as fj:
    json.dump(result, fj, indent=2)

if os.path.exists("/content/results/best_pixel_mlp.pth"):
    shutil.copy(
        "/content/results/best_pixel_mlp.pth",
        "/content/drive/MyDrive/TRM_DATASETS/trm_checkpoint/best_pixel_mlp.pth")

print(f"\n✓ Saved B1_pixel_mlp.json")
print(f"  Val F1:  {best_f1_pix:.4f} (used for selection)")
print(f"  Test F1: {f1:.4f} (report this in paper)")
DRIVE_PIX = "/content/drive/MyDrive/TRM_DATASETS/pixel_test"
os.makedirs(DRIVE_PIX, exist_ok=True)
np.save(f"{DRIVE_PIX}/X_pix_test_probs.npy", all_sc)  # softmax probs
np.save(f"{DRIVE_PIX}/X_pix_test_labels.npy", all_l)  # true labels
np.save(f"{DRIVE_PIX}/X_pix_test_preds.npy",  all_p)  # predictions
print(f"✓ Test arrays saved: {DRIVE_PIX}/")
print(f"\nFINAL: F1={f1:.4f} | AUC={auc:.4f}")
print("✓ All results saved to Drive")


In [ ]:
# CELL 14 │ Phase 4 │ MVF Baseline — Patch Repo + Write Config
aug_path = "/content/MVF/utils/Augmentation.py"
with open(aug_path,"r") as f:
    lines = f.readlines()
fixed = [
    "from datasets.transforms_ss import *\n",
    "try:\n",
    "    from randaugment import RandAugment\n",
    "except Exception:\n",
    "    RandAugment = None\n"
]
for i,l in enumerate(lines):
    if l.startswith("class ") or l.startswith("def "):
        fixed.extend(lines[i:])
        break
with open(aug_path,"w") as f:
    f.writelines(fixed)
print("✓ MVF repo patched")

# Write finetune_full.yaml — 4-class, 30 epochs
with open("/content/MVF/configs/FakeAVCeleb/finetune.yaml","r") as f:
    cfg = yaml.safe_load(f)

cfg["resume"]  = ""
cfg["pretrain"]["split"]    = False
cfg["pretrain"]["pretrain"] = ""

# train and val paths
cfg["data"]["train_face_list"] = "/content/MVF_data/train_frame.txt"
cfg["data"]["val_face_list"]   = "/content/MVF_data/val_frame.txt"

# ADD: test path
cfg["data"]["test_face_list"]  = "/content/MVF_data/test_frame.txt"

cfg["data"]["label_list"]      = "/content/MVF_data/FakeAVCeleb.csv"
cfg["data"]["batch_size"]      = 8
cfg["data"]["workers"]         = 0
cfg["data"]["neg_sample"]      = 1
cfg["data"]["randaug"]["N"]    = 0
cfg["data"]["randaug"]["M"]    = 0
cfg["data"]["exp_name"]        = "FakeAVCeleb_4class_30ep"
cfg["solver"]["lr"]            = 1e-4
cfg["solver"]["epochs"]        = 30

new_cfg = "/content/MVF/configs/FakeAVCeleb/finetune_full.yaml"
with open(new_cfg,"w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print("✓ finetune_full.yaml saved")
print(f"  train: /content/MVF_data/train_frame.txt")
print(f"  val:   /content/MVF_data/val_frame.txt  ← model selection")
print(f"  test:  /content/MVF_data/test_frame.txt ← final report only")
print(f"  epochs: 30 | lr: 1e-4 | 4-class")

# verify all 3 split files exist (each contains 4 classes)
import os
for fname in ["train_frame.txt","val_frame.txt","test_frame.txt"]:
    path = f"/content/MVF_data/{fname}"
    exists = os.path.exists(path)
    lines  = sum(1 for _ in open(path)) if exists else 0
    print(f"  {'✓' if exists else '✗'} {fname}: {lines} samples")

In [ ]:
# CELL 15 │ Phase 4 │ MVF Baseline — Train (40–80 min on T4)

import os, sys
os.environ["WANDB_MODE"] = "disabled"
os.environ["PYTHONPATH"] = "/content/MVF:" + os.environ.get("PYTHONPATH","")
%cd /content/MVF
!{sys.executable} finetune_deepfake.py \
    --config /content/MVF/configs/FakeAVCeleb/finetune_full.yaml \
    --log_time run30ep

In [ ]:
# CELL 16 │ Phase 4 │ MVF Baseline — Find Saved Checkpoint
import os, glob

# find MVF checkpoint
ckpt_patterns = [
    "/content/MVF/checkpoints/**/*.pth",
    "/content/MVF/work_dirs/**/*.pth",
    "/content/MVF/output/**/*.pth",
    "/content/MVF/exp/**/*.pth",
]

found_ckpts = []
for pattern in ckpt_patterns:
    found_ckpts.extend(glob.glob(pattern, recursive=True))

if found_ckpts:
    print(f"✓ Found {len(found_ckpts)} MVF checkpoints:")
    for c in sorted(found_ckpts)[-3:]:  # show last 3
        size = os.path.getsize(c)/1024/1024
        print(f"  {c} ({size:.1f} MB)")
    best_ckpt = sorted(found_ckpts)[-1]
    print(f"\nBest checkpoint: {best_ckpt}")
else:
    print("✗ No checkpoints found")
    print("Check MVF training output above for errors")

In [ ]:
# CELL 17 │ Phase 4 │ MVF Baseline — Patch test.py for Binary AUC

import os, re, textwrap, shutil
test_py   = "/content/MVF/test.py"
backup_py = "/content/MVF/test.py.bak_before_binary_patch"
assert os.path.exists(test_py), f"Missing: {test_py} — run Cell 2 first"

if not os.path.exists(backup_py):
    shutil.copy(test_py, backup_py)
    print("✓ Backup created:", backup_py)
else:
    print("✓ Backup already exists — restoring original before re-patch")
    shutil.copy(backup_py, test_py)

with open(test_py, "r") as f:
    src = f.read()

new_func = textwrap.dedent(r'''
def validate_on_fake_dataset(epoch, val_loader, device, voice_model, face_model, config):
    import os, json
    import numpy as np
    import sklearn.metrics

    voice_model.eval()
    face_model.eval()

    sim_scores = []
    y_true_bin = []

    with torch.no_grad():
        for iii, match_pair in enumerate(tqdm(val_loader)):
            image   = match_pair['img']
            mel     = match_pair['mel']
            list_id = match_pair['label']

            image = image.view((-1, 3) + image.size()[-2:])
            b, c, h, w = image.size()
            image_features = face_model(image.to(device).view(-1, c, h, w))

            voice = mel.view((-1, 1) + mel.size()[-2:])
            b, c, h, w = voice.size()
            voice_features = voice_model(voice.to(device).view(-1, c, h, w))

            image_features /= image_features.norm(dim=-1, keepdim=True)
            voice_features /= voice_features.norm(dim=-1, keepdim=True)

            sim = torch.sum(voice_features * image_features, dim=1).cpu().numpy()
            sim_scores.extend(sim.tolist())

            for label in list_id:
                y_true_bin.append(0 if int(label.cpu()) == 0 else 1)

    sim_scores = np.asarray(sim_scores, dtype=np.float32)
    y_true_bin = np.asarray(y_true_bin, dtype=np.int64)
    fake_scores = -sim_scores
    y_pred_bin  = (sim_scores <= 0.0).astype(np.int64)

    auc     = sklearn.metrics.roc_auc_score(y_true_bin, fake_scores)
    acc     = sklearn.metrics.accuracy_score(y_true_bin, y_pred_bin)
    bal_acc = sklearn.metrics.balanced_accuracy_score(y_true_bin, y_pred_bin)
    f1      = sklearn.metrics.f1_score(y_true_bin, y_pred_bin, average="binary", zero_division=0)
    cm      = sklearn.metrics.confusion_matrix(y_true_bin, y_pred_bin)
    report  = sklearn.metrics.classification_report(
        y_true_bin, y_pred_bin,
        target_names=["consistent", "inconsistent"],
        zero_division=0
    )
    fpr, tpr, _ = sklearn.metrics.roc_curve(y_true_bin, fake_scores)

    try:
        epochs_total = config.solver.epochs
    except Exception:
        epochs_total = config['solver']['epochs']

    print('Epoch: [{}/{}]: auc: {:.4f}, acc: {:.4f}, bal_acc: {:.4f}, f1: {:.4f}'.format(
        epoch, epochs_total, auc, acc, bal_acc, f1))
    print("Binary confusion matrix:")
    print(cm)
    print(report)

    os.makedirs("/content/results", exist_ok=True)
    os.makedirs("/content/drive/MyDrive/TRM_DATASETS/results", exist_ok=True)

    result = {
        "model":                  "MVF (published/reference, binary consistency baseline)",
        "status":                 "binary-only",
        "test_samples":           int(len(y_true_bin)),
        "test_accuracy":          float(acc),
        "test_balanced_accuracy": float(bal_acc),
        "test_binary_f1":         float(f1),
        "test_binary_auc":        float(auc),
        "test_confusion_matrix":  cm.tolist(),
        "roc_curve":              {"fpr": fpr.tolist(), "tpr": tpr.tolist()},
        "evaluation_note": (
            "True MVF checkpoint evaluated as binary face-voice consistency model. "
            "Outputs similarity score, not 4-class subtype logits."
        )
    }

    for path in ["/content/results/B3_mvf.json",
                 "/content/drive/MyDrive/TRM_DATASETS/results/B3_mvf.json"]:
        with open(path, "w") as f:
            json.dump(result, f, indent=2)

    print("✓ Saved: B3_mvf.json locally and to Drive")
    return auc
''')

pattern = r"def validate_on_fake_dataset\(.*?\):.*?return auc"
new_src, n = re.subn(pattern, new_func.strip(), src, flags=re.S)

if n != 1:
    raise RuntimeError(
        f"Expected to replace 1 function, replaced {n}. "
        "Check MVF test.py for validate_on_fake_dataset signature."
    )

with open(test_py, "w") as f:
    f.write(new_src)

print(f"✓ Patched: {test_py}")
print(f"  Replaced {n} function(s)")
print("  validate_on_fake_dataset → saves binary AUC to B3_mvf.json")

In [ ]:
# CELL 18 │ Phase 4 │ MVF Baseline — Final Test Evaluation
import os, sys, re, json, yaml, glob, shutil, subprocess

MVF_ROOT     = "/content/MVF"
BASE_CONFIG  = "/content/MVF/configs/FakeAVCeleb/finetune_full.yaml"
TEST_CONFIG  = "/content/MVF/configs/FakeAVCeleb/test_only.yaml"
TEST_LIST    = "/content/MVF_data/test_frame.txt"
TRAIN_LIST   = "/content/MVF_data/train_frame.txt"
LABEL_CSV    = "/content/MVF_data/FakeAVCeleb.csv"
RESULTS_DIR  = "/content/results"
DRIVE_RESULTS= "/content/drive/MyDrive/TRM_DATASETS/results"
DRIVE_CKPT   = "/content/drive/MyDrive/TRM_DATASETS/trm_checkpoint"

os.makedirs(RESULTS_DIR,   exist_ok=True)
os.makedirs(DRIVE_RESULTS, exist_ok=True)
os.makedirs(DRIVE_CKPT,    exist_ok=True)

# 1: Find checkpoint
mvf_ckpt = ("/content/MVF/exp/Tranf_face/Transf/FakeAVCeleb/"
            "FakeAVCeleb_4class_30ep/run30ep/model_best.pt")
if not os.path.exists(mvf_ckpt):
    found = sorted(glob.glob(
        "/content/MVF/exp/**/*.pt", recursive=True))
    if not found:
        raise FileNotFoundError("MVF checkpoint not found")
    mvf_ckpt = found[-1]

print(f"✓ Checkpoint: {mvf_ckpt}")
print(f"  Size: {os.path.getsize(mvf_ckpt)/1024/1024:.1f} MB")
shutil.copy(mvf_ckpt, f"{DRIVE_CKPT}/best_mvf.pt")
print("✓ Backed up to Drive")

#  2: Check what evaluate flag does in MVF script
with open(f"{MVF_ROOT}/finetune_deepfake.py","r") as f:
    script = f.read()

# find how evaluate is used
eval_lines = [l.strip() for l in script.split("\n")
              if "evaluate" in l.lower()][:20]
print("\nMVF script evaluate usage:")
for l in eval_lines:
    print(f"  {l}")

#  3: Build strict eval-only config
with open(BASE_CONFIG,"r") as f:
    cfg = yaml.safe_load(f)

cfg["data"]["val_face_list"]   = TEST_LIST
cfg["data"]["test_face_list"]  = TEST_LIST
cfg["data"]["train_face_list"] = TRAIN_LIST
cfg["data"]["label_list"]      = LABEL_CSV
cfg["data"]["exp_name"]        = "MVF_test_eval"
cfg["resume"]                  = mvf_ckpt

#  KEY FIXES
cfg["solver"]["evaluate"] = True   # force eval mode
# epochs=1 with evaluate=True means:
# load checkpoint → run val loop once → print Testing: → exit

with open(TEST_CONFIG,"w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f"\n✓ test_only.yaml:")
print(f"  evaluate:      True  ← forces eval mode")
print(f"  epochs:        1     ← single pass only")
print(f"  val_face_list: test_frame.txt")
print(f"  resume:        model_best.pt")

#  4: Run MVF evaluation
env = os.environ.copy()
env["WANDB_MODE"] = "disabled"
env["PYTHONPATH"] = f"{MVF_ROOT}:" + env.get("PYTHONPATH","")

cmd = [sys.executable, "finetune_deepfake.py",
       "--config", TEST_CONFIG,
       "--log_time", "test_eval"]

print("\n" + "="*60)
print("RUNNING TRUE MVF TEST EVALUATION")
print("Expected output: Testing: X.XXXX/X.XXXX")
print("="*60)

proc = subprocess.run(
    cmd, cwd=MVF_ROOT, env=env,
    capture_output=True, text=True)

full_log = proc.stdout + "\n" + proc.stderr

# save log
log_path = f"{RESULTS_DIR}/mvf_test_eval.log"
with open(log_path,"w") as f:
    f.write(full_log)

print(f"Return code: {proc.returncode}")
print("\n----- LOG TAIL (last 50 lines) -----")
print("\n".join(full_log.splitlines()[-50:]))
print("-------------------------------------")

#  5: Parse result
m = re.search(
    r"Testing:\s*([0-9]*\.?[0-9]+)\s*/\s*([0-9]*\.?[0-9]+)",
    full_log)

# try Epoch: [1/1]: auc: X.XXXX
if not m:
    m2 = re.search(
        r"Epoch.*auc:\s*([0-9]*\.?[0-9]+)",
        full_log)
    if m2:
        test_auc = float(m2.group(1))
        print(f"\n✓ Parsed from Epoch line: AUC={test_auc:.4f}")
    else:
        # print full log for manual inspection
        print("\nFULL LOG:")
        print(full_log)
        raise RuntimeError(
            "No test metric found in log.\n"
            "Check full log above for Testing: or auc: line.\n"
            "DO NOT save result until real metric found.")
else:
    test_auc     = float(m.group(1))
    second_metric = float(m.group(2))
    print(f"\n✓ Parsed Testing: AUC={test_auc:.4f} | "
          f"second={second_metric:.4f}")

#  6: Save ONLY if real result found
mvf_json = "/content/results/B3_mvf.json"

if os.path.exists(mvf_json):
    with open(mvf_json, "r") as f:
        result = json.load(f)

    print("\n✓ Found patched MVF result from /content/MVF/test.py")
    for k, v in result.items():
        if k == "roc_curve":
            print("roc_curve: saved")
        else:
            print(f"{k}: {v}")
else:
    print("\n⚠ Patched MVF JSON not found.")
    print("This means /content/MVF/test.py was not patched correctly,")
    print("or the evaluation did not reach validate_on_fake_dataset().")

In [ ]:
# CELL 19 │ Phase 4 │ MVF Baseline — Load + Verify Saved Results
path = "/content/results/B3_mvf.json"
assert os.path.exists(path), "B3_mvf.json not found."

with open(path, "r") as f:
    d = json.load(f)

print("MVF saved result:")
for k, v in d.items():
    if k == "roc_curve":
        print("roc_curve: saved")
    else:
        print(f"{k}: {v}")


In [ ]:
# CELL 20 │ Phase 4 │ MVF Baseline — Print Binary Score Summary

import json

with open("/content/results/B3_mvf.json", "r") as f:
    d = json.load(f)

print("=" * 60)
print("TRUE MVF BINARY SCORES")
print("=" * 60)
print(f"Model:               {d['model']}")
print(f"Status:              {d['status']}")
print(f"Test samples:        {d['test_samples']}")
print(f"Binary AUC:          {d['test_binary_auc']:.4f}")
print(f"Binary Accuracy:     {d['test_accuracy']:.4f}")
print(f"Balanced Accuracy:   {d['test_balanced_accuracy']:.4f}")
print(f"Binary F1:           {d['test_binary_f1']:.4f}")
print("Confusion Matrix:")
print(d["test_confusion_matrix"])

In [ ]:
# CELL 21 │ Phase 5 │ Feature Extraction — Load Encoders + Define Functions
# ArcFace (InsightFace) → 512-dim face embeddings
# ECAPA-TDNN (SpeechBrain) → 192-dim voice embeddings

import numpy as np
import torch
import cv2
import torchaudio
from insightface.app import FaceAnalysis
try:
    from speechbrain.inference.speaker import SpeakerRecognition
except ImportError:
    from speechbrain.pretrained import SpeakerRecognition

# Load ArcFace
face_app = FaceAnalysis(
    name="buffalo_l",
    providers=["CUDAExecutionProvider",
               "CPUExecutionProvider"])
face_app.prepare(ctx_id=0, det_size=(320, 320))
print("✓ ArcFace loaded (buffalo_l, 512-dim)")

# Load ECAPA-TDNN
voice_model = SpeakerRecognition.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="pretrained/ecapa",
    run_opts={"device": "cpu"})
print("✓ ECAPA-TDNN loaded (192-dim)")

#  Define get_face_embedding
def get_face_embedding(img_path):
    """
    Extract 512-dim ArcFace embedding.
    4-strategy fallback if face not detected.
    Returns zeros(512) if all strategies fail.
    """
    img = cv2.imread(img_path)
    if img is None:
        return np.zeros(512, dtype=np.float32)

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Strategy 1: default size
    faces = face_app.get(img_rgb)
    if faces:
        return faces[0].normed_embedding.astype(
            np.float32)

    # Strategy 2: resize larger
    faces = face_app.get(
        cv2.resize(img_rgb, (640, 640)))
    if faces:
        return faces[0].normed_embedding.astype(
            np.float32)

    # Strategy 3: resize smaller
    faces = face_app.get(
        cv2.resize(img_rgb, (160, 160)))
    if faces:
        return faces[0].normed_embedding.astype(
            np.float32)

    # Strategy 4: fail → zeros
    return np.zeros(512, dtype=np.float32)

#  Define get_voice_embedding
def get_voice_embedding(wav_path):
    """
    Extract 192-dim ECAPA-TDNN embedding.
    Returns zeros(192) if extraction fails.
    """
    try:
        signal, fs = torchaudio.load(wav_path)

        # resample to 16kHz
        if fs != 16000:
            signal = torchaudio.transforms.Resample(
                orig_freq=fs,
                new_freq=16000)(signal)

        # mono
        if signal.shape[0] > 1:
            signal = signal.mean(dim=0, keepdim=True)

        # minimum length
        if signal.shape[1] < 1600:
            signal = signal.repeat(
                1, 1600 // signal.shape[1] + 1)
            signal = signal[:, :1600]

        with torch.no_grad():
            emb = voice_model.encode_batch(signal)
        return emb.squeeze().cpu().numpy().astype(
            np.float32)

    except Exception:
        return np.zeros(192, dtype=np.float32)

#  Verify both functions work
print("\nVerifying embedding functions...")

try:
    # find a test image from MVF data
    import os
    test_img = None
    test_wav = None

    with open("/content/MVF_data/test_frame.txt") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 2:
                img = parts[0]
                wav = img.replace(
                    "/face/","/voice/"
                ).replace(".jpg",".wav")
                if (os.path.exists(img) and
                        os.path.exists(wav)):
                    test_img = img
                    test_wav = wav
                    break

    if test_img:
        face_emb  = get_face_embedding(test_img)
        voice_emb = get_voice_embedding(test_wav)

        print(f"✓ face_emb:  {face_emb.shape} | "
              f"zeros={np.all(face_emb==0)}")
        print(f"✓ voice_emb: {voice_emb.shape}")

        if np.all(face_emb == 0):
            print("⚠ Face embedding is zeros — "
                  "face not detected in test image")
            print("  This is OK if rare — "
                  "extraction loop handles it")
        else:
            print("✓ Both functions working correctly")
    else:
        print("⚠ No test image found — "
              "check MVF_data/test_frame.txt exists")

except Exception as e:
    print(f"✗ Verification failed: {e}")
    print("Check InsightFace and SpeechBrain installed")
# load_embedding_data — REQUIRED by the aligned embedding cell
def load_embedding_data(list_file, max_samples=None):
    """
    Read a txt file of (img_path, label) pairs.
    Extract ArcFace + ECAPA embeddings → 704-dim vectors.
    Saves progress every 500 samples.
    """
    X, y, nz = [], [], 0

    with open(list_file) as f:
        lines = f.readlines()
    if max_samples:
        lines = lines[:max_samples]

    for i, line in enumerate(lines):
        parts = line.strip().split()
        if len(parts) != 2:
            continue

        img_path = parts[0]
        label    = int(parts[1])
        wav_path = (img_path
                    .replace("/MVF_data/face/", "/MVF_data/voice/")
                    .replace(".jpg", ".wav"))

        fe = get_face_embedding(img_path)    # 512-dim normed
        ve = get_voice_embedding(wav_path)   # 192-dim

        if np.all(fe == 0):
            nz += 1

        X.append(np.concatenate([fe, ve]))   # 704-dim
        y.append(label)

        if (i + 1) % 500 == 0:
            print(f"  {i+1}/{len(lines)} | "
                  f"zero face: {nz/(i+1)*100:.1f}%")

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int64)

    zero_rate = np.mean(np.all(X[:, :512] == 0, axis=1)) * 100
    print(f"DONE — zero face: {zero_rate:.1f}%  (target: <10%)")
    assert zero_rate < 10.0, \
        f"❌ Too many zero faces: {zero_rate:.1f}% — check ArcFace extraction"

    return X, y

print("  get_face_embedding()   → 512-dim L2-normalized ArcFace")
print("  get_voice_embedding()  → 192-dim ECAPA-TDNN")
print("  load_embedding_data()  → 704-dim concat per sample")

In [ ]:
# CELL 22 │ Phase 5 │ Extract or Load Aligned Embeddings + Build DataLoaders

import os
import numpy as np
import torch
from torch.utils.data import (
    TensorDataset, DataLoader, WeightedRandomSampler)

save_dir   = "/content/drive/MyDrive/TRM_DATASETS/embeddings_mvf_3way"
train_list = "/content/MVF_data/train_frame.txt"
val_list   = "/content/MVF_data/val_frame.txt"
test_list  = "/content/MVF_data/test_frame.txt"

os.makedirs(save_dir, exist_ok=True)

xtrain_path = f"{save_dir}/X_train.npy"
ytrain_path = f"{save_dir}/y_train.npy"
xval_path   = f"{save_dir}/X_val.npy"
yval_path   = f"{save_dir}/y_val.npy"
xtest_path  = f"{save_dir}/X_test.npy"
ytest_path  = f"{save_dir}/y_test.npy"

need_extract = not all(os.path.exists(p) for p in [
    xtrain_path, ytrain_path, xval_path,
    yval_path,   xtest_path,  ytest_path])

if need_extract:
    print("Extracting embeddings from MVF txt files (runs once)...")
    print("Expected runtime: ~60 minutes")
    X_train, y_train = load_embedding_data(train_list)
    X_val,   y_val   = load_embedding_data(val_list)
    X_test,  y_test  = load_embedding_data(test_list)
    np.save(xtrain_path, X_train); np.save(ytrain_path, y_train)
    np.save(xval_path,   X_val);   np.save(yval_path,   y_val)
    np.save(xtest_path,  X_test);  np.save(ytest_path,  y_test)
    print("✓ Saved to Drive")
else:
    print("Loading aligned embeddings from Drive...")
    X_train = np.load(xtrain_path); y_train = np.load(ytrain_path)
    X_val   = np.load(xval_path);   y_val   = np.load(yval_path)
    X_test  = np.load(xtest_path);  y_test  = np.load(ytest_path)

print(f"\n── Loaded arrays ──")
print(f"X_train: {X_train.shape} | {np.bincount(y_train, minlength=4)}")
print(f"X_val:   {X_val.shape}   | {np.bincount(y_val,   minlength=4)}")
print(f"X_test:  {X_test.shape}  | {np.bincount(y_test,  minlength=4)}")

def count_lines(path):
    with open(path) as f:
        return sum(1 for line in f if line.strip())

n_train = count_lines(train_list)
n_val   = count_lines(val_list)
n_test  = count_lines(test_list)

assert X_train.shape[0] == n_train, \
    f"❌ Train mismatch: {X_train.shape[0]} vs {n_train}"
assert X_val.shape[0]   == n_val, \
    f"❌ Val mismatch: {X_val.shape[0]} vs {n_val}"
assert X_test.shape[0]  == n_test, \
    f"❌ Test mismatch: {X_test.shape[0]} vs {n_test}"
assert X_train.shape[1] == 704, \
    f"❌ Wrong feature dim: {X_train.shape[1]}, expected 704"
print(f"✓ Counts match txt files: {n_train} / {n_val} / {n_test}")

for name, y in [("train", y_train),
                ("val",   y_val),
                ("test",  y_test)]:
    counts = np.bincount(y, minlength=4).tolist()
    assert all(c > 0 for c in counts), \
        f"❌ {name} missing a class: {counts}"
    print(f"✓ {name}: all 4 classes present {counts}")

nz_tr = np.mean(np.all(X_train[:, :512] == 0, axis=1)) * 100
nz_va = np.mean(np.all(X_val[:,   :512] == 0, axis=1)) * 100
nz_te = np.mean(np.all(X_test[:,  :512] == 0, axis=1)) * 100
print(f"\nZero-face — train:{nz_tr:.1f}% | val:{nz_va:.1f}% | test:{nz_te:.1f}%")
assert nz_tr < 15, f"❌ Too many zero faces in train: {nz_tr:.1f}%"
assert nz_te < 15, f"❌ Too many zero faces in test:  {nz_te:.1f}%"
print("✓ Face quality OK")

# WHY 1.5x:
#   1.0x → real F1 collapsed to 0.03
#   1.5x → best stable real F1 ~0.05-0.08
#   2.0x → loss oscillation, unstable training
MINORITY_BOOST    = 1.5
class_counts_emb  = np.bincount(y_train, minlength=4).astype(np.float32)
class_weights_emb = len(y_train) / (4.0 * class_counts_emb)
class_weights_emb[0] *= MINORITY_BOOST   # real
class_weights_emb[1] *= MINORITY_BOOST   # fake_audio
weights_emb = torch.tensor(
    class_weights_emb, dtype=torch.float32).to(device)
assert class_weights_emb[0] < 30, \
    f"❌ Real weight too high: {class_weights_emb[0]:.1f}"
print(f"\nClass counts:  {class_counts_emb.astype(int)}")
print(f"Class weights: {np.round(class_weights_emb, 4)}")
print(f"MINORITY_BOOST: {MINORITY_BOOST}x → real + fake_audio")

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_val_t   = torch.tensor(X_val,   dtype=torch.float32)
y_val_t   = torch.tensor(y_val,   dtype=torch.long)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.long)
print(f"\n✓ Tensors: train={tuple(X_train_t.shape)} | "
      f"val={tuple(X_val_t.shape)} | test={tuple(X_test_t.shape)}")

sample_weights = torch.tensor(
    [class_weights_emb[int(y)] for y in y_train],
    dtype=torch.float32)
sampler_emb = WeightedRandomSampler(
    sample_weights, len(sample_weights), replacement=True)

train_ds_emb = TensorDataset(X_train_t, y_train_t)
val_ds_emb   = TensorDataset(X_val_t,   y_val_t)
test_ds_emb  = TensorDataset(X_test_t,  y_test_t)

train_loader_emb = DataLoader(
    train_ds_emb, batch_size=64, sampler=sampler_emb)
val_loader_emb   = DataLoader(
    val_ds_emb,   batch_size=64, shuffle=False)
test_loader      = DataLoader(
    test_ds_emb,  batch_size=64, shuffle=False)

print(f"\n── DataLoaders ready ──")
print(f"Train: {len(train_ds_emb):,} | {len(train_loader_emb)} batches")
print(f"Val:   {len(val_ds_emb):,}   | {len(val_loader_emb)} batches  ← selection only")
print(f"Test:  {len(test_ds_emb):,}  | {len(test_loader)} batches  ← locked")
print("⚠ test_loader locked until ALL training complete")
print("✓ All models use these identical loaders")
print("✓ Issue 1 FIXED — all models use same 17017/2100/2000 split")

In [ ]:
# CELL 24 │ Phase 6 │ Baseline 2 — Embedding MLP
import os, json, shutil
import torch.nn.functional as F

class EmbeddingMLP(nn.Module):
    def __init__(self, input_dim=704, num_classes=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes))
    def forward(self, x):
        return self.net(x)

torch.manual_seed(42)
np.random.seed(42)

emb_mlp       = EmbeddingMLP().to(device)
criterion_emb = FocalLoss(alpha=weights_emb, gamma=2.0)
optimizer_emb = optim.Adam(
    emb_mlp.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler_emb = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_emb, T_max=50)

best_f1_emb, best_epoch_emb = 0, 0

print("="*55)
print("TRAINING EMBEDDING MLP")
print(f"  train: {len(train_loader_emb.dataset)} samples")
print(f"  val:   {len(val_loader_emb.dataset)} samples  ← selection only")
print(f"  test:  {len(test_loader.dataset)} samples   ← locked")
print("="*55)

for epoch in range(50):
    emb_mlp.train()
    for xb, yb in train_loader_emb:
        xb, yb = xb.to(device), yb.to(device)
        optimizer_emb.zero_grad()
        criterion_emb(emb_mlp(xb), yb).backward()
        optimizer_emb.step()
    scheduler_emb.step()

    # val — model selection only
    emb_mlp.eval()
    all_p, all_l = [], []
    with torch.no_grad():
        for xb, yb in val_loader_emb:
            all_p.extend(
                emb_mlp(xb.to(device)).argmax(1).cpu().numpy())
            all_l.extend(yb.numpy())

    f1  = f1_score(all_l, all_p,
                   average='macro', zero_division=0)
    bal = balanced_accuracy_score(all_l, all_p)
    acc = accuracy_score(all_l, all_p)

    if f1 > best_f1_emb:
        best_f1_emb    = f1
        best_epoch_emb = epoch + 1
        torch.save(emb_mlp.state_dict(),
                   "/content/results/best_emb_mlp.pth")

    if (epoch+1) % 10 == 0:
        print(f"Ep {epoch+1:02d}/50 | "
              f"Acc:{acc:.4f} | BalAcc:{bal:.4f} | "
              f"F1:{f1:.4f}"
              + (" ← best" if epoch+1==best_epoch_emb
                 else ""))

print(f"\nBest Val F1: {best_f1_emb:.4f} "
      f"at epoch {best_epoch_emb}")

# TEST evaluation — run once
print("\n" + "="*55)
print("EMBEDDING MLP — FINAL TEST EVALUATION")
print("="*55)

emb_mlp.load_state_dict(
    torch.load("/content/results/best_emb_mlp.pth"))
emb_mlp.eval()

all_p, all_l, all_sc = [], [], []
with torch.no_grad():
    for xb, yb in test_loader:        # ← TEST SET
        logits = emb_mlp(xb.to(device))
        probs  = torch.softmax(logits, dim=1)
        all_p.extend(logits.argmax(1).cpu().numpy())
        all_l.extend(yb.numpy())
        all_sc.extend(probs.cpu().numpy())

all_p  = np.array(all_p)
all_l  = np.array(all_l)
all_sc = np.array(all_sc)

acc = accuracy_score(all_l, all_p)
bal = balanced_accuracy_score(all_l, all_p)
f1  = f1_score(all_l, all_p,
               average='macro', zero_division=0)
auc = roc_auc_score(
    (all_l!=0).astype(int), 1-all_sc[:,0])
cm  = confusion_matrix(all_l, all_p)

print(f"Test samples:      {len(all_l)}")
print(f"Accuracy:          {acc:.4f}")
print(f"Balanced Accuracy: {bal:.4f}")
print(f"Macro F1:          {f1:.4f}  ← report in paper")
print(f"Binary AUC:        {auc:.4f}")
print(classification_report(
    all_l, all_p,
    target_names=["real","fake_audio",
                  "fake_video","fake_both"],
    zero_division=0))
print("Confusion Matrix:")
print(cm)

# Save
DRIVE_RESULTS = ("/content/drive/MyDrive/"
                 "TRM_DATASETS/results")
DRIVE_CKPT    = ("/content/drive/MyDrive/"
                 "TRM_DATASETS/trm_checkpoint")
os.makedirs(DRIVE_RESULTS, exist_ok=True)
os.makedirs(DRIVE_CKPT,    exist_ok=True)
os.makedirs("/content/results", exist_ok=True)

result = {
    "model":                  "Embedding MLP",
    "val_samples":            int(len(val_loader_emb.dataset)),
    "val_macro_f1":           float(best_f1_emb),
    "best_epoch":             int(best_epoch_emb),
    "test_samples":           int(len(all_l)),
    "test_macro_f1":          float(f1),
    "test_binary_auc":        float(auc),
    "test_balanced_accuracy": float(bal),
    "test_accuracy":          float(acc),
    "test_confusion_matrix":  cm.tolist(),
    "evaluation_note":
        "val used for model selection only. "
        "test is held-out identity-disjoint set."
}

with open("/content/results/B2_emb_mlp.json","w") as fj:
    json.dump(result, fj, indent=2)
with open(f"{DRIVE_RESULTS}/B2_emb_mlp.json","w") as fj:
    json.dump(result, fj, indent=2)
shutil.copy(
    "/content/results/best_emb_mlp.pth",
    f"{DRIVE_CKPT}/best_emb_mlp.pth")

print(f"\n✓ B2_emb_mlp.json saved")
print(f"  Val F1:  {best_f1_emb:.4f}")
print(f"  Test F1: {f1:.4f}  ← use in paper")

In [ ]:
# CELL 25 │ Phase 7 │ Baseline 3 — VFD Cosine Similarity
import os, json, shutil
import torch.nn.functional as F
from sklearn.metrics import (roc_curve, roc_auc_score,
    accuracy_score, balanced_accuracy_score,
    f1_score, confusion_matrix)

# PART 1: Train projection layer on train set
print("Training voice→face projection layer...")

voice_proj_layer = torch.nn.Linear(
    192, 512, bias=False).to(device)
optimizer_proj = torch.optim.Adam(
    voice_proj_layer.parameters(), lr=1e-3)

proj_ds     = torch.utils.data.TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32))
proj_loader = torch.utils.data.DataLoader(
    proj_ds, batch_size=256, shuffle=True)

best_auc_proj = 0

for epoch in range(30):
    voice_proj_layer.train()
    total_loss = 0
    for xb, yb in proj_loader:
        xb, yb = xb.to(device), yb.to(device)
        face_norm  = F.normalize(xb[:, :512], dim=1)
        voice_proj = voice_proj_layer(xb[:, 512:])
        voice_norm = F.normalize(voice_proj, dim=1)
        sim        = F.cosine_similarity(face_norm, voice_norm)
        targets    = 1.0 - (yb != 0).float()
        sim_scaled = (sim + 1.0) / 2.0
        loss = F.binary_cross_entropy(sim_scaled, targets)
        optimizer_proj.zero_grad()
        loss.backward()
        optimizer_proj.step()
        total_loss += loss.item()

    # select best on VAL
    voice_proj_layer.eval()
    with torch.no_grad():
        face_v  = F.normalize(
            torch.tensor(X_val[:,:512],
                         dtype=torch.float32).to(device), dim=1)
        voice_v = F.normalize(
            voice_proj_layer(
                torch.tensor(X_val[:,512:],
                             dtype=torch.float32).to(device)),
            dim=1)
        sim_v   = F.cosine_similarity(
            face_v, voice_v).cpu().numpy()
    bin_val = (y_val != 0).astype(int)
    val_auc = roc_auc_score(bin_val, -sim_v)
    if val_auc > best_auc_proj:
        best_auc_proj = val_auc
        torch.save(voice_proj_layer.state_dict(),
                   "/content/results/best_vfd_proj.pth")

    if (epoch+1) % 10 == 0:
        print(f"  Ep {epoch+1}/30 | "
              f"Loss:{total_loss/len(proj_loader):.4f} | "
              f"Val AUC:{val_auc:.4f}")

print(f"✓ Best val AUC: {best_auc_proj:.4f}")

# PART 2: Final evaluation on TEST set
print("\n" + "="*55)
print("VFD — FINAL TEST EVALUATION")
print("="*55)

voice_proj_layer.load_state_dict(
    torch.load("/content/results/best_vfd_proj.pth"))
voice_proj_layer.eval()

with torch.no_grad():
    face_t  = F.normalize(
        torch.tensor(X_test[:,:512],
                     dtype=torch.float32).to(device), dim=1)
    voice_t = F.normalize(
        voice_proj_layer(
            torch.tensor(X_test[:,512:],
                         dtype=torch.float32).to(device)),
        dim=1)
    cos_sim = F.cosine_similarity(
        face_t, voice_t).cpu().numpy()

# binary: 0=real, 1=fake
bin_test   = (y_test != 0).astype(int)
vfd_scores = -cos_sim

# optimal threshold from ROC
fpr_arr, tpr_arr, thresholds = roc_curve(
    bin_test, vfd_scores)
opt_idx    = np.argmax(tpr_arr - fpr_arr)
opt_thresh = thresholds[opt_idx]
vfd_preds  = (vfd_scores > opt_thresh).astype(int)
vfd_auc    = roc_auc_score(bin_test, vfd_scores)
vfd_f1     = f1_score(
    bin_test, vfd_preds,
    average='macro', zero_division=0)
vfd_acc    = accuracy_score(bin_test, vfd_preds)
vfd_bal    = balanced_accuracy_score(bin_test, vfd_preds)
vfd_cm     = confusion_matrix(bin_test, vfd_preds)

print(f"Test samples:      {len(y_test)}")
print(f"AUC:               {vfd_auc:.4f}  ← report in paper")
print(f"Macro F1 (binary): {vfd_f1:.4f}")
print(f"Accuracy:          {vfd_acc:.4f}")
print(f"Balanced Accuracy: {vfd_bal:.4f}")
print(f"Optimal threshold: {opt_thresh:.4f}")
print(f"\nMean sim real: {cos_sim[y_test==0].mean():.4f}")
print(f"Mean sim fake: {cos_sim[y_test!=0].mean():.4f}")
gap = (cos_sim[y_test==0].mean() -
       cos_sim[y_test!=0].mean())
print(f"Gap (real-fake): {gap:.4f} "
      f"{'✓ positive' if gap>0 else '✗ negative'}")
print("\nConfusion Matrix (binary):")
print(vfd_cm)

# PART 3: Save
DRIVE_RESULTS = ("/content/drive/MyDrive/"
                 "TRM_DATASETS/results")
DRIVE_CKPT    = ("/content/drive/MyDrive/"
                 "TRM_DATASETS/trm_checkpoint")
os.makedirs(DRIVE_RESULTS, exist_ok=True)
os.makedirs(DRIVE_CKPT,    exist_ok=True)
os.makedirs("/content/results", exist_ok=True)

result = {
    "model":                  "VFD Cosine Similarity",
    "classes":                "binary (real vs any fake)",
    "val_samples":            int(len(y_val)),
    "val_best_auc":           float(best_auc_proj),
    "test_samples":           int(len(y_test)),
    "test_binary_auc":        float(vfd_auc),
    "test_macro_f1":          float(vfd_f1),
    "test_accuracy":          float(vfd_acc),
    "test_balanced_accuracy": float(vfd_bal),
    "test_confusion_matrix":  vfd_cm.tolist(),
    "mean_sim_real":          float(cos_sim[y_test==0].mean()),
    "mean_sim_fake":          float(cos_sim[y_test!=0].mean()),
    "evaluation_note":
        "learned projection voice 192→512. "
        "Val used for projection selection. "
        "Test is held-out identity-disjoint set."
}

with open("/content/results/B4_vfd.json","w") as fj:
    json.dump(result, fj, indent=2)
with open(f"{DRIVE_RESULTS}/B4_vfd.json","w") as fj:
    json.dump(result, fj, indent=2)
shutil.copy(
    "/content/results/best_vfd_proj.pth",
    f"{DRIVE_CKPT}/best_vfd_proj.pth")

print(f"\n✓ B4_vfd.json saved")
print(f"  Val AUC:  {best_auc_proj:.4f}")
print(f"  Test AUC: {vfd_auc:.4f}  ← use in paper")

In [ ]:
# CELL 26 │ Phase 8 │ Baseline 4 — One-Shot Transformer
import os, json, shutil

class OneShotTransformer(nn.Module):
    def __init__(self, face_dim=512, voice_dim=192,
                 proj_dim=256, num_heads=4, num_classes=4):
        super().__init__()
        self.face_proj  = nn.Sequential(
            nn.Linear(face_dim, proj_dim),
            nn.LayerNorm(proj_dim))
        self.voice_proj = nn.Sequential(
            nn.Linear(voice_dim, proj_dim),
            nn.LayerNorm(proj_dim))
        self.cross_attn = nn.MultiheadAttention(
            proj_dim, num_heads=num_heads,
            batch_first=True, dropout=0.1)
        self.attn_norm  = nn.LayerNorm(proj_dim)
        self.classifier = nn.Sequential(
            nn.Linear(proj_dim, 64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes))

    def forward(self, x):
        face_emb  = self.face_proj(x[:, :512])
        voice_emb = self.voice_proj(x[:, 512:])
        f = face_emb.unsqueeze(1)
        v = voice_emb.unsqueeze(1)
        attended, _ = self.cross_attn(
            query=f, key=v, value=v)
        fused = self.attn_norm(
            attended.squeeze(1) + face_emb)
        return self.classifier(fused)

torch.manual_seed(42)
np.random.seed(42)

ost_model     = OneShotTransformer().to(device)
criterion_ost = FocalLoss(alpha=weights_emb, gamma=2.0)
optimizer_ost = optim.Adam(
    ost_model.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler_ost = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_ost, T_max=50)

best_f1_ost, best_epoch_ost = 0, 0

print("="*55)
print("TRAINING ONE-SHOT TRANSFORMER")
print("="*55)

for epoch in range(50):
    ost_model.train()
    for xb, yb in train_loader_emb:
        xb, yb = xb.to(device), yb.to(device)
        optimizer_ost.zero_grad()
        loss = criterion_ost(ost_model(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            ost_model.parameters(), 1.0)
        optimizer_ost.step()
    scheduler_ost.step()

    # val — model selection only
    ost_model.eval()
    all_p, all_l = [], []
    with torch.no_grad():
        for xb, yb in val_loader_emb:
            all_p.extend(
                ost_model(xb.to(device))
                .argmax(1).cpu().numpy())
            all_l.extend(yb.numpy())

    f1  = f1_score(all_l, all_p,
                   average='macro', zero_division=0)
    bal = balanced_accuracy_score(all_l, all_p)
    acc = accuracy_score(all_l, all_p)

    if f1 > best_f1_ost:
        best_f1_ost    = f1
        best_epoch_ost = epoch + 1
        torch.save(ost_model.state_dict(),
                   "/content/results/best_ost.pth")

    if (epoch+1) % 10 == 0:
        print(f"Ep {epoch+1:02d}/50 | "
              f"Acc:{acc:.4f} | BalAcc:{bal:.4f} | "
              f"F1:{f1:.4f}"
              + (" ← best" if epoch+1==best_epoch_ost
                 else ""))

print(f"\nBest Val F1: {best_f1_ost:.4f} "
      f"at epoch {best_epoch_ost}")

# TEST evaluation — run once
print("\n" + "="*55)
print("ONE-SHOT TRANSFORMER — FINAL TEST EVALUATION")
print("="*55)

ost_model.load_state_dict(
    torch.load("/content/results/best_ost.pth"))
ost_model.eval()

all_p, all_l, all_sc = [], [], []
with torch.no_grad():
    for xb, yb in test_loader:        # ← TEST SET
        logits = ost_model(xb.to(device))
        probs  = torch.softmax(logits, dim=1)
        all_p.extend(logits.argmax(1).cpu().numpy())
        all_l.extend(yb.numpy())
        all_sc.extend(probs.cpu().numpy())

all_p  = np.array(all_p)
all_l  = np.array(all_l)
all_sc = np.array(all_sc)

acc = accuracy_score(all_l, all_p)
bal = balanced_accuracy_score(all_l, all_p)
f1  = f1_score(all_l, all_p,
               average='macro', zero_division=0)
auc = roc_auc_score(
    (all_l!=0).astype(int), 1-all_sc[:,0])
cm  = confusion_matrix(all_l, all_p)

print(f"Test samples:      {len(all_l)}")
print(f"Accuracy:          {acc:.4f}")
print(f"Balanced Accuracy: {bal:.4f}")
print(f"Macro F1:          {f1:.4f}  ← report in paper")
print(f"Binary AUC:        {auc:.4f}")
print(classification_report(
    all_l, all_p,
    target_names=["real","fake_audio",
                  "fake_video","fake_both"],
    zero_division=0))
print("Confusion Matrix:")
print(cm)

# Save
DRIVE_RESULTS = ("/content/drive/MyDrive/"
                 "TRM_DATASETS/results")
DRIVE_CKPT    = ("/content/drive/MyDrive/"
                 "TRM_DATASETS/trm_checkpoint")
os.makedirs(DRIVE_RESULTS, exist_ok=True)
os.makedirs(DRIVE_CKPT,    exist_ok=True)
os.makedirs("/content/results", exist_ok=True)

result = {
    "model":                  "One-Shot Transformer",
    "val_samples":            int(len(val_loader_emb.dataset)),
    "val_macro_f1":           float(best_f1_ost),
    "best_epoch":             int(best_epoch_ost),
    "test_samples":           int(len(all_l)),
    "test_macro_f1":          float(f1),
    "test_binary_auc":        float(auc),
    "test_balanced_accuracy": float(bal),
    "test_accuracy":          float(acc),
    "test_confusion_matrix":  cm.tolist(),
    "evaluation_note":
        "val used for model selection only. "
        "test is held-out identity-disjoint set."
}

with open("/content/results/B5_ost.json","w") as fj:
    json.dump(result, fj, indent=2)
with open(f"{DRIVE_RESULTS}/B5_ost.json","w") as fj:
    json.dump(result, fj, indent=2)
shutil.copy(
    "/content/results/best_ost.pth",
    f"{DRIVE_CKPT}/best_ost.pth")

print(f"\n✓ B5_ost.json saved")
print(f"  Val F1:  {best_f1_ost:.4f}")
print(f"  Test F1: {f1:.4f}  ← use in paper")

In [ ]:
# CELL 27 │ Phase 9 │ Baseline 5 — Simplified TRM (GRU only)
from sklearn.metrics import (
    f1_score, balanced_accuracy_score,
    accuracy_score, roc_auc_score,
    confusion_matrix, classification_report,
)

RESULTS_DIR   = "/content/results"
DRIVE_RESULTS = "/content/drive/MyDrive/TRM_DATASETS/results"
DRIVE_CKPT    = "/content/drive/MyDrive/TRM_DATASETS/trm_checkpoint"
os.makedirs(RESULTS_DIR,   exist_ok=True)
os.makedirs(DRIVE_RESULTS, exist_ok=True)
os.makedirs(DRIVE_CKPT,    exist_ok=True)

# Model
class SimplifiedTRM(nn.Module):
    """
    Recursive GRU without cross-modal attention.
    Face and voice projected separately, concatenated once,
    then recursively refined with a GRUCell.
    """
    def __init__(self, face_dim=512, voice_dim=192,
                 hidden=256, steps=4, num_classes=4):
        super().__init__()
        self.steps  = steps
        self.hidden = hidden
        self.face_proj  = nn.Linear(face_dim,  hidden)
        self.voice_proj = nn.Linear(voice_dim, hidden)
        self.gru        = nn.GRUCell(hidden * 2, hidden)
        self.classifier = nn.Sequential(
            nn.Linear(hidden, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        face_emb  = self.face_proj(x[:, :512])
        voice_emb = self.voice_proj(x[:, 512:])
        combined  = torch.cat([face_emb, voice_emb], dim=1)
        h = torch.zeros(x.size(0), self.hidden, device=x.device)
        for _ in range(self.steps):
            h = self.gru(combined, h)
        return self.classifier(h)

_tmp   = SimplifiedTRM().to(device)
n_strm = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
print(f"SimplifiedTRM params: {n_strm:,}")
del _tmp

print("\n── Split verification ──")
for name, loader in [("train", train_loader_emb),
                     ("val",   val_loader_emb),
                     ("test",  test_loader)]:
    n = len(loader.dataset)
    assert n > 0, f"❌ {name} loader is empty"
    print(f"  {name}: {n:,} samples ✓")

assert len(train_loader_emb.dataset) > 1000, "❌ Train too small"
assert len(val_loader_emb.dataset)   > 100,  "❌ Val too small"
assert len(test_loader.dataset)      > 100,  "❌ Test too small"
assert len(val_loader_emb.dataset) != len(train_loader_emb.dataset), \
    "❌ Val = Train size — wrong loader"
print("✓ Verification passed — no hardcoded counts\n")

torch.manual_seed(42)
np.random.seed(42)

strm_model = SimplifiedTRM(
    face_dim=512, voice_dim=192,
    hidden=256, steps=4, num_classes=4
).to(device)

criterion_strm = FocalLoss(alpha=weights_emb, gamma=2.0)
optimizer_strm = optim.Adam(
    strm_model.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler_strm = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_strm, T_max=100)

best_f1_strm    = -1.0
best_epoch_strm = 0
best_ckpt_local = f"{RESULTS_DIR}/best_strm.pth"
best_ckpt_drive = f"{DRIVE_CKPT}/best_strm.pth"

print("=" * 60)
print("TRAINING SIMPLIFIED TRM")
print(f"  train: {len(train_loader_emb.dataset):,} samples")
print(f"  val:   {len(val_loader_emb.dataset):,} samples  ← selection only")
print(f"  test:  {len(test_loader.dataset):,} samples    ← locked")
print(f"  steps=4 | lr=5e-4 | epochs=100 | patience=30")
print(f"  params={n_strm:,}")
print("=" * 60)

num_epochs                = 100
patience                  = 30
min_epochs_before_early_stop = 50

for epoch in range(num_epochs):
    strm_model.train()
    train_loss = 0.0

    for xb, yb in train_loader_emb:
        xb, yb = xb.to(device), yb.to(device)
        optimizer_strm.zero_grad()
        loss = criterion_strm(strm_model(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(strm_model.parameters(), 1.0)
        optimizer_strm.step()
        train_loss += loss.item()

    scheduler_strm.step()

    strm_model.eval()
    all_p, all_l = [], []
    with torch.no_grad():
        for xb, yb in val_loader_emb:
            preds = strm_model(xb.to(device)).argmax(1).cpu().numpy()
            all_p.extend(preds)
            all_l.extend(yb.numpy())

    f1  = f1_score(all_l, all_p, average="macro", zero_division=0)
    bal = balanced_accuracy_score(all_l, all_p)
    acc = accuracy_score(all_l, all_p)

    is_best = f1 > best_f1_strm
    if is_best:
        best_f1_strm    = f1
        best_epoch_strm = epoch + 1
        torch.save(strm_model.state_dict(), best_ckpt_local)
        torch.save(strm_model.state_dict(), best_ckpt_drive)

    if (epoch + 1) % 10 == 0 or is_best:
        tag = " ← best" if is_best else ""
        print(f"Ep {epoch+1:03d}/{num_epochs} | "
              f"Loss:{train_loss/len(train_loader_emb):.4f} | "
              f"Acc:{acc:.4f} | BalAcc:{bal:.4f} | F1:{f1:.4f}{tag}")

    if (epoch + 1) > min_epochs_before_early_stop:
        if (epoch + 1 - best_epoch_strm) >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

print(f"\nBest Val F1: {best_f1_strm:.4f} at epoch {best_epoch_strm}")

if not os.path.exists(best_ckpt_local):
    raise FileNotFoundError("Best SimplifiedTRM checkpoint not saved.")

#  TEST evaluation — run exactly once
print("\n" + "=" * 60)
print("SIMPLIFIED TRM — FINAL TEST EVALUATION")
print("=" * 60)

strm_model.load_state_dict(
    torch.load(best_ckpt_local, map_location=device))
strm_model.eval()

all_p, all_l, all_sc = [], [], []
with torch.no_grad():
    for xb, yb in test_loader:
        logits = strm_model(xb.to(device))
        probs  = torch.softmax(logits, dim=1)
        all_p.extend(logits.argmax(1).cpu().numpy())
        all_l.extend(yb.numpy())
        all_sc.extend(probs.cpu().numpy())

all_p  = np.array(all_p)
all_l  = np.array(all_l)
all_sc = np.array(all_sc)

acc = accuracy_score(all_l, all_p)
bal = balanced_accuracy_score(all_l, all_p)
f1  = f1_score(all_l, all_p, average="macro", zero_division=0)
auc = roc_auc_score((all_l != 0).astype(int), 1.0 - all_sc[:, 0])
cm  = confusion_matrix(all_l, all_p)
pcf = f1_score(all_l, all_p, average=None, zero_division=0)

print(f"Test samples:      {len(all_l)}")
print(f"Accuracy:          {acc:.4f}")
print(f"Balanced Accuracy: {bal:.4f}")
print(f"Macro F1:          {f1:.4f}  ← report in paper")
print(f"Binary AUC:        {auc:.4f}")
print(f"Best epoch:        {best_epoch_strm}")
print("\nPer-class F1:")
for i, name in enumerate(["real","fake_audio","fake_video","fake_both"]):
    print(f"  {name:<14} {pcf[i]:.4f}")
print("\nClassification Report:")
print(classification_report(
    all_l, all_p,
    target_names=["real","fake_audio","fake_video","fake_both"],
    zero_division=0))
print("Confusion Matrix:")
print(cm)

result = {
    "model":       "Simplified TRM (GRU)",
    "steps":       4,
    "params":      n_strm,
    "split": {
        "train": int(len(train_loader_emb.dataset)),
        "val":   int(len(val_loader_emb.dataset)),
        "test":  int(len(test_loader.dataset)),
    },
    "val_macro_f1":           float(best_f1_strm),
    "best_epoch":             int(best_epoch_strm),
    "test_samples":           int(len(all_l)),
    "test_macro_f1":          float(f1),
    "test_binary_auc":        float(auc),
    "test_balanced_accuracy": float(bal),
    "test_accuracy":          float(acc),
    "test_per_class_f1": {
        "real":       float(pcf[0]),
        "fake_audio": float(pcf[1]),
        "fake_video": float(pcf[2]),
        "fake_both":  float(pcf[3]),
    },
    "test_confusion_matrix": cm.tolist(),
    "evaluation_note": (
        "Val used for model selection only. "
        "Test is held-out identity-disjoint set. "
        "Recursive GRU without attention. "
        f"Split: {len(train_loader_emb.dataset)} train / "
        f"{len(val_loader_emb.dataset)} val / "
        f"{len(test_loader.dataset)} test."
    )
}

for path in [f"{RESULTS_DIR}/B6_strm.json",
             f"{DRIVE_RESULTS}/B6_strm.json"]:
    with open(path, "w") as fj:
        json.dump(result, fj, indent=2)

print(f"\n✓ Saved locally and to Drive")
print(f"✓ Checkpoint: {best_ckpt_drive}")
print(f"\nFINAL: F1={f1:.4f} | AUC={auc:.4f} | epoch={best_epoch_strm}")
print("\n" + "=" * 60)
print("TRAINING COMPLETE — SIMPLIFIED TRM")
print("=" * 60)

In [ ]:
# CELL 28 │ Phase 10 │ Full TRM-VFD — Main Recursive Model
import os, json, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import (
    f1_score, balanced_accuracy_score,
    accuracy_score, roc_auc_score,
    confusion_matrix, classification_report
)

DRIVE_RESULTS = "/content/drive/MyDrive/TRM_DATASETS/results"
DRIVE_CKPT    = "/content/drive/MyDrive/TRM_DATASETS/trm_checkpoint"
os.makedirs(DRIVE_RESULTS,      exist_ok=True)
os.makedirs(DRIVE_CKPT,         exist_ok=True)
os.makedirs("/content/results", exist_ok=True)

class TRM_VFD(nn.Module):
    """
    Full TRM-VFD: Recursive Cross-Modal Biometric Verification
    Input:  704-dim (ArcFace 512 + ECAPA 192)
    Output: 4-class logits
    Steps:  5 recursive refinement iterations
    Params: ~856k
    """
    def __init__(self, face_dim=512, voice_dim=192,
                 proj_dim=256, num_heads=4,
                 num_steps=5, num_classes=4):
        super().__init__()
        self.num_steps = num_steps
        self.proj_dim  = proj_dim

        self.face_proj = nn.Sequential(
            nn.Linear(face_dim, proj_dim),
            nn.LayerNorm(proj_dim)
        )
        self.voice_proj = nn.Sequential(
            nn.Linear(voice_dim, proj_dim),
            nn.LayerNorm(proj_dim)
        )
        self.cross_attn = nn.MultiheadAttention(
            proj_dim, num_heads=num_heads,
            batch_first=True, dropout=0.1
        )
        self.attn_norm  = nn.LayerNorm(proj_dim)
        self.gru        = nn.GRUCell(proj_dim, proj_dim)
        self.classifier = nn.Sequential(
            nn.Linear(proj_dim, 64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        face_emb  = self.face_proj(x[:, :512])
        voice_emb = self.voice_proj(x[:, 512:])
        h = torch.zeros(x.size(0), self.proj_dim, device=x.device)
        for _ in range(self.num_steps):
            attended, _ = self.cross_attn(
                query=face_emb.unsqueeze(1),
                key=voice_emb.unsqueeze(1),
                value=voice_emb.unsqueeze(1)
            )
            fused = self.attn_norm(attended.squeeze(1) + face_emb)
            h = self.gru(fused, h)
        return self.classifier(h)

print("── Split verification ──")
print(f"Train: {X_train_t.shape} | dist: {np.bincount(y_train_t.cpu().numpy(), minlength=4).tolist()}")
print(f"Val:   {X_val_t.shape}   | dist: {np.bincount(y_val_t.cpu().numpy(),   minlength=4).tolist()}")
print(f"Test:  {X_test_t.shape}  | dist: {np.bincount(y_test_t.cpu().numpy(),  minlength=4).tolist()}")

for name, y_t in [("train", y_train_t),
                   ("val",   y_val_t),
                   ("test",  y_test_t)]:
    counts = np.bincount(y_t.cpu().numpy(), minlength=4).tolist()
    assert len(counts) == 4, \
        f"❌ {name} has {len(counts)} classes, expected 4"
    assert all(c > 0 for c in counts), \
        f"❌ {name} missing a class: {counts}"
    print(f"✓ {name}: all 4 classes present {counts}")

assert X_train_t.shape[0] > 1000, \
    f"❌ Train too small: {X_train_t.shape[0]}"
assert X_val_t.shape[0]   > 100, \
    f"❌ Val too small: {X_val_t.shape[0]}"
assert X_test_t.shape[0]  > 100, \
    f"❌ Test too small: {X_test_t.shape[0]}"
assert X_val_t.shape[0]  != X_train_t.shape[0], \
    "❌ Val same size as train — wrong array loaded"
assert X_test_t.shape[0] != X_train_t.shape[0], \
    "❌ Test same size as train — wrong array loaded"

print(f"✓ Sizes: train={X_train_t.shape[0]} | "
      f"val={X_val_t.shape[0]} | "
      f"test={X_test_t.shape[0]}")
print("✓ Verification passed — no hardcoded counts\n")

_tmp = TRM_VFD(num_steps=5).to(device)
n_params = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
print(f"TRM_VFD params: {n_params:,}")
del _tmp

torch.manual_seed(42)
np.random.seed(42)

trm_model     = TRM_VFD(num_steps=5).to(device)
criterion_trm = FocalLoss(alpha=weights_emb, gamma=2.0)
optimizer_trm = optim.Adam(
    trm_model.parameters(), lr=5e-4, weight_decay=1e-4
)
scheduler_trm = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_trm, T_max=100
)

best_f1_trm    = -1.0
best_epoch_trm = 0
no_improve     = 0
LOCAL_CKPT     = "/content/results/best_trm.pth"
DRIVE_CKPT_PTH = f"{DRIVE_CKPT}/best_trm_FINAL.pth"

print("=" * 65)
print("TRAINING FULL TRM-VFD")
print(f"  train:     {len(train_loader_emb.dataset):,} samples")
print(f"  val:       {len(val_loader_emb.dataset):,} samples  ← selection only")
print(f"  test:      {len(test_loader.dataset):,} samples    ← locked")
print(f"  num_steps: 5 | lr: 5e-4 | epochs: 100 | patience: 30")
print("=" * 65)

for epoch in range(100):
    trm_model.train()
    train_loss = 0.0

    for xb, yb in train_loader_emb:
        xb, yb = xb.to(device), yb.to(device)
        optimizer_trm.zero_grad()
        loss = criterion_trm(trm_model(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trm_model.parameters(), 1.0)
        optimizer_trm.step()
        train_loss += loss.item()

    scheduler_trm.step()

    trm_model.eval()
    all_p, all_l = [], []
    with torch.no_grad():
        for xb, yb in val_loader_emb:
            preds = trm_model(xb.to(device)).argmax(1).cpu().numpy()
            all_p.extend(preds)
            all_l.extend(yb.numpy())

    f1  = f1_score(all_l, all_p, average="macro", zero_division=0)
    bal = balanced_accuracy_score(all_l, all_p)
    acc = accuracy_score(all_l, all_p)

    if f1 > best_f1_trm:
        best_f1_trm    = f1
        best_epoch_trm = epoch + 1
        no_improve     = 0
        torch.save(trm_model.state_dict(), LOCAL_CKPT)
        torch.save(trm_model.state_dict(), DRIVE_CKPT_PTH)
    else:
        no_improve += 1

    if (epoch + 1) % 10 == 0 or (epoch + 1) == best_epoch_trm:
        tag = " ← best" if (epoch + 1) == best_epoch_trm else ""
        print(f"Ep {epoch+1:03d}/100 | "
              f"Loss:{train_loss/len(train_loader_emb):.4f} | "
              f"Acc:{acc:.4f} | BalAcc:{bal:.4f} | F1:{f1:.4f}{tag}")

    if no_improve >= 30 and epoch > 50:
        print(f"Early stopping at epoch {epoch+1}")
        break

print(f"\nBest Val F1: {best_f1_trm:.4f} at epoch {best_epoch_trm}")

# TEST evaluation — run exactly once
print("\n" + "=" * 65)
print("FULL TRM-VFD — FINAL TEST EVALUATION")
print("=" * 65)

trm_model.load_state_dict(
    torch.load(LOCAL_CKPT, map_location=device)
)
trm_model.eval()

all_p, all_l, all_sc = [], [], []
with torch.no_grad():
    for xb, yb in test_loader:
        logits = trm_model(xb.to(device))
        probs  = torch.softmax(logits, dim=1)
        all_p.extend(logits.argmax(1).cpu().numpy())
        all_l.extend(yb.numpy())
        all_sc.extend(probs.cpu().numpy())

all_p  = np.array(all_p)
all_l  = np.array(all_l)
all_sc = np.array(all_sc)

acc = accuracy_score(all_l, all_p)
bal = balanced_accuracy_score(all_l, all_p)
f1  = f1_score(all_l, all_p, average="macro", zero_division=0)
auc = roc_auc_score((all_l != 0).astype(int), 1.0 - all_sc[:, 0])
cm  = confusion_matrix(all_l, all_p)
pcf = f1_score(all_l, all_p, average=None, zero_division=0)

print(f"Test samples:      {len(all_l)}")
print(f"Accuracy:          {acc:.4f}")
print(f"Balanced Accuracy: {bal:.4f}")
print(f"Macro F1:          {f1:.4f}  ← report in paper")
print(f"Binary AUC:        {auc:.4f}")
print(f"Best epoch:        {best_epoch_trm}")
print(f"Classes predicted: {sorted(set(all_p.tolist()))}")
print("\nPer-class F1:")
for i, name in enumerate(["real","fake_audio","fake_video","fake_both"]):
    print(f"  {name:<14} {pcf[i]:.4f}")
print("\nClassification Report:")
print(classification_report(
    all_l, all_p,
    target_names=["real","fake_audio","fake_video","fake_both"],
    zero_division=0
))
print("Confusion Matrix:")
print(cm)

#  Save result
result = {
    "model":                  "Full TRM-VFD",
    "num_steps":              5,
    "train_samples":          int(X_train_t.shape[0]),
    "train_dist":             np.bincount(y_train_t.cpu().numpy(), minlength=4).tolist(),
    "val_samples":            int(X_val_t.shape[0]),
    "val_dist":               np.bincount(y_val_t.cpu().numpy(), minlength=4).tolist(),
    "val_macro_f1":           float(best_f1_trm),
    "best_epoch":             int(best_epoch_trm),
    "test_samples":           int(len(all_l)),
    "test_dist":              np.bincount(all_l, minlength=4).tolist(),
    "test_macro_f1":          float(f1),
    "test_binary_auc":        float(auc),
    "test_balanced_accuracy": float(bal),
    "test_accuracy":          float(acc),
    "test_per_class_f1": {
        "real":       float(pcf[0]),
        "fake_audio": float(pcf[1]),
        "fake_video": float(pcf[2]),
        "fake_both":  float(pcf[3]),
    },
    "test_confusion_matrix":  cm.tolist(),
    "architecture": {
        "face_encoder":  "ArcFace InsightFace buffalo_l 512-dim",
        "voice_encoder": "ECAPA-TDNN SpeechBrain VoxCeleb 192-dim",
        "input_dim":     704,
        "proj_dim":      256,
        "num_heads":     4,
        "num_steps":     5,
        "gru_hidden":    256,
        "num_classes":   4,
        "params":        n_params,
    },
    "evaluation_note": (
        "Val used for model selection only. "
        "Test is held-out identity-disjoint set. "
        f"Split: {X_train_t.shape[0]} train / "
        f"{X_val_t.shape[0]} val / "
        f"{X_test_t.shape[0]} test. "
        "No hardcoded split counts — verified dynamically."
    )
}

for path in ["/content/results/TRM_VFD.json",
             f"{DRIVE_RESULTS}/TRM_VFD.json"]:
    with open(path, "w") as fj:
        json.dump(result, fj, indent=2)

shutil.copy(LOCAL_CKPT, DRIVE_CKPT_PTH)

print(f"\n✓ TRM_VFD.json saved locally and to Drive")
print(f"✓ Checkpoint saved: {DRIVE_CKPT_PTH}")
print(f"\nFINAL: F1={f1:.4f} | AUC={auc:.4f} | epoch={best_epoch_trm}")
print("\n" + "=" * 65)
print("TRAINING COMPLETE — THIS IS YOUR FINAL TRM-VFD RESULT")
print("=" * 65)

In [ ]:
# CELL 29 │ Phase 10 │ Improved Full TRM — Tokenized Architecture
# Architecture from paper:
#   Face:  512-dim → tokenized into 8×64 segments
#   Voice: 192-dim → tokenized into 6×32 segments
#   Both projected to shared 128-dim token space
#   Token-level cross-attention + recursive GRU (4 steps)
#   Loss: CrossEntropy with WeightedRandomSampler
#   Params: ~377.5k (vs Full TRM-VFD 856k)

import os, json, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import (
    f1_score, balanced_accuracy_score,
    accuracy_score, roc_auc_score,
    confusion_matrix, classification_report
)

DRIVE_RESULTS = "/content/drive/MyDrive/TRM_DATASETS/results"
DRIVE_CKPT    = "/content/drive/MyDrive/TRM_DATASETS/trm_checkpoint"
os.makedirs(DRIVE_RESULTS,      exist_ok=True)
os.makedirs(DRIVE_CKPT,         exist_ok=True)
os.makedirs("/content/results", exist_ok=True)

#  Model Definition
class ImprovedFullTRM(nn.Module):
    """
    Improved Full TRM:
      - Face 512-dim  → 8 tokens × 64-dim → projected to 128-dim each
      - Voice 192-dim → 6 tokens × 32-dim → projected to 128-dim each
      - Token-level cross-attention (face tokens attend to voice tokens)
      - Recursive GRU refinement (4 steps)
      - ~377.5k params (vs Full TRM-VFD 856k)
    """
    def __init__(self,
                 face_dim=512,   voice_dim=192,
                 face_tokens=8,  voice_tokens=6,
                 token_dim=128,  num_heads=4,
                 num_steps=4,    num_classes=4,
                 dropout=0.3):
        super().__init__()

        self.face_tokens  = face_tokens
        self.voice_tokens = voice_tokens
        self.token_dim    = token_dim
        self.num_steps    = num_steps

        # face 512 → 8 segments of 64 → each projected to 128
        face_seg_dim  = face_dim  // face_tokens   # 64
        voice_seg_dim = voice_dim // voice_tokens  # 32

        self.face_tok_proj  = nn.Linear(face_seg_dim,  token_dim)
        self.voice_tok_proj = nn.Linear(voice_seg_dim, token_dim)

        self.face_norm  = nn.LayerNorm(token_dim)
        self.voice_norm = nn.LayerNorm(token_dim)

        # token-level cross-attention: face queries → voice keys/values
        self.cross_attn = nn.MultiheadAttention(
            token_dim, num_heads=num_heads,
            batch_first=True, dropout=0.1
        )
        self.attn_norm = nn.LayerNorm(token_dim)

        # pool face tokens to single vector → GRU input
        # face_tokens × token_dim → token_dim
        self.pool_proj = nn.Linear(face_tokens * token_dim, token_dim)
        self.pool_norm = nn.LayerNorm(token_dim)

        # recursive GRU
        self.gru = nn.GRUCell(token_dim, token_dim)

        # classifier
        self.classifier = nn.Sequential(
            nn.Linear(token_dim, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        B = x.size(0)

        #  Tokenize
        # face: [B, 512] → [B, 8, 64] → [B, 8, 128]
        face_raw  = x[:, :512].view(B, self.face_tokens, -1)
        face_tok  = self.face_norm(self.face_tok_proj(face_raw))

        # voice: [B, 192] → [B, 6, 32] → [B, 6, 128]
        voice_raw = x[:, 512:].view(B, self.voice_tokens, -1)
        voice_tok = self.voice_norm(self.voice_tok_proj(voice_raw))

        #  Recursive cross-attention + GRU
        h = torch.zeros(B, self.token_dim, device=x.device)

        for _ in range(self.num_steps):
            # face tokens attend to voice tokens
            attended, _ = self.cross_attn(
                query=face_tok,    # [B, 8, 128]
                key=voice_tok,     # [B, 6, 128]
                value=voice_tok    # [B, 6, 128]
            )                      # → [B, 8, 128]

            fused = self.attn_norm(attended + face_tok)  # residual

            # pool all 8 face tokens → single vector [B, 128]
            pooled = self.pool_norm(
                self.pool_proj(fused.reshape(B, -1))
            )

            h = self.gru(pooled, h)

        return self.classifier(h)

#  Verify architecture
_tmp = ImprovedFullTRM().to(device)
n_params = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
print(f"ImprovedFullTRM params: {n_params:,}")
# verify forward pass
_x = torch.randn(4, 704).to(device)
_out = _tmp(_x)
assert _out.shape == (4, 4), f"Wrong output shape: {_out.shape}"
print(f"✓ Forward pass: input {_x.shape} → output {_out.shape}")
del _tmp, _x, _out

#  Dynamic split verification
print("\n── Split verification ──")
for name, X_t, y_t in [("train", X_train_t, y_train_t),
                        ("val",   X_val_t,   y_val_t),
                        ("test",  X_test_t,  y_test_t)]:
    counts = np.bincount(y_t.cpu().numpy(), minlength=4).tolist()
    assert all(c > 0 for c in counts), f"❌ {name} missing a class"
    print(f"  {name}: {X_t.shape} | dist: {counts} ✓")

assert X_train_t.shape[0] > 1000, "❌ Train too small"
assert X_val_t.shape[0]  != X_train_t.shape[0], "❌ Val = Train size"
assert X_test_t.shape[0] != X_train_t.shape[0], "❌ Test = Train size"
print(f"✓ train={X_train_t.shape[0]} | val={X_val_t.shape[0]} | test={X_test_t.shape[0]}")
print("✓ No hardcoded counts\n")

# WeightedRandomSampler (CrossEntropy — no FocalLoss here)
# Paper spec: CrossEntropy + WeightedRandomSampler (not FocalLoss)
class_counts_impr  = np.bincount(y_train_t.cpu().numpy(), minlength=4).astype(float)
class_weights_impr = len(y_train_t) / (4.0 * class_counts_impr)
class_weights_impr[0] *= 1.5
class_weights_impr[1] *= 1.5

print(f"Class counts:  {class_counts_impr.astype(int)}")
print(f"Class weights: {np.round(class_weights_impr, 4)}")

# CrossEntropy weights for loss
ce_weights = torch.tensor(class_weights_impr, dtype=torch.float32).to(device)
criterion_impr = nn.CrossEntropyLoss(weight=ce_weights)

# WeightedRandomSampler for dataloader
sample_w_impr = torch.tensor(
    [class_weights_impr[int(y)] for y in y_train_t.cpu().numpy()],
    dtype=torch.float32
)
sampler_impr = torch.utils.data.WeightedRandomSampler(
    sample_w_impr, len(sample_w_impr), replacement=True
)

from torch.utils.data import TensorDataset, DataLoader

train_ds_impr = TensorDataset(X_train_t, y_train_t)
val_ds_impr   = TensorDataset(X_val_t,   y_val_t)
test_ds_impr  = TensorDataset(X_test_t,  y_test_t)

train_loader_impr = DataLoader(train_ds_impr, batch_size=64, sampler=sampler_impr)
val_loader_impr   = DataLoader(val_ds_impr,   batch_size=64, shuffle=False)
test_loader_impr  = DataLoader(test_ds_impr,  batch_size=64, shuffle=False)

print(f"\nLoaders: train={len(train_loader_impr)} | "
      f"val={len(val_loader_impr)} | "
      f"test={len(test_loader_impr)} batches")

#  Setup
torch.manual_seed(42)
np.random.seed(42)

impr_model    = ImprovedFullTRM(num_steps=4).to(device)
optimizer_impr = optim.Adam(
    impr_model.parameters(), lr=5e-4, weight_decay=1e-4
)
scheduler_impr = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_impr, T_max=100
)

best_f1_impr    = -1.0
best_epoch_impr = 0
no_improve      = 0
LOCAL_CKPT_IMPR = "/content/results/best_impr_trm.pth"
DRIVE_CKPT_IMPR = f"{DRIVE_CKPT}/best_improved_trm_FINAL.pth"

print("=" * 65)
print("TRAINING IMPROVED FULL TRM")
print(f"  train:     {len(train_loader_impr.dataset):,} samples")
print(f"  val:       {len(val_loader_impr.dataset):,} samples  ← selection only")
print(f"  test:      {len(test_loader_impr.dataset):,} samples    ← locked")
print(f"  num_steps: 4 | token_dim: 128 | face_tokens: 8 | voice_tokens: 6")
print(f"  loss: CrossEntropy | sampler: WeightedRandom | lr: 5e-4")
print(f"  params: {n_params:,}")
print("=" * 65)

#  Training loop
for epoch in range(100):
    impr_model.train()
    train_loss = 0.0

    for xb, yb in train_loader_impr:
        xb, yb = xb.to(device), yb.to(device)
        optimizer_impr.zero_grad()
        loss = criterion_impr(impr_model(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(impr_model.parameters(), 1.0)
        optimizer_impr.step()
        train_loss += loss.item()

    scheduler_impr.step()

    #  Val — model selection only
    impr_model.eval()
    all_p, all_l = [], []
    with torch.no_grad():
        for xb, yb in val_loader_impr:
            preds = impr_model(xb.to(device)).argmax(1).cpu().numpy()
            all_p.extend(preds)
            all_l.extend(yb.numpy())

    f1  = f1_score(all_l, all_p, average="macro", zero_division=0)
    bal = balanced_accuracy_score(all_l, all_p)
    acc = accuracy_score(all_l, all_p)

    if f1 > best_f1_impr:
        best_f1_impr    = f1
        best_epoch_impr = epoch + 1
        no_improve      = 0
        torch.save(impr_model.state_dict(), LOCAL_CKPT_IMPR)
        torch.save(impr_model.state_dict(), DRIVE_CKPT_IMPR)
    else:
        no_improve += 1

    if (epoch + 1) % 10 == 0 or (epoch + 1) == best_epoch_impr:
        tag = " ← best" if (epoch + 1) == best_epoch_impr else ""
        print(f"Ep {epoch+1:03d}/100 | "
              f"Loss:{train_loss/len(train_loader_impr):.4f} | "
              f"Acc:{acc:.4f} | BalAcc:{bal:.4f} | F1:{f1:.4f}{tag}")

    if no_improve >= 30 and epoch > 50:
        print(f"Early stopping at epoch {epoch+1}")
        break

print(f"\nBest Val F1: {best_f1_impr:.4f} at epoch {best_epoch_impr}")

#  TEST evaluation — run exactly once
print("\n" + "=" * 65)
print("IMPROVED FULL TRM — FINAL TEST EVALUATION")
print("=" * 65)

impr_model.load_state_dict(
    torch.load(LOCAL_CKPT_IMPR, map_location=device)
)
impr_model.eval()

all_p, all_l, all_sc = [], [], []
with torch.no_grad():
    for xb, yb in test_loader_impr:
        logits = impr_model(xb.to(device))
        probs  = torch.softmax(logits, dim=1)
        all_p.extend(logits.argmax(1).cpu().numpy())
        all_l.extend(yb.numpy())
        all_sc.extend(probs.cpu().numpy())

all_p  = np.array(all_p)
all_l  = np.array(all_l)
all_sc = np.array(all_sc)

acc = accuracy_score(all_l, all_p)
bal = balanced_accuracy_score(all_l, all_p)
f1  = f1_score(all_l, all_p, average="macro", zero_division=0)
auc = roc_auc_score((all_l != 0).astype(int), 1.0 - all_sc[:, 0])
cm  = confusion_matrix(all_l, all_p)
pcf = f1_score(all_l, all_p, average=None, zero_division=0)

print(f"Test samples:      {len(all_l)}")
print(f"Accuracy:          {acc:.4f}")
print(f"Balanced Accuracy: {bal:.4f}")
print(f"Macro F1:          {f1:.4f}  ← report in paper")
print(f"Binary AUC:        {auc:.4f}")
print(f"Best epoch:        {best_epoch_impr}")
print(f"Classes predicted: {sorted(set(all_p.tolist()))}")
print("\nPer-class F1:")
for i, name in enumerate(["real","fake_audio","fake_video","fake_both"]):
    print(f"  {name:<14} {pcf[i]:.4f}")
print("\nClassification Report:")
print(classification_report(
    all_l, all_p,
    target_names=["real","fake_audio","fake_video","fake_both"],
    zero_division=0
))
print("Confusion Matrix:")
print(cm)

#  Save
result = {
    "model": "Improved Full TRM",
    "architecture": {
        "face_tokens":   8,
        "voice_tokens":  6,
        "token_dim":     128,
        "num_heads":     4,
        "num_steps":     4,
        "loss":          "CrossEntropyLoss (weighted)",
        "sampler":       "WeightedRandomSampler",
        "params":        n_params,
        "face_encoder":  "ArcFace 512-dim → 8×64 tokens",
        "voice_encoder": "ECAPA-TDNN 192-dim → 6×32 tokens",
    },
    "split": {
        "train": int(X_train_t.shape[0]),
        "val":   int(X_val_t.shape[0]),
        "test":  int(X_test_t.shape[0]),
        "train_dist": np.bincount(y_train_t.cpu().numpy(), minlength=4).tolist(),
        "val_dist":   np.bincount(y_val_t.cpu().numpy(),   minlength=4).tolist(),
        "test_dist":  np.bincount(y_test_t.cpu().numpy(),  minlength=4).tolist(),
    },
    "val_macro_f1":           float(best_f1_impr),
    "best_epoch":             int(best_epoch_impr),
    "test_samples":           int(len(all_l)),
    "test_macro_f1":          float(f1),
    "test_binary_auc":        float(auc),
    "test_balanced_accuracy": float(bal),
    "test_accuracy":          float(acc),
    "test_per_class_f1": {
        "real":       float(pcf[0]),
        "fake_audio": float(pcf[1]),
        "fake_video": float(pcf[2]),
        "fake_both":  float(pcf[3]),
    },
    "test_confusion_matrix": cm.tolist(),
    "evaluation_note": (
        "Val used for model selection only. "
        "Test is held-out identity-disjoint set. "
        "Tokenized attention + GRU. CrossEntropy loss. "
        f"Split: {X_train_t.shape[0]} train / "
        f"{X_val_t.shape[0]} val / {X_test_t.shape[0]} test."
    )
}

for path in ["/content/results/IMPROVED_FULL_TRM_CE_FINAL.json",
             f"{DRIVE_RESULTS}/IMPROVED_FULL_TRM_CE_FINAL.json"]:
    with open(path, "w") as fj:
        json.dump(result, fj, indent=2)

shutil.copy(LOCAL_CKPT_IMPR, DRIVE_CKPT_IMPR)

print(f"\n✓ IMPROVED_FULL_TRM_CE_FINAL.json saved locally and to Drive")
print(f"✓ Checkpoint saved: {DRIVE_CKPT_IMPR}")
print(f"\nFINAL: F1={f1:.4f} | AUC={auc:.4f} | epoch={best_epoch_impr}")
print("\n" + "=" * 65)
print("TRAINING COMPLETE — IMPROVED FULL TRM")
print("=" * 65)

In [ ]:
# CELL 30 │ Phase 10 │ Two-Stage MLP Pipeline
# Stage 1: real (0) vs fake (1,2,3)
# Stage 2: fake_audio (1) vs fake_video (2) vs fake_both (3)
# Final:   if stage1=real → class 0, else stage2 subtype + 1

import os
import json
import shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.metrics import (
    f1_score, balanced_accuracy_score,
    accuracy_score, roc_auc_score,
    confusion_matrix, classification_report,
)

# CONFIG
SEED         = 42
RUN_ENSEMBLE = False  # set True only if mixup checkpoint provenance is clean
PIN_MEMORY   = (device == "cuda")
NUM_WORKERS  = 0

torch.manual_seed(SEED)
np.random.seed(SEED)

RESULTS_DIR   = "/content/results"
DRIVE_RESULTS = "/content/drive/MyDrive/TRM_DATASETS/results"
DRIVE_CKPT    = "/content/drive/MyDrive/TRM_DATASETS/trm_checkpoint"
os.makedirs(RESULTS_DIR,   exist_ok=True)
os.makedirs(DRIVE_RESULTS, exist_ok=True)
os.makedirs(DRIVE_CKPT,    exist_ok=True)

#  HELPERS
def to_tensor_float(x):
    if isinstance(x, torch.Tensor):
        return x.detach().clone().float()
    return torch.tensor(x, dtype=torch.float32)

def to_tensor_long(x):
    if isinstance(x, torch.Tensor):
        return x.detach().clone().long()
    return torch.tensor(x, dtype=torch.long)

def make_loader(X, y, batch_size=256, shuffle=False):
    ds = TensorDataset(X, y)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

def make_balanced_loader(X, y, batch_size=64):
    y_np = y.detach().cpu().numpy()
    counts = np.bincount(y_np)
    sample_weights = 1.0 / counts[y_np]
    sampler = WeightedRandomSampler(
        torch.tensor(sample_weights, dtype=torch.float32),
        num_samples=len(sample_weights), replacement=True
    )
    ds = TensorDataset(X, y)
    return DataLoader(ds, batch_size=batch_size, sampler=sampler,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        logp   = F.log_softmax(inputs, dim=1)
        p      = torch.exp(logp)
        logp_t = logp.gather(1, targets.unsqueeze(1)).squeeze(1)
        p_t    = p.gather(1, targets.unsqueeze(1)).squeeze(1)
        loss   = -((1 - p_t) ** self.gamma) * logp_t
        if self.alpha is not None:
            loss = self.alpha[targets] * loss
        return loss.mean()

X_train_t = to_tensor_float(X_train)
y_train_t = to_tensor_long(y_train)
X_val_t   = to_tensor_float(X_val)
y_val_t   = to_tensor_long(y_val)
X_test_t  = to_tensor_float(X_test)
y_test_t  = to_tensor_long(y_test)

print("── Split verification ──")
for name, X_t, y_t in [("train", X_train_t, y_train_t),
                        ("val",   X_val_t,   y_val_t),
                        ("test",  X_test_t,  y_test_t)]:
    counts = np.bincount(y_t.cpu().numpy(), minlength=4).tolist()
    assert len(counts) == 4,          f"❌ {name} has wrong number of classes"
    assert all(c > 0 for c in counts),f"❌ {name} missing a class: {counts}"
    print(f"  {name}: {X_t.shape} | dist: {counts} ✓")

assert X_train_t.shape[0] > 1000, "❌ Train too small"
assert X_val_t.shape[0]   > 100,  "❌ Val too small"
assert X_test_t.shape[0]  > 100,  "❌ Test too small"
assert X_val_t.shape[0]  != X_train_t.shape[0], "❌ Val = Train size"
assert X_test_t.shape[0] != X_train_t.shape[0], "❌ Test = Train size"
print(f"✓ Verified: train={X_train_t.shape[0]} | "
      f"val={X_val_t.shape[0]} | test={X_test_t.shape[0]}")
print("✓ No hardcoded counts — works with any split\n")

train_counts = np.bincount(y_train_t.cpu().numpy(), minlength=4).tolist()
val_counts   = np.bincount(y_val_t.cpu().numpy(),   minlength=4).tolist()
test_counts  = np.bincount(y_test_t.cpu().numpy(),  minlength=4).tolist()

y_train_bin = (y_train_t != 0).long()
y_val_bin   = (y_val_t   != 0).long()
y_test_bin  = (y_test_t  != 0).long()

fake_mask_train = (y_train_t != 0)
fake_mask_val   = (y_val_t   != 0)
fake_mask_test  = (y_test_t  != 0)

X_train_fake = X_train_t[fake_mask_train]
y_train_fake = (y_train_t[fake_mask_train] - 1).long()
X_val_fake   = X_val_t[fake_mask_val]
y_val_fake   = (y_val_t[fake_mask_val] - 1).long()
X_test_fake  = X_test_t[fake_mask_test]
y_test_fake  = (y_test_t[fake_mask_test] - 1).long()

print("Stage 1 — binary dist:")
print(f"  train: {np.bincount(y_train_bin.cpu().numpy()).tolist()}")
print(f"  val:   {np.bincount(y_val_bin.cpu().numpy()).tolist()}")
print("Stage 2 — fake subtype dist:")
print(f"  train: {np.bincount(y_train_fake.cpu().numpy()).tolist()}")
print(f"  val:   {np.bincount(y_val_fake.cpu().numpy()).tolist()}")

# MODELS
class BinaryDetector(nn.Module):
    def __init__(self, input_dim=704, d=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, d), nn.LayerNorm(d), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d, d),         nn.LayerNorm(d), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d, 64),        nn.GELU(),        nn.Dropout(0.2),
            nn.Linear(64, 2),
        )
    def forward(self, x): return self.net(x)

class FakeSubClassifier(nn.Module):
    def __init__(self, input_dim=704, d=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, d), nn.LayerNorm(d), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d, d),         nn.LayerNorm(d), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d, 64),        nn.GELU(),        nn.Dropout(0.2),
            nn.Linear(64, 3),
        )
    def forward(self, x): return self.net(x)

#  TRAINING FUNCTION
def train_stage(model, train_X, train_y, val_X, val_y,
                num_classes, ckpt_path,
                lr=2e-4, weight_decay=2e-4,
                num_epochs=100, patience=25,
                gamma=2.0, stage_name="Stage"):

    train_loader = make_balanced_loader(train_X, train_y, batch_size=64)
    val_loader   = make_loader(val_X, val_y, batch_size=256, shuffle=False)

    counts  = np.bincount(train_y.cpu().numpy(), minlength=num_classes).astype(float)
    alpha   = counts.sum() / (num_classes * counts)
    alpha   = alpha / alpha.sum() * num_classes
    alpha_t = torch.tensor(alpha, dtype=torch.float32, device=device)
    print(f"  [{stage_name}] Focal alpha: {np.round(alpha, 3)}")

    criterion = FocalLoss(alpha=alpha_t, gamma=gamma)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=20, T_mult=2)

    best_f1, best_epoch, no_improve = -1.0, 0, 0

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
        scheduler.step()

        model.eval()
        all_p, all_l = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                all_p.extend(model(xb.to(device)).argmax(1).cpu().numpy())
                all_l.extend(yb.numpy())

        f1  = f1_score(all_l, all_p, average="macro", zero_division=0)
        bal = balanced_accuracy_score(all_l, all_p)
        acc = accuracy_score(all_l, all_p)

        if f1 > best_f1:
            best_f1, best_epoch, no_improve = f1, epoch + 1, 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            no_improve += 1

        if (epoch + 1) % 10 == 0 or (epoch + 1) == best_epoch:
            tag = " ← best" if (epoch + 1) == best_epoch else ""
            print(f"  Ep {epoch+1:03d} | Loss:{train_loss/len(train_loader):.4f} | "
                  f"Acc:{acc:.4f} | BalAcc:{bal:.4f} | F1:{f1:.4f}{tag}")

        if no_improve >= patience:
            print(f"  Early stop at epoch {epoch+1}")
            break

    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()
    print(f"\n  Best {stage_name} Val F1: {best_f1:.4f} at epoch {best_epoch}")
    return model, float(best_f1), int(best_epoch)

#  STAGE 1
print("\n" + "=" * 65)
print("STAGE 1 — BINARY DETECTOR (real vs fake)")
print(f"  train: {len(y_train_bin):,} | val: {len(y_val_bin):,}")
print(f"  train dist: {np.bincount(y_train_bin.cpu().numpy()).tolist()}")
print("=" * 65)

torch.manual_seed(SEED)
s1_model      = BinaryDetector(d=256, dropout=0.3).to(device)
s1_ckpt_local = f"{RESULTS_DIR}/best_stage1_binary_clean.pth"
s1_ckpt_drive = f"{DRIVE_CKPT}/best_stage1_binary_clean.pth"

s1_model, s1_f1, s1_epoch = train_stage(
    s1_model, X_train_t, y_train_bin, X_val_t, y_val_bin,
    num_classes=2, ckpt_path=s1_ckpt_local,
    lr=2e-4, weight_decay=2e-4, num_epochs=100,
    patience=25, gamma=2.5, stage_name="Stage1-Binary"
)
shutil.copy(s1_ckpt_local, s1_ckpt_drive)

#  STAGE 2
print("\n" + "=" * 65)
print("STAGE 2 — FAKE SUB-CLASSIFIER (fake_audio | fake_video | fake_both)")
print(f"  train: {len(y_train_fake):,} | val: {len(y_val_fake):,}")
print(f"  train dist: {np.bincount(y_train_fake.cpu().numpy()).tolist()}")
print("=" * 65)

torch.manual_seed(SEED)
s2_model      = FakeSubClassifier(d=256, dropout=0.3).to(device)
s2_ckpt_local = f"{RESULTS_DIR}/best_stage2_fake_sub_clean.pth"
s2_ckpt_drive = f"{DRIVE_CKPT}/best_stage2_fake_sub_clean.pth"

s2_model, s2_f1, s2_epoch = train_stage(
    s2_model, X_train_fake, y_train_fake, X_val_fake, y_val_fake,
    num_classes=3, ckpt_path=s2_ckpt_local,
    lr=2e-4, weight_decay=2e-4, num_epochs=100,
    patience=25, gamma=2.0, stage_name="Stage2-FakeSub"
)
shutil.copy(s2_ckpt_local, s2_ckpt_drive)

#  CHAINED PROBABILITIES
def two_stage_probs(s1_model, s2_model, X_tensor):
    s1_model.eval(); s2_model.eval()
    with torch.no_grad():
        s1_probs = torch.softmax(s1_model(X_tensor.to(device)), dim=1).cpu().numpy()
        s2_probs = torch.softmax(s2_model(X_tensor.to(device)), dim=1).cpu().numpy()

    p_real = s1_probs[:, 0]
    p_fake = s1_probs[:, 1]

    final_probs = np.zeros((len(X_tensor), 4), dtype=np.float32)
    final_probs[:, 0] = p_real
    final_probs[:, 1] = p_fake * s2_probs[:, 0]
    final_probs[:, 2] = p_fake * s2_probs[:, 1]
    final_probs[:, 3] = p_fake * s2_probs[:, 2]
    final_probs /= final_probs.sum(axis=1, keepdims=True)
    return final_probs

#  VALIDATION THRESHOLD TUNING
print("\n" + "=" * 65)
print("THRESHOLD TUNING — ON VAL ONLY (test never touched here)")
print("=" * 65)

val_probs     = two_stage_probs(s1_model, s2_model, X_val_t)
val_labels_np = y_val_t.cpu().numpy()

base_preds = val_probs.argmax(axis=1)
base_f1  = f1_score(val_labels_np, base_preds, average="macro", zero_division=0)
base_bal = balanced_accuracy_score(val_labels_np, base_preds)
print(f"Baseline val → F1: {base_f1:.4f} | BalAcc: {base_bal:.4f}")
print(f"Val pred dist: {np.bincount(base_preds).tolist()}")

best_val_f1 = -1.0
best_b0, best_b1 = 1.0, 1.0
grid_results = []

for b0 in np.arange(1.0, 6.0, 0.25):
    for b1 in np.arange(1.0, 4.0, 0.25):
        p = val_probs.copy()
        p[:, 0] *= b0
        p[:, 1] *= b1
        p /= p.sum(axis=1, keepdims=True)
        preds = p.argmax(axis=1)
        f1  = f1_score(val_labels_np, preds, average="macro", zero_division=0)
        bal = balanced_accuracy_score(val_labels_np, preds)
        grid_results.append((f1, bal, float(b0), float(b1)))
        if f1 > best_val_f1:
            best_val_f1, best_b0, best_b1 = f1, b0, b1

grid_results.sort(reverse=True)
print("\nTop 5 val configurations:")
print(f"{'F1':>7} {'BalAcc':>7}  b0(real)  b1(fake_audio)")
print("-" * 45)
for f1_g, bal_g, b0_g, b1_g in grid_results[:5]:
    print(f"  {f1_g:.4f}  {bal_g:.4f}  {b0_g:6.2f}x  {b1_g:6.2f}x")
print(f"\n✓ Best val F1: {best_val_f1:.4f}")
print(f"  real boost: {best_b0:.2f}x | fake_audio boost: {best_b1:.2f}x")

#  FINAL TEST — run exactly once
print("\n" + "=" * 65)
print("FINAL TEST — TWO-STAGE PIPELINE")
print(f"  Boosts from val: real={best_b0:.2f}x | fake_audio={best_b1:.2f}x")
print("=" * 65)

test_probs     = two_stage_probs(s1_model, s2_model, X_test_t)
test_labels_np = y_test_t.cpu().numpy()

p_test = test_probs.copy()
p_test[:, 0] *= best_b0
p_test[:, 1] *= best_b1
p_test /= p_test.sum(axis=1, keepdims=True)
test_preds = p_test.argmax(axis=1)

acc = accuracy_score(test_labels_np, test_preds)
bal = balanced_accuracy_score(test_labels_np, test_preds)
f1  = f1_score(test_labels_np, test_preds, average="macro", zero_division=0)
auc = roc_auc_score((test_labels_np != 0).astype(int), 1.0 - p_test[:, 0])
cm  = confusion_matrix(test_labels_np, test_preds)
pcf = f1_score(test_labels_np, test_preds, average=None, zero_division=0)

print(f"\nTest samples:      {len(test_labels_np)}")
print(f"Test pred dist:    {np.bincount(test_preds).tolist()}")
print(f"Accuracy:          {acc:.4f}")
print(f"Balanced Accuracy: {bal:.4f}")
print(f"Macro F1:          {f1:.4f}  ← report in paper")
print(f"Binary AUC:        {auc:.4f}")
print("\nPer-class F1:")
for i, name in enumerate(["real","fake_audio","fake_video","fake_both"]):
    print(f"  {name:<14} {pcf[i]:.4f}")
print("\nClassification Report:")
print(classification_report(
    test_labels_np, test_preds,
    target_names=["real","fake_audio","fake_video","fake_both"],
    zero_division=0))
print("Confusion Matrix:")
print(cm)

#  SAVE
two_stage_result = {
    "model": "Two-Stage MLP Pipeline",
    "split": {
        "train": int(X_train_t.shape[0]),
        "val":   int(X_val_t.shape[0]),
        "test":  int(X_test_t.shape[0]),
        "train_dist": train_counts,
        "val_dist":   val_counts,
        "test_dist":  test_counts,
    },
    "stage1": {
        "type":       "BinaryDetector (real vs fake)",
        "val_f1":     s1_f1,
        "best_epoch": s1_epoch,
        "checkpoint": s1_ckpt_drive,
    },
    "stage2": {
        "type":       "FakeSubClassifier (fake_audio|fake_video|fake_both)",
        "val_f1":     s2_f1,
        "best_epoch": s2_epoch,
        "checkpoint": s2_ckpt_drive,
    },
    "threshold_boosts": {
        "real":       float(best_b0),
        "fake_audio": float(best_b1),
    },
    "val_macro_f1_baseline": float(base_f1),
    "val_macro_f1_tuned":    float(best_val_f1),
    "test_samples":           int(len(test_labels_np)),
    "test_accuracy":          float(acc),
    "test_balanced_accuracy": float(bal),
    "test_macro_f1":          float(f1),
    "test_binary_auc":        float(auc),
    "test_per_class_f1": {
        "real":       float(pcf[0]),
        "fake_audio": float(pcf[1]),
        "fake_video": float(pcf[2]),
        "fake_both":  float(pcf[3]),
    },
    "test_confusion_matrix": cm.tolist(),
    "evaluation_note": (
        "Two-stage pipeline. All thresholds tuned on val only. "
        "Test run exactly once. No hardcoded split counts. "
        f"Split: {X_train_t.shape[0]} train / "
        f"{X_val_t.shape[0]} val / {X_test_t.shape[0]} test."
    ),
}

for path in [f"{RESULTS_DIR}/TRM_VFD_TWO_STAGE_FINAL_CLEAN.json",
             f"{DRIVE_RESULTS}/TRM_VFD_TWO_STAGE_FINAL_CLEAN.json"]:
    with open(path, "w") as fj:
        json.dump(two_stage_result, fj, indent=2)

print(f"\n✓ Saved locally and to Drive")
print(f"\n" + "=" * 65)
print("FINAL RESULT SUMMARY — TWO-STAGE MLP PIPELINE")
print("=" * 65)
print(f"  Stage1 Val F1:     {s1_f1:.4f} (epoch {s1_epoch})")
print(f"  Stage2 Val F1:     {s2_f1:.4f} (epoch {s2_epoch})")
print(f"  Val F1 (tuned):    {best_val_f1:.4f}")
print(f"  Test Macro F1:     {f1:.4f}  ← paper Table 2")
print(f"  Test Binary AUC:   {auc:.4f}  ← paper Table 3")
print(f"  Test BalAcc:       {bal:.4f}")
print("=" * 65)

In [ ]:
# CELL 31 │ Phase 11 │ Binary Consistency Table — Setup + Load Checkpoints
import inspect
RESULTS_DIR   = "/content/results"
DRIVE_RESULTS = "/content/drive/MyDrive/TRM_DATASETS/results"
DRIVE_CKPT    = "/content/drive/MyDrive/TRM_DATASETS/trm_checkpoint"

def to_tensor_float(x):
    if isinstance(x, torch.Tensor):
        return x.detach().clone().float()
    return torch.tensor(x, dtype=torch.float32)

def to_tensor_long(x):
    if isinstance(x, torch.Tensor):
        return x.detach().clone().long()
    return torch.tensor(x, dtype=torch.long)

X_val_t  = to_tensor_float(X_val if "X_val" in globals() else X_val)
y_val_t  = to_tensor_long(y_val if "y_val" in globals() else y_val)
X_test_t = to_tensor_float(X_test)
y_test_t = to_tensor_long(y_test)

print(f"Val  tensor: {X_val_t.shape}  labels: {np.bincount(y_val_t.numpy(), minlength=4)}")
print(f"Test tensor: {X_test_t.shape} labels: {np.bincount(y_test_t.numpy(), minlength=4)}")

val_loader_bin  = DataLoader(TensorDataset(X_val_t,  y_val_t),  batch_size=256, shuffle=False)
test_loader_bin = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=256, shuffle=False)

def best_threshold_from_val(y_true_bin, scores):
    best_thr, best_f1, best_bal = 0.5, -1.0, -1.0
    for thr in np.arange(0.05, 0.951, 0.01):
        preds = (scores >= thr).astype(int)
        f1 = f1_score(y_true_bin, preds, average="binary", zero_division=0)
        bal = balanced_accuracy_score(y_true_bin, preds)
        if (f1 > best_f1) or (abs(f1 - best_f1) < 1e-12 and bal > best_bal):
            best_f1, best_thr, best_bal = f1, thr, bal
    return float(best_thr), float(best_f1)

def eval_4class_as_binary(model, val_loader, test_loader, device):
    model.eval()

    def collect(loader):
        all_probs, all_labels = [], []
        with torch.no_grad():
            for xb, yb in loader:
                probs = torch.softmax(model(xb.to(device)), dim=1).cpu().numpy()
                all_probs.append(probs)
                all_labels.append(yb.numpy())
        return np.concatenate(all_probs, axis=0), np.concatenate(all_labels, axis=0)

    val_probs_m,  val_labels_m  = collect(val_loader)
    test_probs_m, test_labels_m = collect(test_loader)

    y_val_bin  = (val_labels_m  != 0).astype(int)
    y_test_bin = (test_labels_m != 0).astype(int)

    val_scores  = 1.0 - val_probs_m[:, 0]
    test_scores = 1.0 - test_probs_m[:, 0]

    thr, val_best_f1 = best_threshold_from_val(y_val_bin, val_scores)
    test_preds = (test_scores >= thr).astype(int)

    return {
        "val_threshold": thr,
        "val_binary_f1_at_thr": val_best_f1,
        "test_binary_auc": float(roc_auc_score(y_test_bin, test_scores)),
        "test_binary_f1": float(f1_score(y_test_bin, test_preds, average="binary", zero_division=0)),
        "test_binary_balacc": float(balanced_accuracy_score(y_test_bin, test_preds)),
        "test_binary_acc": float(accuracy_score(y_test_bin, test_preds)),
    }

def eval_probs_as_binary(val_probs_arr, val_labels_arr, test_probs_arr, test_labels_arr):
    y_val_bin  = (np.array(val_labels_arr)  != 0).astype(int)
    y_test_bin = (np.array(test_labels_arr) != 0).astype(int)

    val_scores  = 1.0 - np.array(val_probs_arr)[:, 0]
    test_scores = 1.0 - np.array(test_probs_arr)[:, 0]

    thr, val_best_f1 = best_threshold_from_val(y_val_bin, val_scores)
    test_preds = (test_scores >= thr).astype(int)

    return {
        "val_threshold": thr,
        "val_binary_f1_at_thr": val_best_f1,
        "test_binary_auc": float(roc_auc_score(y_test_bin, test_scores)),
        "test_binary_f1": float(f1_score(y_test_bin, test_preds, average="binary", zero_division=0)),
        "test_binary_balacc": float(balanced_accuracy_score(y_test_bin, test_preds)),
        "test_binary_acc": float(accuracy_score(y_test_bin, test_preds)),
    }

def build_model_flex(cls, **preferred_kwargs):
    sig = inspect.signature(cls.__init__)
    allowed = set(sig.parameters.keys()) - {"self"}
    filtered = {k: v for k, v in preferred_kwargs.items() if k in allowed}
    print(f"Building {cls.__name__} with args: {filtered}")
    return cls(**filtered)

results = []

# Embedding MLP
emb_ckpt = f"{DRIVE_CKPT}/best_emb_mlp.pth"
if os.path.exists(emb_ckpt):
    _m = build_model_flex(EmbeddingMLP, input_dim=704, num_classes=4).to(device)
    _m.load_state_dict(torch.load(emb_ckpt, map_location=device))
    r = eval_4class_as_binary(_m, val_loader_bin, test_loader_bin, device)
    r["model"] = "Embedding MLP"
    results.append(r)

# One-Shot Transformer
ost_ckpt = f"{DRIVE_CKPT}/best_ost.pth"
if os.path.exists(ost_ckpt):
    _m = build_model_flex(
        OneShotTransformer,
        face_dim=512, voice_dim=192,
        proj_dim=256, num_heads=4,
        num_classes=4
    ).to(device)
    _m.load_state_dict(torch.load(ost_ckpt, map_location=device))
    r = eval_4class_as_binary(_m, val_loader_bin, test_loader_bin, device)
    r["model"] = "One-Shot Transformer"
    results.append(r)

# Simplified TRM
strm_ckpt = f"{DRIVE_CKPT}/best_strm.pth"
if os.path.exists(strm_ckpt):
    _m = build_model_flex(
        SimplifiedTRM,
        face_dim=512, voice_dim=192,
        proj_dim=256, d=256, hidden_dim=256,
        num_heads=4, heads=4,
        num_steps=4, steps=4,
        num_classes=4
    ).to(device)
    _m.load_state_dict(torch.load(strm_ckpt, map_location=device))
    r = eval_4class_as_binary(_m, val_loader_bin, test_loader_bin, device)
    r["model"] = "Simplified TRM"
    results.append(r)

# Full TRM-VFD
trm_ckpt = f"{DRIVE_CKPT}/best_trm_FINAL.pth"
if os.path.exists(trm_ckpt):
    _m = build_model_flex(
        TRM_VFD,
        face_dim=512, voice_dim=192,
        proj_dim=256, num_heads=4,
        num_steps=5, num_classes=4
    ).to(device)
    _m.load_state_dict(torch.load(trm_ckpt, map_location=device))
    r = eval_4class_as_binary(_m, val_loader_bin, test_loader_bin, device)
    r["model"] = "Full TRM-VFD"
    results.append(r)

# Two-Stage MLP Pipeline
two_stage_json = f"{RESULTS_DIR}/TRM_VFD_TWO_STAGE_FINAL_CLEAN.json"
if os.path.exists(two_stage_json):
    with open(two_stage_json, "r") as f:
        d = json.load(f)
    results.append({
        "model":                "Two-Stage MLP Pipeline",
        "val_threshold":        np.nan,
        "val_binary_f1_at_thr": np.nan,
        "test_binary_auc":      float(d["test_binary_auc"]),
        "test_binary_f1":       float(d.get("test_per_class_f1", {})
                                       .get("fake_both", np.nan)),
        "test_binary_balacc":   float(d["test_balanced_accuracy"]),
        "test_binary_acc":      float(d["test_accuracy"]),
    })
    print(f"✓ Two-Stage loaded from JSON: AUC={d['test_binary_auc']:.4f}")
else:
    print("⚠ Two-Stage JSON not found")

# Improved Full TRM
improved_json = f"{RESULTS_DIR}/IMPROVED_FULL_TRM_CE_FINAL.json"
if os.path.exists(improved_json):
    with open(improved_json, "r") as f:
        d = json.load(f)
    results.append({
        "model":                "Improved Full TRM",
        "val_threshold":        np.nan,
        "test_binary_auc":      float(d["test_binary_auc"]),
        "test_binary_f1":       float(d.get("test_per_class_f1", {})
                                       .get("fake_both", np.nan)),
        "test_binary_balacc":   float(d["test_balanced_accuracy"]),
        "test_binary_acc":      float(d["test_accuracy"]),
    })

# VFD Cosine
vfd_json = "/content/results/B4_vfd.json"
if os.path.exists(vfd_json):
    with open(vfd_json, "r") as f:
        d = json.load(f)
    results.append({
        "model":               "VFD Cosine",
        "val_threshold":       np.nan,
        "val_binary_f1_at_thr": np.nan,
        "test_binary_auc":     d["test_binary_auc"],
        "test_binary_f1":      d["test_macro_f1"],
        "test_binary_balacc":  d["test_balanced_accuracy"],
        "test_binary_acc":     d["test_accuracy"],
    })
else:
    print("⚠ B4_vfd.json not found — run VFD cell first")

# MVF binary baseline
mvf_json = "/content/results/B3_mvf.json"
if os.path.exists(mvf_json):
    with open(mvf_json, "r") as f:
        d = json.load(f)
    results.append({
        "model": "MVF (binary baseline)",
        "val_threshold": np.nan,
        "val_binary_f1_at_thr": np.nan,
        "test_binary_auc": d["test_binary_auc"],
        "test_binary_f1": d["test_binary_f1"],
        "test_binary_balacc": d["test_balanced_accuracy"],
        "test_binary_acc": d["test_accuracy"],
    })
else:
    print("⚠ MVF binary JSON not found. Rerun patched Cell 18.")

df_bin = (
    pd.DataFrame(results)
    .sort_values("test_binary_auc", ascending=False)
    .reset_index(drop=True)
)

print("\n" + "=" * 72)
print("BINARY CONSISTENCY TABLE — FINAL")
print("=" * 72)
print(df_bin[[
    "model",
    "test_binary_auc",
    "test_binary_f1",
    "test_binary_balacc",
    "test_binary_acc",
    "val_threshold",
]].round(2).to_string(index=False))

csv_path  = "/content/results/binary_consistency_table_final.csv"
json_path = "/content/results/binary_consistency_table_final.json"
drv_path  = "/content/drive/MyDrive/TRM_DATASETS/results/binary_consistency_table_final.json"

df_bin.to_csv(csv_path, index=False)
df_bin.to_json(json_path, orient="records", indent=2)
df_bin.to_json(drv_path, orient="records", indent=2)

print(f"\n✓ Saved: {csv_path}")
print(f"✓ Saved: {json_path}")
print(f"✓ Saved: {drv_path}")

In [ ]:
# CELL 32 │ Phase 11 │ Ablation — Recursive Steps K=1 to 5
# Purpose: show recursion helps
# Uses VAL set — this is hyperparameter selection
# NOT final evaluation (that uses test)

import pandas as pd
import matplotlib.pyplot as plt
import os, shutil

print("="*50)
print("ABLATION: num_steps 1 to 5")
print("30 epochs | lr=5e-5 | seed=42")
print("Val set used — hyperparameter study")
print("="*50)

ablation_results = []

for steps in [1, 2, 3, 4, 5]:
    torch.manual_seed(42)
    np.random.seed(42)

    m = TRM_VFD(num_steps=steps).to(device)
    c = FocalLoss(alpha=weights_emb, gamma=2.0)
    o = optim.Adam(
        m.parameters(),
        lr=5e-5,
        weight_decay=1e-4)
    s = optim.lr_scheduler.CosineAnnealingLR(
        o, T_max=30)

    best_f1 = 0
    for epoch in range(30):
        m.train()
        for xb, yb in train_loader_emb:
            xb, yb = xb.to(device), yb.to(device)
            o.zero_grad()
            loss = c(m(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                m.parameters(), 1.0)
            o.step()
        s.step()

        m.eval()
        all_p, all_l = [], []
        with torch.no_grad():
            for xb, yb in val_loader_emb:  # ← VAL correct
                all_p.extend(
                    m(xb.to(device))
                    .argmax(1).cpu().numpy())
                all_l.extend(yb.numpy())
        f1 = f1_score(
            all_l, all_p,
            average='macro', zero_division=0)
        if f1 > best_f1:
            best_f1 = f1

    ablation_results.append({
        "num_steps": steps,
        "macro_f1":  round(best_f1, 4)
    })
    print(f"K={steps} | Val F1={best_f1:.4f}")

# Save
df_ab = pd.DataFrame(ablation_results)

os.makedirs("/content/results", exist_ok=True)
os.makedirs("/content/report_figures", exist_ok=True)
DRIVE_DIR = "/content/drive/MyDrive/TRM_DATASETS"

df_ab.to_csv(
    "/content/results/ablation_num_steps.csv",
    index=False)
df_ab.to_csv(
    f"{DRIVE_DIR}/ablation_num_steps.csv",
    index=False)

# Print table
print("\nABLATION TABLE (val set)")
print("="*40)
f1_vals = df_ab["macro_f1"].tolist()
for _, row in df_ab.iterrows():
    bar = "#" * int(row["macro_f1"] * 40)
    print(f"  K={int(row['num_steps'])} | "
          f"F1={row['macro_f1']:.4f} | {bar}")

# Check monotonic
is_mono = all(
    f1_vals[i] <= f1_vals[i+1]
    for i in range(len(f1_vals)-1))
print(f"\nMonotonic: "
      f"{'YES ✓' if is_mono else 'NOT MONOTONIC'}")

optimal_steps = int(
    df_ab.loc[df_ab["macro_f1"].idxmax(), "num_steps"])
optimal_f1    = df_ab["macro_f1"].max()
print(f"Optimal: K={optimal_steps} | F1={optimal_f1:.4f}")

# Plot
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df_ab["num_steps"], df_ab["macro_f1"],
        marker='o', color='#9b59b6',
        linewidth=2, markersize=8)
ax.scatter(
    [optimal_steps],
    [optimal_f1],
    color='#e91e63', s=180, zorder=5,
    label=f"Chosen (K={optimal_steps}, F1={optimal_f1:.3f})")
ax.set_xlabel("Number of Recursive Steps (K)")
ax.set_ylabel("Macro F1 Score (val set)")
ax.set_title(
    "Ablation: Recursive Steps in TRM-VFD\n"
    "(30 epochs, lr=5e-5, seed=42, val set)")
ax.set_xticks([1, 2, 3, 4, 5])
ax.grid(True, alpha=0.3)
ax.legend()
for _, row in df_ab.iterrows():
    ax.annotate(
        f"{row['macro_f1']:.3f}",
        (row['num_steps'], row['macro_f1']),
        textcoords="offset points",
        xytext=(0, 12),
        ha='center', fontsize=10)
plt.tight_layout()
plt.savefig(
    "/content/report_figures/ablation_steps.png",
    dpi=150)
plt.savefig(
    f"{DRIVE_DIR}/ablation_steps.png",
    dpi=150)
plt.show()
plt.close()
print("✓ Ablation chart saved")
print("\nNote: ablation uses val set for hyperparameter")
print("selection. Final results use held-out test set.")

In [ ]:
# CELL 33 │ Phase 11 │ Ablation — Plot Recursive Steps Chart
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df_ab["num_steps"], df_ab["macro_f1"],
        marker='o', color='#9b59b6', linewidth=2, markersize=8)

ax.scatter([optimal_steps], [optimal_f1],
           color='#e91e63', s=180, zorder=5,
           label=f"Chosen (steps={optimal_steps}, F1={optimal_f1:.3f})")

ax.set_xlabel("Number of Recursive Steps")
ax.set_ylabel("Macro F1 Score")
ax.set_title("Ablation: Recursive Steps in TRM-VFD\n"
             "(30 epochs each, identical conditions)")  # ← honest
ax.set_xticks([1, 2, 3, 4, 5])
ax.grid(True, alpha=0.3)
ax.legend()

for _, row in df_ab.iterrows():
    ax.annotate(f"{row['macro_f1']:.3f}",
                (row['num_steps'], row['macro_f1']),
                textcoords="offset points",
                xytext=(0, 10), ha='center', fontsize=10)

plt.tight_layout()
plt.savefig("/content/report_figures/ablation_steps.png", dpi=150)
plt.show()
plt.close()
print("Ablation chart saved")


In [ ]:
# CELL 34 │ Phase 11 │ Visualizations — Confidence + Confusion Matrix
trm_model.eval()
label_names = ["real", "fake_audio", "fake_video", "fake_both"]
colors      = ["#2ecc71", "#e74c3c", "#3498db", "#9b59b6"]

class_correct_conf = []

for cls in range(4):
    correct_confs = []
    for i in idx:
        with torch.no_grad():
            logits = trm_model(x)
            probs  = torch.softmax(logits, dim=1).squeeze().cpu().numpy()
        correct_confs.append(probs[cls])
    class_correct_conf.append(
        np.mean(correct_confs) if correct_confs else 0.0)

#  FIGURE 1: Per-class prediction confidence
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(label_names, class_correct_conf,
            color=colors, edgecolor='white', width=0.5)
axes[0].set_ylabel("Mean Confidence on Correct Class")
axes[0].set_title("TRM-VFD: Mean Prediction Confidence Per Class")
axes[0].set_ylim(0, 1.0)
axes[0].axhline(y=0.25, color='black', linestyle='--',
                linewidth=1, alpha=0.5, label='Random baseline (0.25)')
axes[0].legend()
for i, val in enumerate(class_correct_conf):
    axes[0].text(i, val + 0.02, f"{val:.3f}",
                 ha='center', fontsize=10, fontweight='bold')

# FIGURE 2: Confusion heatmap (normalized)
with open("/content/results/TRM_VFD.json") as f:
    trm_r = json.load(f)

cm = np.array(trm_r["test_confusion_matrix"])
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

im = axes[1].imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=axes[1])
axes[1].set_xticks(range(4)); axes[1].set_yticks(range(4))
axes[1].set_xticklabels(label_names, rotation=30, ha='right')
axes[1].set_yticklabels(label_names)
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")
axes[1].set_title("TRM-VFD: Normalized Confusion Matrix (Test Set)")
for i in range(4):
    for j in range(4):
        axes[1].text(j, i, f"{cm_norm[i,j]:.2f}",
                     ha='center', va='center', fontsize=10,
                     fontweight='bold',
                     color='white' if cm_norm[i,j] > 0.5 else 'black')

plt.tight_layout()
plt.savefig("/content/report_figures/attention_visualization.png", dpi=150)
plt.savefig(
    "/content/drive/MyDrive/TRM_DATASETS/attention_visualization.png",
    dpi=150)
plt.show()
plt.close()

with open("/content/results/TRM_VFD.json") as f:
    r = json.load(f)

if not r.get("test_binary_auc"):
    trm_model.eval()
    all_sc, all_bin = [], []
    with torch.no_grad():
        for xb, yb in test_loader:       # ← use test_loader, not val
            probs = torch.softmax(trm_model(xb.to(device)), dim=1)
            all_sc.extend(probs.cpu().numpy())
            all_bin.extend((yb.numpy() != 0).astype(int))
    auc = roc_auc_score(all_bin, 1 - np.array(all_sc)[:, 0])
    r["test_binary_auc"] = round(float(auc), 4)
    with open("/content/results/TRM_VFD.json", "w") as f:
        json.dump(r, f, indent=2)
    print(f"AUC added to TRM_VFD.json: {auc:.4f}")

print("✓ Confidence visualization saved")
print("\nPer-class confidence (val set):")
for name, conf in zip(label_names, class_correct_conf):
    print(f"  {name}: {conf:.4f}")

In [ ]:
# CELL 35 │ Phase 11 │ Main Results Table (Table 2)
# Table 2: 4-class comparable results (8 models, same test set)
# Table 3: Binary/reference results (MVF + cross-dataset note)
# MVF is NOT in Table 2 — binary-only, not 4-class comparable
import os, json
import numpy as np
import pandas as pd

RESULTS_DIR   = "/content/results"
DRIVE_RESULTS = "/content/drive/MyDrive/TRM_DATASETS/results"
PREFERRED_DIRS = [DRIVE_RESULTS, RESULTS_DIR]

def load_json_prefer_order(fname):
    for base in PREFERRED_DIRS:
        path = os.path.join(base, fname)
        if os.path.exists(path):
            with open(path, "r") as f:
                return json.load(f), path
    return None, None

def first_non_null(*vals):
    for v in vals:
        if v is not None:
            return v
    return None

def extract_metrics(data):
    if data is None:
        return np.nan, np.nan, np.nan, np.nan
    acc = first_non_null(
        data.get("test_accuracy"),
        data.get("accuracy"))
    bal = first_non_null(
        data.get("test_balanced_accuracy"),
        data.get("balanced_accuracy"))
    f1  = first_non_null(
        data.get("test_macro_f1"),
        data.get("macro_f1"))
    auc = first_non_null(
        data.get("test_binary_auc"),
        data.get("binary_auc"),
        data.get("auc"))
    return (
        float(acc) if acc is not None else np.nan,
        float(bal) if bal is not None else np.nan,
        float(f1)  if f1  is not None else np.nan,
        float(auc) if auc is not None else np.nan,
    )

# TABLE 2 — 4-class comparable models only
TABLE2_REGISTRY = [
    ("B1_pixel_mlp.json",                  "Pixel MLP (lower bound)"),
    # VFD Cosine omitted — binary-only objective, in Table 3 only
    ("IMPROVED_FULL_TRM_CE_FINAL.json",    "Improved Full TRM"),
    ("B5_ost.json",                        "One-Shot Transformer"),
    ("TRM_VFD.json",                       "Full TRM-VFD"),
    ("TRM_VFD_TWO_STAGE_FINAL_CLEAN.json", "Two-Stage MLP Pipeline"),
    ("B2_emb_mlp.json",                    "Embedding MLP"),
    ("B6_strm.json",                       "Simplified TRM"),
]

rows2 = []
missing2 = []
for fname, model_name in TABLE2_REGISTRY:
    data, path = load_json_prefer_order(fname)
    if data is None:
        missing2.append(fname)
        rows2.append({
            "Model": model_name,
            "Accuracy": np.nan,
            "Balanced Acc.": np.nan,
            "Macro F1": np.nan,
            "Binary AUC": np.nan,
        })
        continue
    acc, bal, f1, auc = extract_metrics(data)
    rows2.append({
        "Model":          model_name,
        "Accuracy":       acc,
        "Balanced Acc.":  bal,
        "Macro F1":       f1,
        "Binary AUC":     auc,
    })

df2 = pd.DataFrame(rows2)
df2["Rank"] = df2["Macro F1"].rank(
    ascending=False, method="min", na_option="bottom").astype(int)
df2 = df2.sort_values("Rank").reset_index(drop=True)

print("=" * 90)
print("TABLE 2 — 4-CLASS COMPARABLE RESULTS")
print("Identity-disjoint test set | n=2,000 | Primary: Macro F1")
print("=" * 90)
print(df2[["Model","Accuracy","Balanced Acc.",
           "Macro F1","Binary AUC","Rank"]
     ].round(2).to_string(index=False))
print("\n‡ VFD Cosine: binary detection only — no 4-class supervision")
print("★ Two-Stage MLP Pipeline: best Macro F1")
print("★ Full TRM-VFD: best Binary AUC")

# TABLE 3 — Binary/Reference results
print("\n" + "=" * 90)
print("TABLE 3 — BINARY CONSISTENCY (ALL MODELS, TEST SET n=2,000)")
print("Binary scores: 1-P(real) for 4-class models | Stage-1 P(fake) for Two-Stage")
print("=" * 90)

# Load binary AUC from all model JSONs
binary_rows = [
    ("Full TRM-VFD",         "TRM_VFD.json",                    True),
    ("One-Shot Transformer",  "B5_ost.json",                     True),
    ("Simplified TRM",        "B6_strm.json",                    True),
    ("Embedding MLP",         "B2_emb_mlp.json",                 True),
    ("Two-Stage MLP",         "TRM_VFD_TWO_STAGE_FINAL_CLEAN.json", True),
    ("Improved Full TRM",     "IMPROVED_FULL_TRM_CE_FINAL.json", False),
    ("VFD Cosine",            "B4_vfd.json",                     False),
    ("MVF (published)†",      "B3_mvf.json",                     False),
]

bin_data = []
for model_name, fname, threshold_tuned in binary_rows:
    data, path = load_json_prefer_order(fname)
    if data is None:
        continue
    auc = first_non_null(
        data.get("test_binary_auc"),
        data.get("binary_auc"),
        data.get("auc"))
    bin_data.append({
        "Model":           model_name,
        "Binary AUC":      round(float(auc), 2) if auc else float("nan"),
        "Threshold Tuned": "Yes" if threshold_tuned else "No",
    })

df3 = pd.DataFrame(bin_data).sort_values(
    "Binary AUC", ascending=False).reset_index(drop=True)

print(df3.to_string(index=False))
print("\n† MVF: threshold=0.0 (raw similarity). Binary F1=0.0 because")
print("  all samples predicted as 'consistent'. AUC=0.66 uses raw scores.")
print("  Not comparable to 4-class models — reference only.")

# Save both tables
for path in [f"{RESULTS_DIR}/table2_4class.csv",
             f"{DRIVE_RESULTS}/table2_4class.csv"]:
    df2.to_csv(path, index=False)

for path in [f"{RESULTS_DIR}/table3_binary_reference.csv",
             f"{DRIVE_RESULTS}/table3_binary_reference.csv"]:
    df3.to_csv(path, index=False)

# Combined JSON for reproducibility
combined = {
    "table2_4class":           df2.round(2).to_dict(orient="records"),
    "table3_binary_reference": df3.round(2).to_dict(orient="records"),
    "split":  "17017 train / 2100 val / 2000 test",
    "protocol": "identity-disjoint, val=selection only, test=once",
    "note": "MVF excluded from Table 2 — binary-only model."
}
for path in [f"{RESULTS_DIR}/main_results_table_final_correct.json",
             f"{DRIVE_RESULTS}/main_results_table_final_correct.json"]:
    with open(path, "w") as fj:
        json.dump(combined, fj, indent=2)

if missing2:
    print(f"\n⚠ Missing files: {missing2}")

print(f"\n✓ Saved: table2_4class.csv")
print(f"✓ Saved: table3_binary_reference.csv")
print(f"✓ Saved: main_results_table_final_correct.json")
best_f1_row = df2.loc[df2["Macro F1"].idxmax()]
best_auc_row = df3.loc[df3["Binary AUC"].idxmax()]
mvf_row = df3[df3["Model"].str.contains("MVF", case=False, na=False)]

print(f"\n Best Macro F1:   {best_f1_row['Model']}    {best_f1_row['Macro F1']:.2f}")
print(f" Best Binary AUC: {best_auc_row['Model']}    {best_auc_row['Binary AUC']:.2f}")
if len(mvf_row):
    r = mvf_row.iloc[0]
    print(f"MVF Binary AUC:  {r['Binary AUC']:.2f} (reference only, Table 3)")

In [ ]:
# CELL 36 │ Phase 11 │ ROC Curves — Selected Test-Set Models
# SELF-CONTAINED: Works after complete Colab session restart.
# Mounts Drive, rebuilds models, loads checkpoints, computes
# ROC live from test set. NO saved .npy arrays used.
import os, json
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc as sklearn_auc
from torch.utils.data import DataLoader, TensorDataset
from PIL import Image
from torchvision import transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
np.random.seed(42)
print(f"✓ Device: {device}")

from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
    print("✓ Drive mounted")
else:
    print("✓ Drive already mounted")

RESULTS_DIR = "/content/results"
DRIVE_CKPT  = "/content/drive/MyDrive/TRM_DATASETS/trm_checkpoint"
DRIVE_EMB   = "/content/drive/MyDrive/TRM_DATASETS/embeddings_mvf_3way"
DRIVE_FIG   = "/content/drive/MyDrive/TRM_DATASETS"
PIX_SPLIT   = "/content/MVF_data/test_frame.txt"
os.makedirs("/content/report_figures", exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

assert os.path.exists(DRIVE_CKPT), \
    f"❌ Checkpoint dir not found: {DRIVE_CKPT}"
assert os.path.exists(DRIVE_EMB), \
    f"❌ Embedding dir not found: {DRIVE_EMB}"
print(f"✓ Checkpoints: {DRIVE_CKPT}")
print(f"✓ Embeddings:  {DRIVE_EMB}")

print("\n" + "=" * 60)
print("STEP 1 — Rebuilding model architectures")
print("=" * 60)

class EmbeddingMLP(nn.Module):
    def __init__(self, input_dim=704, num_classes=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes))
    def forward(self, x): return self.net(x)

class SimpleMLP(nn.Module):
    def __init__(self, input_dim=12288, num_classes=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes))
    def forward(self, x): return self.net(x)

class OneShotTransformer(nn.Module):
    def __init__(self, face_dim=512, voice_dim=192,
                 proj_dim=256, num_heads=4, num_classes=4):
        super().__init__()
        self.face_proj  = nn.Sequential(
            nn.Linear(face_dim, proj_dim),
            nn.LayerNorm(proj_dim))
        self.voice_proj = nn.Sequential(
            nn.Linear(voice_dim, proj_dim),
            nn.LayerNorm(proj_dim))
        self.cross_attn = nn.MultiheadAttention(
            proj_dim, num_heads,
            batch_first=True, dropout=0.1)
        self.attn_norm  = nn.LayerNorm(proj_dim)
        self.classifier = nn.Sequential(
            nn.Linear(proj_dim, 64), nn.GELU(),
            nn.Dropout(0.3), nn.Linear(64, num_classes))
    def forward(self, x):
        f = self.face_proj(x[:, :512]).unsqueeze(1)
        v = self.voice_proj(x[:, 512:]).unsqueeze(1)
        fused, _ = self.cross_attn(
            query=f, key=v, value=v)
        fused = self.attn_norm(
            fused.squeeze(1) + f.squeeze(1))
        return self.classifier(fused)

class SimplifiedTRM(nn.Module):
    def __init__(self, face_dim=512, voice_dim=192,
                 hidden=256, steps=4, num_classes=4):
        super().__init__()
        self.steps      = steps
        self.hidden     = hidden
        self.face_proj  = nn.Linear(face_dim,  hidden)
        self.voice_proj = nn.Linear(voice_dim, hidden)
        self.gru        = nn.GRUCell(hidden * 2, hidden)
        self.classifier = nn.Sequential(
            nn.Linear(hidden, 64), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(64, num_classes))
    def forward(self, x):
        f = self.face_proj(x[:, :512])
        v = self.voice_proj(x[:, 512:])
        c = torch.cat([f, v], dim=1)
        h = torch.zeros(
            x.size(0), self.hidden, device=x.device)
        for _ in range(self.steps):
            h = self.gru(c, h)
        return self.classifier(h)

class TRM_VFD(nn.Module):
    def __init__(self, face_dim=512, voice_dim=192,
                 proj_dim=256, num_heads=4,
                 num_steps=5, num_classes=4):
        super().__init__()
        self.num_steps  = num_steps
        self.face_proj  = nn.Sequential(
            nn.Linear(face_dim, proj_dim),
            nn.LayerNorm(proj_dim))
        self.voice_proj = nn.Sequential(
            nn.Linear(voice_dim, proj_dim),
            nn.LayerNorm(proj_dim))
        self.cross_attn = nn.MultiheadAttention(
            proj_dim, num_heads, batch_first=True)
        self.attn_norm  = nn.LayerNorm(proj_dim)
        self.gru        = nn.GRUCell(proj_dim, proj_dim)
        self.classifier = nn.Sequential(
            nn.Linear(proj_dim, 64), nn.GELU(),
            nn.Dropout(0.3), nn.Linear(64, num_classes))
    def forward(self, x):
        f = self.face_proj(x[:, :512]).unsqueeze(1)
        v = self.voice_proj(x[:, 512:]).unsqueeze(1)
        h = torch.zeros(
            x.size(0), f.size(2), device=x.device)
        for _ in range(self.num_steps):
            attn_out, _ = self.cross_attn(
                query=f, key=v, value=v)
            f = self.attn_norm(attn_out + f)
            h = self.gru(f.squeeze(1), h)
        return self.classifier(h)

class BinaryDetector(nn.Module):
    def __init__(self, input_dim=704,
                 d=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, d),
            nn.LayerNorm(d), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d, d),
            nn.LayerNorm(d), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d, 64), nn.GELU(),
            nn.Dropout(0.2), nn.Linear(64, 2))
    def forward(self, x): return self.net(x)

# Instantiate
emb_mlp    = EmbeddingMLP().to(device)
pixel_mlp  = SimpleMLP(input_dim=12288).to(device)
ost_model  = OneShotTransformer().to(device)
strm_model = SimplifiedTRM().to(device)
trm_model  = TRM_VFD(num_steps=5).to(device)
s1_model   = BinaryDetector().to(device)
print("✓ All model architectures instantiated")

print("\n" + "=" * 60)
print("STEP 2 — Loading best checkpoints from Drive")
print("=" * 60)

ckpt_map = {
    "TRM_VFD":        (trm_model,   "best_trm_FINAL.pth"),
    "SimplifiedTRM":  (strm_model,  "best_strm.pth"),
    "OneShotTRM":     (ost_model,   "best_ost.pth"),
    "EmbeddingMLP":   (emb_mlp,     "best_emb_mlp.pth"),
    "SimpleMLP":      (pixel_mlp,   "best_pixel_mlp.pth"),
    "BinaryDetector": (s1_model,    "best_stage1_binary_clean.pth"),
}

loaded = {}
for key, (model_obj, fname) in ckpt_map.items():
    path = f"{DRIVE_CKPT}/{fname}"
    if os.path.exists(path):
        model_obj.load_state_dict(
            torch.load(path, map_location=device))
        model_obj.eval()
        loaded[key] = True
        print(f"  ✓ {key:<18} ← {fname}")
    else:
        loaded[key] = False
        print(f"  ⚠ {key:<18} NOT FOUND: {fname}")

print("\n" + "=" * 60)
print("STEP 3 — Building test loaders")
print("=" * 60)

# Embedding test loader
X_test = np.load(f"{DRIVE_EMB}/X_test.npy")
y_test = np.load(f"{DRIVE_EMB}/y_test.npy")
assert X_test.shape == (2000, 704), \
    f"❌ Wrong shape: {X_test.shape}"
assert len(np.unique(y_test)) == 4, \
    "❌ Missing classes in test set"

X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)
test_loader = DataLoader(
    TensorDataset(X_test_t, y_test_t),
    batch_size=256, shuffle=False)
print(f"  ✓ Embedding test loader: "
      f"{X_test.shape[0]:,} samples | "
      f"dist: {np.bincount(y_test, minlength=4).tolist()}")

# Pixel test loader
test_loader_pix = None
if os.path.exists(PIX_SPLIT):
    transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor()])
    X_pix, y_pix = [], []
    with open(PIX_SPLIT) as f:
        lines = f.readlines()
    for line in lines:
        parts = line.strip().split()
        if len(parts) != 2:
            continue
        try:
            img = Image.open(parts[0]).convert("RGB")
            X_pix.append(
                transform(img).numpy().reshape(-1))
            y_pix.append(int(parts[1]))
        except Exception:
            continue
    if X_pix:
        X_pix_t = torch.tensor(
            np.array(X_pix, dtype=np.float32))
        y_pix_t = torch.tensor(
            np.array(y_pix, dtype=np.int64))
        test_loader_pix = DataLoader(
            TensorDataset(X_pix_t, y_pix_t),
            batch_size=256, shuffle=False)
        print(f"  ✓ Pixel test loader:     "
              f"{len(X_pix):,} samples | dim=12288")
    else:
        print("  ⚠ No valid pixel images found")
else:
    print(f"  ⚠ Pixel split not found: {PIX_SPLIT}")

# STEP 4 — Build model_configs (only loaded models)
print("\n" + "=" * 60)
print("STEP 4 — Building model list")
print("=" * 60)

model_configs = []

if loaded.get("TRM_VFD", False):
    model_configs.append((
        "Full TRM-VFD",
        trm_model, test_loader,
        "#e91e63", "TRM_VFD.json"))
    print("  ✓ Full TRM-VFD")

if loaded.get("SimplifiedTRM", False):
    model_configs.append((
        "Simplified TRM (GRU)",
        strm_model, test_loader,
        "#9b59b6", "B6_strm.json"))
    print("  ✓ Simplified TRM (GRU)")

if loaded.get("OneShotTRM", False):
    model_configs.append((
        "One-Shot Transformer",
        ost_model, test_loader,
        "#3498db", "B5_ost.json"))
    print("  ✓ One-Shot Transformer")

if loaded.get("EmbeddingMLP", False):
    model_configs.append((
        "Embedding MLP",
        emb_mlp, test_loader,
        "#2ecc71", "B2_emb_mlp.json"))
    print("  ✓ Embedding MLP")

if loaded.get("SimpleMLP", False) and \
        test_loader_pix is not None:
    try:
        sample_dim = \
            test_loader_pix.dataset[0][0].shape[0]
        if sample_dim == 12288:
            model_configs.append((
                "Pixel MLP",
                pixel_mlp, test_loader_pix,
                "#e67e22", "B1_pixel_mlp.json"))
            print("  ✓ Pixel MLP (dim=12288)")
        else:
            print(f"  ⚠ Pixel MLP skipped "
                  f"— dim={sample_dim}, need 12288")
    except Exception as e:
        print(f"  ⚠ Pixel MLP skipped — {e}")
else:
    print("  ⚠ Pixel MLP skipped "
          "— checkpoint or loader missing")

print(f"\n  {len(model_configs)} models ready for ROC")

# STEP 5 — Compute ROC curves (live inference)
print("\n" + "=" * 60)
print("STEP 5 — Computing ROC (live, TEST SET n=2,000)")
print("=" * 60)

fig, ax = plt.subplots(figsize=(9, 8))
roc_results = {}

for name, model, loader, color, json_fname in model_configs:
    model.eval()
    scores, binary = [], []
    with torch.no_grad():
        for xb, yb in loader:
            probs = torch.softmax(
                model(xb.to(device)), dim=1)
            scores.extend(
                (1.0 - probs[:, 0]).cpu().numpy())
            binary.extend(
                (yb.numpy() != 0).astype(int))

    scores = np.array(scores, dtype=np.float32)
    binary = np.array(binary, dtype=np.int32)

    assert len(scores) == 2000, \
        f"❌ {name}: wrong count {len(scores)}"
    assert set(np.unique(binary)) <= {0, 1}, \
        f"❌ {name}: bad binary labels"

    fpr, tpr, _ = roc_curve(binary, scores)
    roc_auc     = sklearn_auc(fpr, tpr)
    roc_results[name] = {
        "fpr": fpr, "tpr": tpr, "auc": roc_auc}

    ax.plot(fpr, tpr, color=color, lw=2.5,
            label=f"{name} (AUC={roc_auc:.2f})")

    jp = f"{RESULTS_DIR}/{json_fname}"
    if os.path.exists(jp):
        with open(jp) as f:
            d = json.load(f)
        saved = float(d.get("test_binary_auc",
                      d.get("binary_auc",
                      d.get("auc", 0))))
        diff  = abs(roc_auc - saved)
        check = "✅" if diff < 0.005 else f"⚠ Δ={diff:.2f}"
        print(f"  {check} {name:<28} "
              f"AUC={roc_auc:.2f} (JSON={saved:.2f})")
    else:
        print(f"  ✓ {name:<28} AUC={roc_auc:.2f}")
    for fname in [
    "TRM_VFD.json", "B6_strm.json",
    "B5_ost.json",  "B2_emb_mlp.json",
    "B1_pixel_mlp.json",
    "TRM_VFD_TWO_STAGE_FINAL_CLEAN.json",]:
      src = f"{DRIVE_RESULTS}/{fname}"
      dst = f"{RESULTS_DIR}/{fname}"
      if os.path.exists(src) and \
        not os.path.exists(dst):
          shutil.copy(src, dst)
          print(f"  ✓ Synced: {fname}")

# Two-Stage MLP
if loaded.get("BinaryDetector", False):
    s1_model.eval()
    scores_ts, binary_ts = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            s1p = torch.softmax(
                s1_model(xb.to(device)), dim=1)
            scores_ts.extend(
                s1p[:, 1].cpu().numpy())
            binary_ts.extend(
                (yb.numpy() != 0).astype(int))

    scores_ts = np.array(scores_ts, dtype=np.float32)
    binary_ts = np.array(binary_ts, dtype=np.int32)
    assert len(scores_ts) == 2000, \
        f"❌ Two-Stage: wrong count {len(scores_ts)}"

    fpr_ts, tpr_ts, _ = roc_curve(binary_ts, scores_ts)
    auc_ts = sklearn_auc(fpr_ts, tpr_ts)
    roc_results["Two-Stage MLP"] = {
        "fpr": fpr_ts, "tpr": tpr_ts, "auc": auc_ts}

    ax.plot(fpr_ts, tpr_ts, color="#1abc9c", lw=2.5,
            label=f"Two-Stage MLP (AUC={auc_ts:.2f})")

    jp = (f"{RESULTS_DIR}/"
          "TRM_VFD_TWO_STAGE_FINAL_CLEAN.json")
    if os.path.exists(jp):
        with open(jp) as f:
            d = json.load(f)
        saved = float(d.get("test_binary_auc", 0))
        diff  = abs(auc_ts - saved)
        check = "✅" if diff < 0.005 else f"⚠ Δ={diff:.2f}"
        print(f"  {check} {'Two-Stage MLP':<28} "
              f"AUC={auc_ts:.2f} (JSON={saved:.2f})")
    else:
        print(f"  ✓ {'Two-Stage MLP':<28} "
              f"AUC={auc_ts:.2f}")
else:
    print("  ⚠ Two-Stage skipped — checkpoint missing")

# Random baseline
ax.plot([0, 1], [0, 1], 'k--', lw=1.5,
        label="Random (AUC=0.50)")

# Figure formatting
ax.set_xlabel("False Positive Rate", fontsize=13)
ax.set_ylabel("True Positive Rate",  fontsize=13)
ax.set_title(
    "ROC Curves — Selected Test-Set Models\n"
    "FakeAVCeleb v1.2 TEST SET\n"
    "(binary: real vs any fake, n=2,000)",
    fontsize=12)
ax.legend(loc="lower right", fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.02])
plt.tight_layout()

# Save figure
for sp in [
    "/content/report_figures/roc_all_models.png",
    f"{DRIVE_FIG}/roc_all_models.png",
]:
    plt.savefig(sp, dpi=150, bbox_inches="tight")
    print(f"\n  ✓ Saved: {sp}")

plt.show()
plt.close()
print("\n" + "=" * 60)
print("STEP 6 — Final AUC Verification")
print("=" * 60)
print(f"  {'Model':<28} {'Live':>8} {'JSON':>8} {'Match':>7}")
print("  " + "-" * 55)

json_map = {
    "Full TRM-VFD":
        "TRM_VFD.json",
    "Simplified TRM (GRU)":
        "B6_strm.json",
    "One-Shot Transformer":
        "B5_ost.json",
    "Embedding MLP":
        "B2_emb_mlp.json",
    "Pixel MLP":
        "B1_pixel_mlp.json",
    "Two-Stage MLP":
        "TRM_VFD_TWO_STAGE_FINAL_CLEAN.json",
}

for mname, fname in json_map.items():
    if mname not in roc_results:
        print(f"  {'⏭'} {mname:<28} {'SKIPPED':>8}")
        continue
    live = roc_results[mname]["auc"]
    jp   = f"{RESULTS_DIR}/{fname}"
    if os.path.exists(jp):
        with open(jp) as f:
            d = json.load(f)
        saved = float(d["test_binary_auc"])

        diff  = abs(live - saved)
        mark  = "✅" if diff < 0.005 else f"⚠ Δ={diff:.2f}"
        print(f"  {mark} {mname:<28} "
              f"{live:>8.2f} {saved:>8.2f}")
    else:
        print(f"  ✓ {mname:<28} "
              f"{live:>8.2f} {'N/A':>8}")

print("\n" + "=" * 60)
print("✓ TRUSTWORTHY ROC FIGURE COMPLETE")
print("=" * 60)
print("  Source:     live inference on test set")
print("  Checkpoints: best val F1 per model")
print("  Test set:   2,000 identity-disjoint samples")
print("  Score:      1-P(real) | Two-Stage: P(fake)")
print("  No .npy:    all scores computed fresh")
print(f"  Models:     {len(roc_results)} plotted")
print("\n  → Use roc_all_models.png as selected ROC panel in paper")

In [ ]:
# CELL 37 │ Phase 11 │ Confusion Matrices — All Models
def plot_cm(cm, labels, title, save_path):
    fig, ax = plt.subplots(figsize=(6,5))
    im = ax.imshow(cm, cmap='Blues')
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_yticklabels(labels)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(title)
    thresh = cm.max()/2
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=11,fontweight='bold',
                    color='white' if cm[i,j]>thresh else 'black')
    plt.tight_layout(); plt.savefig(save_path,dpi=150); plt.show(); plt.close()

labels_4 = ['real','fake_audio','fake_video','fake_both']
for fname, title, labels in [
    ("B1_pixel_mlp.json", "Pixel MLP", labels_4),
    ("B2_emb_mlp.json", "Embedding MLP", labels_4),
    ("B5_ost.json", "One-Shot Transformer", labels_4),
    ("B6_strm.json", "Simplified TRM (GRU)", labels_4),
    ("TRM_VFD.json", "Full TRM-VFD", labels_4),
]:
    path = f"/content/results/{fname}"
    if os.path.exists(path):
        with open(path) as f:
            r = json.load(f)

        cm = None
        if "test_confusion_matrix" in r:
            cm = np.array(r["test_confusion_matrix"])
        elif "confusion_matrix" in r:
            cm = np.array(r["confusion_matrix"])

        if cm is not None:
            save = f"/content/report_figures/cm_{fname.replace('.json','')}.png"
            plot_cm(cm, labels, f"Confusion Matrix — {title}", save)

In [ ]:
# CELL 38 │ Phase 11 │ Macro F1 Comparison Bar Chart
RESULTS_DIR   = "/content/results"
DRIVE_RESULTS = "/content/drive/MyDrive/TRM_DATASETS/results"

# Same key-safe loader as master table
def safe_get(d, *keys, default=0.0):
    for k in keys:
        if k in d:
            return d[k]
    return default

def load_f1(fname):
    for base in [RESULTS_DIR, DRIVE_RESULTS]:
        path = os.path.join(base, fname)
        if os.path.exists(path):
            with open(path) as f:
                d = json.load(f)
            return safe_get(d,
                "test_macro_f1", "macro_f1",
                "test_f1", "val_macro_f1",
                default=0.0)
    return 0.0   # file not found

# Models in display order (worst → best)
model_configs = [
    ("B1_pixel_mlp.json", "Pixel MLP"),
    ("B4_vfd.json",       "VFD Cosine"),
    ("B3_mvf.json",       "MVF (30 ep)"),
    ("B5_ost.json",       "One-Shot Transformer"),
    ("B2_emb_mlp.json",   "Embedding MLP"),
    ("TRM_VFD.json",      "Full TRM-VFD"),
    ("B6_strm.json",      "Simplified TRM (GRU)"),  # ← was B6_simplified_trm.json
]
models = [m for _, m in model_configs]
f1s    = [load_f1(f) for f, _ in model_configs]

# Sort by F1 ascending for the chart
paired = sorted(zip(f1s, models))
f1s_sorted    = [x[0] for x in paired]
models_sorted = [x[1] for x in paired]

# Debug print — confirm values before plotting
print("F1 values loaded:")
for m, v in zip(models_sorted, f1s_sorted):
    flag = " ← ZERO (JSON key mismatch?)" if v == 0.0 else ""
    print(f"  {m:<25} {v:.2f}{flag}")

colors = ["#e74c3c" if v == 0 else
          "#e91e63" if "Full TRM" in m else
          "#9b59b6" if "Simplified" in m else
          "#3498db"
          for v, m in zip(f1s_sorted, models_sorted)]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(models_sorted, f1s_sorted,
              color=colors, edgecolor="white")

ax.set_ylabel("Macro F1 Score")
ax.set_title(
    "Macro F1 Score — All Models\n"
    "FakeAVCeleb v1.2, 4-class identity-disjoint"
)
ax.set_ylim(0, 1.0)
ax.axhline(y=0.25, color="red", linestyle="--",
           linewidth=1, label="Random chance (0.25)")
ax.legend()

for bar, val in zip(bars, f1s_sorted):
    if val > 0:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{val:.3f}",
            ha="center", fontsize=9, fontweight="bold"
        )

plt.xticks(rotation=15, ha="right")
plt.tight_layout()

fig_path_local = "/content/report_figures/fig_f1_comparison.png"
fig_path_drive = "/content/drive/MyDrive/TRM_DATASETS/fig_f1_comparison.png"
plt.savefig(fig_path_local, dpi=150)
plt.savefig(fig_path_drive, dpi=150)
plt.show()
plt.close()
print("F1 comparison chart saved")

# If still all zeros, print actual keys to diagnose
if all(v == 0.0 for v in f1s_sorted):
    print("\nWARNING: All F1 values are zero.")
    print("Actual keys found in each JSON:")
    for fname, model_name in model_configs:
        for base in [RESULTS_DIR, DRIVE_RESULTS]:
            path = os.path.join(base, fname)
            if os.path.exists(path):
                with open(path) as f:
                    d = json.load(f)
                print(f"  {model_name}: {list(d.keys())}")
                break
        else:
            print(f"  {model_name}: FILE NOT FOUND")

In [ ]:
# CELL 39 │ Phase 11 │ t-SNE Visualization of ArcFace+ECAPA Embeddings

from sklearn.manifold import TSNE
idx = np.random.choice(len(X_val), min(600,len(X_val)), replace=False)
X_s = X_val[idx]; y_s = y_val[idx]

print("Running t-SNE (2-3 min)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_2d = tsne.fit_transform(X_s)

fig, ax = plt.subplots(figsize=(9,7))
label_names = {0:"real",1:"fake_audio",2:"fake_video",3:"fake_both"}
colors_tsne = ["#2ecc71","#e74c3c","#3498db","#9b59b6"]
for cls in range(4):
    mask = y_s == cls
    ax.scatter(X_2d[mask,0], X_2d[mask,1], c=colors_tsne[cls],
               label=label_names[cls], alpha=0.6, s=15)
ax.set_title("t-SNE: ArcFace + ECAPA Identity Embeddings (4-class)")
ax.legend(); ax.set_xlabel("t-SNE dim 1"); ax.set_ylabel("t-SNE dim 2")
plt.tight_layout()
plt.savefig("/content/report_figures/tsne_embeddings.png", dpi=150)
plt.savefig(
    "/content/drive/MyDrive/TRM_DATASETS/tsne_embeddings.png",
    dpi=150)
plt.show(); plt.close()
print("t-SNE saved")

In [ ]:
#CELL 40 │ Phase 11 │ Download All Results + Figures
from google.colab import files
import zipfile, glob, os

with zipfile.ZipFile("/content/TRM_results.zip","w") as z:
    for f in glob.glob("/content/results/*.json"):
        z.write(f, os.path.basename(f))
    for f in glob.glob("/content/results/*.csv"):
        z.write(f, os.path.basename(f))

with zipfile.ZipFile("/content/TRM_figures.zip","w") as z:
    for f in glob.glob("/content/report_figures/*.png"):
        z.write(f, os.path.basename(f))

files.download("/content/TRM_results.zip")
files.download("/content/TRM_figures.zip")
print("Downloaded everything")

In [ ]:
# CELL 41 │ Phase 11 │ Seed Stability Study [OPTIONAL]
# Not required for main paper results. Runtime ~45 min.

import time, os, json
import numpy as np
import torch
import torch.optim as optim
from sklearn.metrics import f1_score

seeds     = [42, 123, 456]
save_dir  = "/content/drive/MyDrive/TRM_DATASETS"
ckpt_dir  = "/content/drive/MyDrive/TRM_DATASETS/trm_checkpoint"
local_dir = "/content/results"
os.makedirs(save_dir,  exist_ok=True)
os.makedirs(ckpt_dir,  exist_ok=True)
os.makedirs(local_dir, exist_ok=True)

# Results store: val + test per run
sig_results = {
    "Embedding MLP":  [],   # each entry: {"val_f1": x, "test_f1": y}
    "Simplified TRM": [],
    "Full TRM-VFD":   []
}

# Helper: evaluate on any loader
def evaluate(model, loader, device):
    model.eval()
    all_p, all_l = [], []
    with torch.no_grad():
        for xb, yb in loader:
            all_p.extend(
                model(xb.to(device)).argmax(1).cpu().numpy())
            all_l.extend(yb.numpy())
    return f1_score(all_l, all_p,
                    average='macro', zero_division=0)

# Helper: save partial results to Drive
def save_partial(results, save_dir):
    summary = {}
    for k, runs in results.items():
        if not runs:
            summary[k] = {"mean_val": 0, "std_val": 0,
                          "mean_test": 0, "std_test": 0, "runs": []}
        else:
            val_f1s  = [r["val_f1"]  for r in runs]
            test_f1s = [r["test_f1"] for r in runs]
            summary[k] = {
                "mean_val":  float(np.mean(val_f1s)),
                "std_val":   float(np.std(val_f1s)),
                "mean_test": float(np.mean(test_f1s)),
                "std_test":  float(np.std(test_f1s)),
                "runs":      runs
            }
    with open(f"{save_dir}/statistical_significance.json", "w") as f:
        json.dump(summary, f, indent=2)

# Verify loaders and models are in memory
print("── Pre-run verification ──")
print(f"Train: {len(train_loader_emb.dataset):,} samples")
print(f"Val:   {len(val_loader_emb.dataset):,} samples")
print(f"Test:  {len(test_loader.dataset):,} samples")
assert len(test_loader.dataset) > 0, "❌ test_loader is empty"
assert EmbeddingMLP  is not None, "❌ EmbeddingMLP not defined"
assert SimplifiedTRM is not None, "❌ SimplifiedTRM not defined"
assert TRM_VFD       is not None, "❌ TRM_VFD not defined"
assert FocalLoss     is not None, "❌ FocalLoss not defined"
assert weights_emb   is not None, "❌ weights_emb not defined"
print("✓ All required objects in memory\n")

total_start = time.time()

for seed in seeds:
    torch.manual_seed(seed)
    np.random.seed(seed)

    print(f"\n{'='*55}")
    print(f"SEED {seed}")
    print(f"{'='*55}")

    # MODEL 1 — Embedding MLP
    print("\n[Embedding MLP]")
    m1      = EmbeddingMLP().to(device)
    o1      = optim.Adam(m1.parameters(), lr=5e-4, weight_decay=1e-4)
    s1      = optim.lr_scheduler.CosineAnnealingLR(o1, T_max=50)
    c1      = FocalLoss(alpha=weights_emb, gamma=2.0)
    ckpt_m1 = f"{local_dir}/seed{seed}_emb_mlp.pth"

    best1_val = -1.0
    for epoch in range(50):
        m1.train()
        for xb, yb in train_loader_emb:
            xb, yb = xb.to(device), yb.to(device)
            o1.zero_grad()
            c1(m1(xb), yb).backward()
            o1.step()
        s1.step()

        f1_val = evaluate(m1, val_loader_emb, device)
        if f1_val > best1_val:
            best1_val = f1_val
            torch.save(m1.state_dict(), ckpt_m1)

    # Load best and eval on TEST
    m1.load_state_dict(torch.load(ckpt_m1, map_location=device))
    best1_test = evaluate(m1, test_loader, device)

    sig_results["Embedding MLP"].append({
        "seed":     seed,
        "val_f1":  round(best1_val,  4),
        "test_f1": round(best1_test, 4)
    })
    print(f"  Val F1:  {best1_val:.4f}")
    print(f"  Test F1: {best1_test:.4f}  ← paper number")
    save_partial(sig_results, save_dir)

    # MODEL 2 — Simplified TRM
    print("\n[Simplified TRM]")
    m2      = SimplifiedTRM().to(device)
    o2      = optim.Adam(m2.parameters(), lr=5e-4, weight_decay=1e-4)
    s2      = optim.lr_scheduler.CosineAnnealingLR(o2, T_max=50)
    c2      = FocalLoss(alpha=weights_emb, gamma=2.0)
    ckpt_m2 = f"{local_dir}/seed{seed}_strm.pth"

    best2_val = -1.0
    for epoch in range(50):
        m2.train()
        for xb, yb in train_loader_emb:
            xb, yb = xb.to(device), yb.to(device)
            o2.zero_grad()
            loss = c2(m2(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m2.parameters(), 1.0)
            o2.step()
        s2.step()

        f1_val = evaluate(m2, val_loader_emb, device)
        if f1_val > best2_val:
            best2_val = f1_val
            torch.save(m2.state_dict(), ckpt_m2)

    # Load best and eval on TEST
    m2.load_state_dict(torch.load(ckpt_m2, map_location=device))
    best2_test = evaluate(m2, test_loader, device)

    sig_results["Simplified TRM"].append({
        "seed":     seed,
        "val_f1":  round(best2_val,  4),
        "test_f1": round(best2_test, 4)
    })
    print(f"  Val F1:  {best2_val:.4f}")
    print(f"  Test F1: {best2_test:.4f}  ← paper number")
    save_partial(sig_results, save_dir)

    # MODEL 3 — Full TRM-VFD
    print("\n[Full TRM-VFD]")
    m3      = TRM_VFD(num_steps=5).to(device)
    o3      = optim.Adam(m3.parameters(), lr=5e-4, weight_decay=1e-4)
    s3      = optim.lr_scheduler.CosineAnnealingLR(o3, T_max=50)
    c3      = FocalLoss(alpha=weights_emb, gamma=2.0)
    ckpt_m3 = f"{local_dir}/seed{seed}_trm.pth"

    best3_val = -1.0
    for epoch in range(50):
        m3.train()
        for xb, yb in train_loader_emb:
            xb, yb = xb.to(device), yb.to(device)
            o3.zero_grad()
            loss = c3(m3(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m3.parameters(), 1.0)
            o3.step()
        s3.step()

        f1_val = evaluate(m3, val_loader_emb, device)
        if f1_val > best3_val:
            best3_val = f1_val
            torch.save(m3.state_dict(), ckpt_m3)

    # Load best and eval on TEST
    m3.load_state_dict(torch.load(ckpt_m3, map_location=device))
    best3_test = evaluate(m3, test_loader, device)

    sig_results["Full TRM-VFD"].append({
        "seed":     seed,
        "val_f1":  round(best3_val,  4),
        "test_f1": round(best3_test, 4)
    })
    print(f"  Val F1:  {best3_val:.4f}")
    print(f"  Test F1: {best3_test:.4f}  ← paper number")
    save_partial(sig_results, save_dir)
    print(f"\n  ✓ Seed {seed} complete — saved to Drive")

# Final summary table
total_time = (time.time() - total_start) / 60
print(f"\nTotal time: {total_time:.1f} minutes")

print("\n" + "=" * 65)
print("STATISTICAL SIGNIFICANCE TABLE (3 seeds)")
print("=" * 65)
print(f"{'Model':<20} {'Val Mean':>9} {'Val Std':>8} "
      f"{'Test Mean':>10} {'Test Std':>9}  Runs")
print("-" * 65)

final = {}
for name, runs in sig_results.items():
    val_f1s  = [r["val_f1"]  for r in runs]
    test_f1s = [r["test_f1"] for r in runs]
    mean_val  = float(np.mean(val_f1s))
    std_val   = float(np.std(val_f1s))
    mean_test = float(np.mean(test_f1s))
    std_test  = float(np.std(test_f1s))

    print(f"{name:<20} {mean_val:>9.3f} {std_val:>8.3f} "
          f"{mean_test:>10.3f} {std_test:>9.3f}  {test_f1s}")
    print(f"  → Paper: {mean_test:.3f} ± {std_test:.3f} "
      f"(test F1, n=3 seeds)")

    final[name] = {
        "mean_val":  mean_val,
        "std_val":   std_val,
        "mean_test": mean_test,
        "std_test":  std_test,
        "runs":      runs
    }

# Save final
for path in ["/content/results/statistical_significance.json",
             f"{save_dir}/statistical_significance.json"]:
    with open(path, "w") as f:
        json.dump(final, f, indent=2)

print("\n✓ Saved statistical_significance.json locally and to Drive")

# Paper-ready output
print("\n" + "=" * 65)
print("COPY INTO PAPER (Table 4):")
print("=" * 65)
print(f"{'Model':<20} {'Val F1':>15} {'Test F1':>15}")
print("-" * 55)
for name, data in final.items():
    print(f"{name:<20} "
          f"{data['mean_val']:.3f} ± {data['std_val']:.3f}   "
          f"{data['mean_test']:.3f} ± {data['std_test']:.3f}")

print("\nNote: Val F1 = model selection metric (reported for transparency)")
print("      Test F1 = paper-reported number (held-out, touched once per seed)")
print("      Split: "
      f"{len(train_loader_emb.dataset):,} train / "
      f"{len(val_loader_emb.dataset):,} val / "
      f"{len(test_loader.dataset):,} test")
print("\n" + "=" * 65)
print("NOTE ON SEED STABILITY REPORTING:")
print("=" * 65)
print("  This study reports Macro F1 variance across seeds.")
print("  Binary AUC variance is NOT reported (not computed per seed).")
print("  Paper wording must say:")
print("  'We report mean ± std of Macro F1 across 3 random seeds'")
print("  NOT 'full metric variance' or 'all metrics stable'")

In [ ]:
# CELL 42 │ Phase 11 │ Cross-Dataset — Check Celeb-DF v2 Exists
# Check all possible locations
paths = [
    "/content/drive/MyDrive/Celeb-DF-v2.zip",
    "/content/drive/MyDrive/CelebDF/Celeb-DF-v2.zip",
    "/content/drive/MyDrive/Celeb_DF_v2.zip",
    "/content/celebdf",
]
for p in paths:
    if os.path.exists(p):
        print(f"✓ FOUND: {p}")
        break
else:
    print("NOT FOUND on Drive")
    print("Go here: github.com/yuezunli/celeb-deepfakeforensics")
    print("Fill Google Form → get download link")
    print("Download zip to your Google Drive")
    print("File is 3GB — takes 15-20 minutes")

In [ ]:
# CELL 43 │ Phase 11 │ Cross-Dataset — Unzip Celeb-DF v2
import zipfile, os

# change this path to wherever your zip is
celeb_zip = "/content/drive/MyDrive/Celeb-DF-v2.zip"

if os.path.exists(celeb_zip):
    print("Unzipping Celeb-DF v2...")
    os.makedirs("/content/celebdf", exist_ok=True)
    with zipfile.ZipFile(celeb_zip, 'r') as z:
        z.extractall("/content/celebdf")
    print("Done")
    vids = glob.glob("/content/celebdf/**/*.mp4", recursive=True)
    print(f"Total videos: {len(vids)}")
    print("Sample paths:", vids[:3])
else:
    print("Zip not found at that path")
    print("Check your Drive and update the path above")

In [ ]:
# CELL 44 │ Phase 11 │ Cross-Dataset — Verify Celeb-DF Structure
celeb_root = "/content/celebdf"

# find actual root (may be nested)
for root, dirs, files in os.walk(celeb_root):
    mp4s = [f for f in files if f.endswith('.mp4')]
    if mp4s:
        print(f"Found videos in: {root}")
        break

# check expected folders
for folder in ["Celeb-real", "YouTube-real", "Celeb-synthesis"]:
    path = os.path.join(celeb_root, folder)
    # also check one level deeper
    if not os.path.exists(path):
        path = os.path.join(celeb_root,
                            "Celeb-DF-v2", folder)
    if os.path.exists(path):
        count = len(glob.glob(
            os.path.join(path,"*.mp4")))
        print(f"✓ {folder}: {count} videos")
    else:
        print(f"✗ {folder}: not found")

In [ ]:
# CELL 45 │ Phase 11 │ Face-Only Transfer Probe — Celeb-DF v2
# FACE-ONLY TRANSFER PROBE — FakeAVCeleb → Celeb-DF v2
#
# Zero retraining. Models trained on FakeAVCeleb evaluated on Celeb-DF v2.
# Celeb-DF is BINARY (real=0, fake=1) — no audio in most videos.
# Voice embeddings are zeros(192) → face-only evaluation.
# Metric: Binary AUC (primary), Binary F1, Accuracy.
#
# Models evaluated:
#   - Embedding MLP          (emb_mlp)
#   - Simplified TRM         (strm_model)
#   - Full TRM-VFD           (trm_model)
#   - Two-Stage MLP Pipeline      (s1_model + s2_model)
#
# PRE-REQUISITE: run the Celeb-DF embedding extraction cell first.
#   X_celeb, y_celeb must exist in memory (or saved to Drive).

print("FACE-ONLY TRANSFER PROBE: FakeAVCeleb → Celeb-DF v2")
print("Voice = zeros(192). Primary metric: Binary AUC only.")
print("F1/Accuracy not comparable — class ratio 2:1 vs 1:39")

RESULTS_DIR   = "/content/results"
DRIVE_RESULTS = "/content/drive/MyDrive/TRM_DATASETS/results"
os.makedirs(RESULTS_DIR,   exist_ok=True)
os.makedirs(DRIVE_RESULTS, exist_ok=True)

celebdf_save_dir = "/content/drive/MyDrive/TRM_DATASETS/celebdf_embeddings"

try:
    _ = X_celeb, y_celeb
    print(f"✓ Using X_celeb already in memory | shape: {X_celeb.shape}")
except NameError:
    print("Loading Celeb-DF embeddings from Drive...")
    X_celeb = np.load(f"{celebdf_save_dir}/X_celeb.npy")
    y_celeb = np.load(f"{celebdf_save_dir}/y_celeb.npy")
    print(f"✓ Loaded | shape: {X_celeb.shape}")

print(f"\nCeleb-DF v2 test set:")
print(f"  Total: {len(y_celeb)}")
print(f"  Real:  {np.sum(y_celeb==0)}")
print(f"  Fake:  {np.sum(y_celeb==1)}")
print(f"  Voice: zeros(192) — face-only evaluation")

# Check for high zero-face rate
X_face_part = X_celeb[:, :512]
zero_face_mask = np.all(X_face_part == 0, axis=1)
zero_face_rate = zero_face_mask.mean()
print(f"  Zero face rate: {zero_face_rate*100:.1f}%")
if zero_face_rate > 0.3:
    print("  ⚠ High zero-face rate — ArcFace failed on many frames.")
    print("    Results will be noisy. Report with this caveat.")

# Remove samples with zero face embedding — they are uninformative
valid_mask = ~zero_face_mask
X_celeb_clean = X_celeb[valid_mask]
y_celeb_clean = y_celeb[valid_mask]
print(f"\nAfter removing zero-face: {len(y_celeb_clean)} samples")
print(f"  Real: {np.sum(y_celeb_clean==0)} | Fake: {np.sum(y_celeb_clean==1)}")

if len(y_celeb_clean) < 50:
    raise RuntimeError(
        "Too few valid samples after zero-face filtering. "
        "Re-run the embedding extraction cell with a better face detector."
    )

# DataLoader for cross-dataset eval
X_celeb_t = torch.tensor(X_celeb_clean, dtype=torch.float32)
y_celeb_t  = torch.tensor(y_celeb_clean, dtype=torch.long)

celeb_loader = DataLoader(
    TensorDataset(X_celeb_t, y_celeb_t),
    batch_size=256,
    shuffle=False
)

def eval_4class_as_binary(model, loader, device):
    """
    For 4-class models (cls 0=real, 1/2/3=fake).
    Binary pred: 0 if argmax=0, else 1.
    Binary score: 1 - P(real) for AUC.
    """
    model.eval()
    all_p, all_sc, all_l = [], [], []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb.to(device))
            probs  = torch.softmax(logits, dim=1)
            pred   = (logits.argmax(1) != 0).long()
            score  = 1.0 - probs[:, 0]     # higher = more likely fake
            all_p.extend(pred.cpu().numpy())
            all_sc.extend(score.cpu().numpy())
            all_l.extend(yb.numpy())
    return np.array(all_l), np.array(all_p), np.array(all_sc)

def eval_two_stage_as_binary(s1_model, s2_model, loader, device):
    """
    For Two-Stage MLP Pipeline.
    Binary pred: Stage 1 output directly (0=real, 1=fake).
    Binary score: s1 P(fake) for AUC.
    """
    s1_model.eval()
    s2_model.eval()
    all_p, all_sc, all_l = [], [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            s1_probs = torch.softmax(s1_model(xb), dim=1)
            pred     = s1_probs.argmax(1)          # 0=real, 1=fake
            score    = s1_probs[:, 1]              # P(fake) for AUC
            all_p.extend(pred.cpu().numpy())
            all_sc.extend(score.cpu().numpy())
            all_l.extend(yb.numpy())
    return np.array(all_l), np.array(all_p), np.array(all_sc)

def compute_metrics(all_l, all_p, all_sc):
    auc = roc_auc_score(all_l, all_sc)
    f1  = f1_score(all_l, all_p, average="macro", zero_division=0)
    acc = accuracy_score(all_l, all_p)
    bal = balanced_accuracy_score(all_l, all_p)
    return auc, f1, acc, bal

# RUN CROSS-DATASET EVALUATION
print("\n" + "=" * 60)
print("CROSS-DATASET: FakeAVCeleb → Celeb-DF v2")
print("Zero retraining — pure generalization test")
print("Voice = zeros(192), face-only signal")
print("=" * 60)
print(f"\n{'Model':<25} {'AUC':>7} {'F1':>7} {'Acc':>7} {'BalAcc':>8}")
print("-" * 58)

cross_results = {}

# 4-class models
model_list = []

try:
    _ = emb_mlp
    model_list.append(("Embedding MLP", emb_mlp))
except NameError:
    print("⚠ emb_mlp not in memory — skipping")

try:
    _ = strm_model
    model_list.append(("Simplified TRM", strm_model))
except NameError:
    print("⚠ strm_model not in memory — skipping")

try:
    _ = trm_model
    model_list.append(("Full TRM-VFD", trm_model))
except NameError:
    print("⚠ trm_model not in memory — skipping")

for name, model in model_list:
    all_l, all_p, all_sc = eval_4class_as_binary(model, celeb_loader, device)
    auc, f1, acc, bal = compute_metrics(all_l, all_p, all_sc)
    cross_results[name] = {
        "auc": round(float(auc), 4),
        "f1":  round(float(f1),  4),
        "acc": round(float(acc), 4),
        "bal": round(float(bal), 4),
    }
    print(f"{name:<25} {auc:>7.4f} {f1:>7.4f} {acc:>7.4f} {bal:>8.4f}")

# Two-Stage MLP Pipeline
try:
    _ = s1_model, s2_model
    all_l, all_p, all_sc = eval_two_stage_as_binary(
        s1_model, s2_model, celeb_loader, device
    )
    auc, f1, acc, bal = compute_metrics(all_l, all_p, all_sc)
    cross_results["Two-Stage MLP Pipeline"] = {
        "auc": round(float(auc), 4),
        "f1":  round(float(f1),  4),
        "acc": round(float(acc), 4),
        "bal": round(float(bal), 4),
    }
    print(f"{'Two-Stage MLP Pipeline':<25} {auc:>7.4f} {f1:>7.4f} {acc:>7.4f} {bal:>8.4f}")
except NameError:
    print("⚠ s1_model/s2_model not in memory — skipping Two-Stage")

print("-" * 58)

print("\nGeneralization analysis (AUC — higher is better):")
for name, res in cross_results.items():
    print(f"  {name:<25} AUC={res['auc']:.4f}")

# Check if TRM generalizes better than Simplified TRM
trm_auc   = cross_results.get("Full TRM-VFD",     {}).get("auc", 0)
strm_auc  = cross_results.get("Simplified TRM",   {}).get("auc", 0)
ts_auc    = cross_results.get("Two-Stage MLP Pipeline", {}).get("auc", 0)
emb_auc   = cross_results.get("Embedding MLP",     {}).get("auc", 0)

if trm_auc and strm_auc:
    print(f"\n  TRM vs sTRM gap:    {trm_auc - strm_auc:+.4f} "
          f"({'TRM better' if trm_auc > strm_auc else 'sTRM better'})")
if ts_auc and trm_auc:
    print(f"  Two-Stage vs TRM:   {ts_auc - trm_auc:+.4f} "
          f"({'Two-Stage better' if ts_auc > trm_auc else 'TRM better'})")
if trm_auc and emb_auc:
    print(f"  TRM vs EmbMLP gap:  {trm_auc - emb_auc:+.4f} "
          f"({'TRM generalizes better' if trm_auc > emb_auc else 'EmbMLP generalizes better'})")

output = {
    "train_dataset":   "FakeAVCeleb v1.2",
    "test_dataset":    "Celeb-DF v2",
    "n_test_raw":      int(len(y_celeb)),
    "n_test_valid":    int(len(y_celeb_clean)),
    "zero_face_rate":  float(zero_face_rate),
    "note": (
        "Voice embeddings set to zeros(192) — Celeb-DF has no synchronized audio. "
        "Face-only generalization test. Zero-face samples removed before evaluation. "
        "Binary AUC is the primary metric for this cross-dataset setting."
    ),
    "results": cross_results,
}

local_path = f"{RESULTS_DIR}/cross_dataset_celebdf.json"
drive_path = f"{DRIVE_RESULTS}/cross_dataset_celebdf.json"

with open(local_path, "w") as f:
    json.dump(output, f, indent=2)
with open(drive_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"\n✓ Saved: {local_path}")
print(f"✓ Saved: {drive_path}")

print("\n" + "=" * 60)
print("CROSS-DATASET: FakeAVCeleb → Celeb-DF v2 (FACE-ONLY)")
print("⚠ Voice = zeros(192) — no audio in Celeb-DF")
print("⚠ AUC only — class ratio 2:1 differs from FakeAVCeleb 1:39")
print("⚠ F1/Accuracy NOT comparable across datasets")
print("=" * 60)